In [ ]:

"""
Moudel 1(a)
"""

import pandas as pd
import numpy as np

# --- Set output filename ---
OUTPUT_EXCEL_FILE = 'Question1_Answer1.xlsx'

# --- Load data ---
df_patient_info = pd.read_excel('Table1-Patient_List_and_Clinical_Information.xlsx')
df_imaging_volume = pd.read_excel('Table2-Patient_Imaging-Volume_and_Location.xlsx')
df_time_lookup = pd.read_excel('Appendix1-Retrieval_Table-SerialNumber_vs_Time.xlsx')

# --- Select the first 100 patients ---
train_ids = [f"sub{str(i).zfill(3)}" for i in range(1, 101)]
df_patient_info = df_patient_info[df_patient_info['ID'].isin(train_ids)].set_index('ID')
df_imaging_volume = df_imaging_volume[df_imaging_volume['ID'].isin(train_ids)].set_index('ID')
df_time_lookup = df_time_lookup[df_time_lookup['ID'].isin(train_ids)].set_index('ID')

# --- Convert time columns to datetime ---
for col in df_time_lookup.columns:
    if 'time_point' in col.lower():
        df_time_lookup[col] = pd.to_datetime(df_time_lookup[col].astype(str).str.strip(), errors='coerce')

# --- Extract hematoma volume columns ---
hmvol = df_imaging_volume.filter(regex=r'HM_volume.*')

# --- Time from onset to first imaging (in hours) ---
time_onset_to_first_scan = df_patient_info[['Time_from_Onset_to_First_Imaging']]

# --- Calculate time differences between follow-up and first scan ---
start_time_col = 'First_check_time_point'
delta_time_list = []

for i in range(1, 14):
    followup_col = f'Follow_up_{i}_check_time_point'
    if start_time_col in df_time_lookup.columns and followup_col in df_time_lookup.columns:
        time_diff = (df_time_lookup[followup_col] - df_time_lookup[start_time_col]).dt.total_seconds() / 3600
        delta_time_list.append(time_diff)

# --- Combine all time differences into one DataFrame ---
time_diffs_df = pd.concat(delta_time_list, axis=1)
time_diffs_df.columns = [f'deltatime_followup_{i}' for i in range(len(delta_time_list))]

# --- Define hematoma expansion rules ---
def is_enlarged_by_volume(delta_volume):
    if pd.isna(delta_volume): return 0
    return 1 if delta_volume > 6000 else 0

def is_enlarged_by_percentage(delta_percent):
    if pd.isna(delta_percent): return 0
    return 1 if delta_percent > 0.33 else 0

# --- Check each follow-up for expansion ---
result_df = pd.DataFrame(index=hmvol.index)
time_df = pd.DataFrame(index=hmvol.index)
initial_vol = hmvol.iloc[:, 0]
num_loops = min(hmvol.shape[1] - 1, time_diffs_df.shape[1])

for i in range(1, num_loops + 1):
    delta_volume = hmvol.iloc[:, i] - initial_vol
    delta_percent = delta_volume / initial_vol.replace(0, np.nan)

    enlarged_volume_flag = delta_volume.apply(is_enlarged_by_volume)
    enlarged_percent_flag = delta_percent.apply(is_enlarged_by_percentage)
    enlarged_flag = enlarged_volume_flag | enlarged_percent_flag

    result_df[f'y_{i}'] = enlarged_flag

    # Expansion time = onset to first imaging + follow-up delta
    expansion_time = time_onset_to_first_scan.iloc[:, 0] + time_diffs_df.iloc[:, i - 1]
    time_df[f'dt{i}'] = np.where(enlarged_flag, expansion_time, np.nan)

# --- Final aggregation ---
any_expansion = result_df.max(axis=1)
first_expansion_time = time_df.min(axis=1)

final_result = pd.concat([any_expansion, first_expansion_time], axis=1)
final_result.columns = ['Whether_hematoma_dilation_occurs', 'Time_of_hematoma_expansion']

# --- Remove expansion if it occurred after 48 hours ---
final_result.loc[final_result['Time_of_hematoma_expansion'] > 48, 'Whether_hematoma_dilation_occurs'] = 0
final_result.loc[final_result['Time_of_hematoma_expansion'] > 48, 'Time_of_hematoma_expansion'] = np.nan

# --- Round time to 2 decimals ---
final_result['Time_of_hematoma_expansion'] = final_result['Time_of_hematoma_expansion'].round(2)

# --- Export to Excel ---
final_result.to_excel(OUTPUT_EXCEL_FILE, sheet_name='ans')
print(f"✅ Processing complete. Output saved to: {OUTPUT_EXCEL_FILE}")
print(final_result.head(10))


In [ ]:
"""
Moudel 1(b)

This script is responsible for loading and integrating data from clinical information (Table 1), 
imaging volume/location (Table 2), and imaging shape/gray distribution (Table 3).
It performs a series of preprocessing and feature engineering steps to prepare the final
feature matrix for subsequent modeling tasks.
The script concludes by outputting the total number of features, their names, and a data preview.
"""

# --- Step 0: Import Required Libraries ---
# pandas is used for data manipulation and reading/writing files (e.g., Excel)
# numpy is used for efficient numerical computations, especially for handling missing values
import pandas as pd
import numpy as np
import warnings

# Ignore common warnings that may arise during execution for a cleaner output
warnings.filterwarnings('ignore')

print("--- Script Start: Executing Data Loading and Feature Engineering ---")

# ===================================================================
# --- Step 1: Define Data File Paths ---
# ===================================================================
# Please ensure these Excel files are in the same directory as your Python script,
# or provide the full absolute path to each file.
path_table1 = "Table1-Patient_List_and_Clinical_Information.xlsx"
path_table2 = "Table2-Patient_Imaging-Volume_and_Location.xlsx"
path_table3 = "Table3-Patient_Imaging-Shape_and_Gray_Distribution.xlsx"
path_label = "Question1_Answer1.xlsx"

print("✅ File paths defined.")

# ===================================================================
# --- Step 2: Load and Preprocess Each Data Table ---
# ===================================================================
# Use a try-except block to catch potential file reading errors, making the code more robust.
try:
    # --- Load Table 1: Patient Clinical Information ---
    # Read the raw data, treating all columns as strings initially to avoid data type issues.
    df_info = pd.read_excel(path_table1, dtype=str)
    # As per requirements, specify the clinical feature columns to be used (fields E to W).
    clin_columns = [
        'Age', 'Gender', 'mRS_Score_before_cerebral_hemorrhage', 'History_of_Hypertension', 
        'History_of_Stroke', 'History_of_Diabetes', 'History_of_Atrial_Fibrillation', 
        'History_of_Coronary_Artery_Disease', 'History_of_Smoking', 'History_of_Alcohol_Consumption', 
        'Time_from_Onset_to_First_Imaging', 'Blood_pressure', 'Ventricular_drainage_treatment', 
        'Hemostatic_treatment', 'Reducing_intracranial_pressure_treatment', 
        'Blood_pressure_lowering_treatment', 'Sedation_and_analgesic_treatment', 
        'Stop_vomiting_and_protect_stomach_treatment', 'Neurotrophic_treatment'
    ]
    # Select the ID, First_SerialNumber, and the clinical columns defined above.
    df_info_sub = df_info[['ID', 'First_SerialNumber'] + clin_columns].copy()
    # Clean leading/trailing whitespace from the serial number to ensure accurate matching later.
    df_info_sub['First_SerialNumber'] = df_info_sub['First_SerialNumber'].astype(str).str.strip()

    # --- Load Table 2: Imaging Volume and Location ---
    # Again, read all columns as strings.
    df_vol = pd.read_excel(path_table2, dtype=str)
    cols_vol = df_vol.columns.tolist()
    # **Core Logic**: Extract records for the first imaging scan only (fields C to X).
    # We identify the boundary of the first scan's features by finding the 'Follow_up_1_SerialNumber' column.
    if 'Follow_up_1_SerialNumber' in cols_vol:
        idx_end_first_scan = cols_vol.index('Follow_up_1_SerialNumber')
        # Extract columns from 'C' (index 2) up to the first follow-up column, plus the serial number.
        first_scan_vol_cols = ['First_SerialNumber'] + cols_vol[2:idx_end_first_scan]
    else: # If no follow-up columns exist, assume all imaging columns belong to the first scan.
        first_scan_vol_cols = ['First_SerialNumber'] + cols_vol[2:]
    
    # Create a new DataFrame containing only the first scan's volume data.
    df_vol_first = df_vol[first_scan_vol_cols].copy()
    # Rename the serial number column to 'SerialNumber' for consistent merging.
    df_vol_first.rename(columns={'First_SerialNumber': 'SerialNumber'}, inplace=True)
    df_vol_first['SerialNumber'] = df_vol_first['SerialNumber'].astype(str).str.strip()

    # --- Load Table 3: Imaging Shape and Gray Distribution ---
    # Table 3 contains two sheets ('Hemo' and 'ED'), which must be read separately.
    # Use add_prefix() to add 'Hemo_' and 'ED_' to column names to avoid conflicts.
    df_shape_hemo = pd.read_excel(path_table3, sheet_name='Hemo', dtype=str).add_prefix('Hemo_')
    df_shape_ed = pd.read_excel(path_table3, sheet_name='ED', dtype=str).add_prefix('ED_')
    # *** FIX: REMOVE UNNAMED INDEX COLUMNS THAT MIGHT BE READ FROM EXCEL ***
    if 'Hemo_Unnamed: 0' in df_shape_hemo.columns:
        df_shape_hemo = df_shape_hemo.drop(columns=['Hemo_Unnamed: 0'])
        print("Removed 'Hemo_Unnamed: 0' column.")
    if 'ED_Unnamed: 0' in df_shape_ed.columns:
        df_shape_ed = df_shape_ed.drop(columns=['ED_Unnamed: 0'])
        print("Removed 'ED_Unnamed: 0' column.")

    # Similarly, rename serial number columns and clean whitespace.
    df_shape_hemo.rename(columns={'Hemo_SerialNumber': 'SerialNumber'}, inplace=True)
    df_shape_ed.rename(columns={'ED_SerialNumber': 'SerialNumber'}, inplace=True)
    df_shape_hemo['SerialNumber'] = df_shape_hemo['SerialNumber'].astype(str).str.strip()
    df_shape_ed['SerialNumber'] = df_shape_ed['SerialNumber'].astype(str).str.strip()

    # --- Load Label Data ---
    # This file contains the target variable (whether hematoma expansion occurred).
    df_label = pd.read_excel(path_label, dtype=str)
    # Rename the column for easier access.
    df_label.rename(columns={'Whether_hematoma_dilation_occurs': 'Hematoma_Expansion'}, inplace=True)
    # Convert the target variable to a numeric type; set failed conversions to NaN (Not a Number).
    df_label['Hematoma_Expansion'] = pd.to_numeric(df_label['Hematoma_Expansion'], errors='coerce')
    
    print("✅ All source data tables loaded and preprocessed successfully.")

except FileNotFoundError as e:
    print(f"❌ CRITICAL ERROR: File not found. Please check the file paths. Details: {e}")
    exit()
except Exception as e:
    print(f"❌ CRITICAL ERROR during data loading. Details: {e}")
    exit()

# ===================================================================
# --- Step 3: Merge All Data Sources ---
# ===================================================================
# Starting with the clinical info table (df_info_sub), progressively merge the other tables.
# Using a 'left' join ensures that all patient records from the primary table are kept.
# 1. Merge Table 2's first scan volume data.
df_full = pd.merge(df_info_sub, df_vol_first, left_on='First_SerialNumber', right_on='SerialNumber', how='left')
# 2. Merge Table 3's hematoma shape data.
df_full = pd.merge(df_full, df_shape_hemo, on='SerialNumber', how='left')
# 3. Merge Table 3's edema shape data.
df_full = pd.merge(df_full, df_shape_ed, on='SerialNumber', how='left')
# 4. Merge the label data.
df_full = pd.merge(df_full, df_label[['ID', 'Hematoma_Expansion']], on='ID', how='left')

# The 'SerialNumber' column is now redundant and can be dropped.
df_full.drop(columns=['SerialNumber'], inplace=True)

print("✅ All data tables have been successfully merged into a single master table.")

# ===================================================================
# --- Step 4: Feature Engineering ---
# ===================================================================
# In this step, we will create new, potentially more predictive features based on the raw data.

# --- Convert all columns intended for numerical calculation to numeric types ---
# For values that cannot be converted (e.g., text), set them to NaN for uniform handling later.
numeric_cols = ['Age', 'Time_from_Onset_to_First_Imaging', 'HM_volume', 'ED_volume'] + \
               [col for col in df_full.columns if 'Ratio' in col or 'Hemo_' in col or 'ED_' in col]
for col in numeric_cols:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce')

# --- Create new interaction and ratio features ---
# Create the Edema-to-Hematoma Volume Ratio (ED/HM Ratio), a clinically relevant metric.
# Add a small epsilon (1e-6) to the denominator to prevent division-by-zero errors.
df_full['ED_HM_Ratio'] = df_full['ED_volume'] / (df_full['HM_volume'] + 1e-6)

# --- Process Blood Pressure features ---
if 'Blood_pressure' in df_full.columns:
    # Split strings like '140/90' into separate 'Systolic_BP' and 'Diastolic_BP' columns.
    bp_data = df_full['Blood_pressure'].str.split('/', expand=True)
    df_full['Systolic_BP'] = pd.to_numeric(bp_data[0], errors='coerce')
    df_full['Diastolic_BP'] = pd.to_numeric(bp_data[1], errors='coerce')
    # Convert history of hypertension to a numeric value.
    df_full['History_of_Hypertension'] = pd.to_numeric(df_full['History_of_Hypertension'], errors='coerce')
    # Create an interaction feature: Systolic BP * History of Hypertension.
    df_full['BP_x_Hypertension'] = df_full['Systolic_BP'] * df_full['History_of_Hypertension']

# --- One-Hot Encode Categorical Variables ---
# Convert the 'Gender' column (e.g., 'Male', 'Female') into numerical 0/1 columns.
# --- Clean Gender column explicitly before one-hot ---
if 'Gender' in df_full.columns:
    df_full['Gender'] = df_full['Gender'].astype(str).str.strip()
    df_full['Gender'] = df_full['Gender'].replace({'': np.nan, 'nan': np.nan})
    df_full = pd.get_dummies(df_full, columns=['Gender'], dummy_na=False, prefix='Gender')
    # Ensure both Gender_Female and Gender_Male exist
    for col in ['Gender_Female', 'Gender_Male']:
        if col not in df_full.columns:
            df_full[col] = 0

df_full['Gender_Female'] = df_full['Gender_Female'].astype(int)
df_full['Gender_Male'] = df_full['Gender_Male'].astype(int)

# --- Standardize all binary variables (e.g., histories, treatments) to 0/1 numeric format ---
binary_vars = [
    'mRS_Score_before_cerebral_hemorrhage', 'History_of_Hypertension', 'History_of_Stroke', 
    'History_of_Diabetes', 'History_of_Atrial_Fibrillation', 'History_of_Coronary_Artery_Disease', 
    'History_of_Smoking', 'History_of_Alcohol_Consumption', 'Ventricular_drainage_treatment', 
    'Hemostatic_treatment', 'Reducing_intracranial_pressure_treatment', 'Blood_pressure_lowering_treatment', 
    'Sedation_and_analgesic_treatment', 'Stop_vomiting_and_protect_stomach_treatment', 'Neurotrophic_treatment'
]
for col in binary_vars:
    if col in df_full.columns:
        # Convert to numeric, then fill any resulting missing values with 0.
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce').fillna(0)

# --- Data Cleaning ---
# Replace any infinite values (inf) that may have resulted from division with NaN.
df_full.replace([np.inf, -np.inf], np.nan, inplace=True)

print("✅ Feature engineering is complete.")

# ===================================================================
# --- Step 5: Final Feature Summary and Preview ---
# ===================================================================
# Define the final feature set for modeling.
# Select all columns from the table that have a numeric data type.
features_to_use = df_full.select_dtypes(include=np.number).columns.tolist()
# Remove identifiers and the target variable itself from the feature list.
if 'Hematoma_Expansion' in features_to_use:
    features_to_use.remove('Hematoma_Expansion')

# Create the final feature DataFrame.
final_features_df = df_full[features_to_use].copy()


# --- 1. Count and Print the Total Number of Features ---
pd.set_option('display.max_columns', 100)  
num_features = final_features_df.shape[1]
print("\n" + "="*60)
print("--- Final Feature Summary ---")
print(f"After data processing and feature engineering, the total number of features is: {num_features}")
print("="*60)
 

# --- 2. List the Specific Names of All Feature Variables ---
print("\nThe specific feature variable names are listed below:")
# To display more cleanly, print 5 features per line.
feature_names = final_features_df.columns.tolist()
for i in range(0, len(feature_names), 5):
    print("  ".join(feature_names[i:i+5]))
print("="*60)
  

# --- 3. Display a Preview of the Final Data (First 5 Rows) ---
print("\nPreview of the final feature data (first 5 rows):")
# .fillna(0) is used here for display purposes only, to make the preview table look cleaner.
# Note: For actual modeling, a more systematic imputation method (like filling with the median) should be used.
print(final_features_df.head().fillna(0))
print("="*60)

# Define feature set: use all numeric columns, exclude identifiers and the target
all_features = df_full.select_dtypes(include=np.number).columns.tolist()
all_features.remove('Hematoma_Expansion')

# Create the training set (first 100 patients) and the full dataset for prediction
train_ids = {f"sub{str(i).zfill(3)}" for i in range(1, 101)}
train_df = df_full[df_full['ID'].isin(train_ids) & df_full['Hematoma_Expansion'].notna()].copy()

X_train_full = train_df[all_features]
y_train = train_df['Hematoma_Expansion'].astype(int)

# This is the full dataset (160 patients) we will predict on
X_all = df_full[all_features]

print(f"✅ Data prepared for modeling. Training on {len(X_train_full)} patients.")

print("\n--- Script End ---")

In [ ]:
# ===============================================================
# Hematoma Expansion Prediction:
#   - Model 1: L1-Logistic (26 features) + Repeated CV + Bootstrap
#   - Model 2: GAM (20 features, ExtraTrees) + Repeated CV + Lambda stability + Bootstrap
#   - Model 3: Stacking (Logit + GAM + CatBoost, LGBM meta) + Repeated CV + Bootstrap
# ===============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Sklearn core
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import RFECV

# Pipelines / imbalance
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Tree / boosting models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier  # not used in this script, but kept if you want later
from catboost import CatBoostClassifier

# For GAM
from pygam import LogisticGAM

# For bootstrap
from sklearn.utils import resample

# Display options
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ===============================================================
# 1. Data Loading and Feature Engineering
#    (this section follows your current logic)
# ===============================================================
print("--- Script Start: Data Loading and Feature Engineering ---")

path_table1 = "Table1-Patient_List_and_Clinical_Information.xlsx"
path_table2 = "Table2-Patient_Imaging-Volume_and_Location.xlsx"
path_table3 = "Table3-Patient_Imaging-Shape_and_Gray_Distribution.xlsx"
path_label = "Question1_Answer1.xlsx"

try:
    # --- Table 1: Clinical information ---
    df_info = pd.read_excel(path_table1, dtype=str)
    clin_columns = [
        'Age', 'Gender', 'mRS_Score_before_cerebral_hemorrhage', 'History_of_Hypertension',
        'History_of_Stroke', 'History_of_Diabetes', 'History_of_Atrial_Fibrillation',
        'History_of_Coronary_Artery_Disease', 'History_of_Smoking', 'History_of_Alcohol_Consumption',
        'Time_from_Onset_to_First_Imaging', 'Blood_pressure', 'Ventricular_drainage_treatment',
        'Hemostatic_treatment', 'Reducing_intracranial_pressure_treatment',
        'Blood_pressure_lowering_treatment', 'Sedation_and_analgesic_treatment',
        'Stop_vomiting_and_protect_stomach_treatment', 'Neurotrophic_treatment'
    ]
    df_info_sub = df_info[['ID', 'First_SerialNumber'] + clin_columns].copy()
    df_info_sub['First_SerialNumber'] = df_info_sub['First_SerialNumber'].astype(str).str.strip()

    # --- Table 2: Volume and location (first scan only) ---
    df_vol = pd.read_excel(path_table2, dtype=str)
    cols_vol = df_vol.columns.tolist()
    if 'Follow_up_1_SerialNumber' in cols_vol:
        idx_end_first_scan = cols_vol.index('Follow_up_1_SerialNumber')
        first_scan_vol_cols = ['First_SerialNumber'] + cols_vol[2:idx_end_first_scan]
    else:
        first_scan_vol_cols = ['First_SerialNumber'] + cols_vol[2:]

    df_vol_first = df_vol[first_scan_vol_cols].copy()
    df_vol_first.rename(columns={'First_SerialNumber': 'SerialNumber'}, inplace=True)
    df_vol_first['SerialNumber'] = df_vol_first['SerialNumber'].astype(str).str.strip()

    # --- Table 3: Shape & gray distribution (Hemo + ED) ---
    df_shape_hemo = pd.read_excel(path_table3, sheet_name='Hemo', dtype=str).add_prefix('Hemo_')
    df_shape_ed = pd.read_excel(path_table3, sheet_name='ED', dtype=str).add_prefix('ED_')

    if 'Hemo_Unnamed: 0' in df_shape_hemo.columns:
        df_shape_hemo = df_shape_hemo.drop(columns=['Hemo_Unnamed: 0'])
    if 'ED_Unnamed: 0' in df_shape_ed.columns:
        df_shape_ed = df_shape_ed.drop(columns=['ED_Unnamed: 0'])

    df_shape_hemo.rename(columns={'Hemo_SerialNumber': 'SerialNumber'}, inplace=True)
    df_shape_ed.rename(columns={'ED_SerialNumber': 'SerialNumber'}, inplace=True)
    df_shape_hemo['SerialNumber'] = df_shape_hemo['SerialNumber'].astype(str).str.strip()
    df_shape_ed['SerialNumber'] = df_shape_ed['SerialNumber'].astype(str).str.strip()

    # --- Labels (hematoma expansion) ---
    df_label = pd.read_excel(path_label, dtype=str)
    df_label.rename(columns={'Whether_hematoma_dilation_occurs': 'Hematoma_Expansion'}, inplace=True)
    df_label['Hematoma_Expansion'] = pd.to_numeric(df_label['Hematoma_Expansion'], errors='coerce')

    print("✅ All source data tables loaded.")

except FileNotFoundError as e:
    print(f"❌ File not found. Details: {e}")
    raise
except Exception as e:
    print(f"❌ Error during data loading. Details: {e}")
    raise

# --- Merge all tables into one df_full ---
df_full = pd.merge(df_info_sub, df_vol_first,
                   left_on='First_SerialNumber', right_on='SerialNumber', how='left')
df_full = pd.merge(df_full, df_shape_hemo, on='SerialNumber', how='left')
df_full = pd.merge(df_full, df_shape_ed, on='SerialNumber', how='left')
df_full = pd.merge(df_full, df_label[['ID', 'Hematoma_Expansion']], on='ID', how='left')
df_full.drop(columns=['SerialNumber'], inplace=True)

print("✅ All tables merged into df_full.")

# ------------------------------
# Feature engineering
# ------------------------------
numeric_cols = ['Age', 'Time_from_Onset_to_First_Imaging', 'HM_volume', 'ED_volume'] + \
               [col for col in df_full.columns if 'Ratio' in col or 'Hemo_' in col or 'ED_' in col]

for col in numeric_cols:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce')

# ED/HM ratio
df_full['ED_HM_Ratio'] = df_full['ED_volume'] / (df_full['HM_volume'] + 1e-6)

# Blood pressure split
if 'Blood_pressure' in df_full.columns:
    bp_data = df_full['Blood_pressure'].str.split('/', expand=True)
    df_full['Systolic_BP'] = pd.to_numeric(bp_data[0], errors='coerce')
    df_full['Diastolic_BP'] = pd.to_numeric(bp_data[1], errors='coerce')
    df_full['History_of_Hypertension'] = pd.to_numeric(df_full['History_of_Hypertension'], errors='coerce')
    df_full['BP_x_Hypertension'] = df_full['Systolic_BP'] * df_full['History_of_Hypertension']

# Gender one-hot
if 'Gender' in df_full.columns:
    df_full['Gender'] = df_full['Gender'].astype(str).str.strip()
    df_full['Gender'] = df_full['Gender'].replace({'': np.nan, 'nan': np.nan})
    df_full = pd.get_dummies(df_full, columns=['Gender'], dummy_na=False, prefix='Gender')
    for col in ['Gender_Female', 'Gender_Male']:
        if col not in df_full.columns:
            df_full[col] = 0
    df_full['Gender_Female'] = df_full['Gender_Female'].astype(int)
    df_full['Gender_Male'] = df_full['Gender_Male'].astype(int)

# Binary variables standardization
binary_vars = [
    'mRS_Score_before_cerebral_hemorrhage', 'History_of_Hypertension', 'History_of_Stroke',
    'History_of_Diabetes', 'History_of_Atrial_Fibrillation', 'History_of_Coronary_Artery_Disease',
    'History_of_Smoking', 'History_of_Alcohol_Consumption', 'Ventricular_drainage_treatment',
    'Hemostatic_treatment', 'Reducing_intracranial_pressure_treatment', 'Blood_pressure_lowering_treatment',
    'Sedation_and_analgesic_treatment', 'Stop_vomiting_and_protect_stomach_treatment', 'Neurotrophic_treatment'
]
for col in binary_vars:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce').fillna(0)

df_full.replace([np.inf, -np.inf], np.nan, inplace=True)

print("✅ Feature engineering complete.")

# ------------------------------
# Final numeric feature list & train set
# ------------------------------
all_features = df_full.select_dtypes(include=np.number).columns.tolist()
if 'Hematoma_Expansion' in all_features:
    all_features.remove('Hematoma_Expansion')

# training: first 100 subjects (sub001 ~ sub100) with non-missing labels
train_ids = {f"sub{str(i).zfill(3)}" for i in range(1, 101)}
train_df = df_full[df_full['ID'].isin(train_ids) & df_full['Hematoma_Expansion'].notna()].copy()

X_train_full = train_df[all_features].copy()
y_train = train_df['Hematoma_Expansion'].astype(int)

X_all = df_full[all_features].copy()
ids_all = df_full['ID'].values

print(f"✅ Data prepared for modeling. Training on {len(X_train_full)} patients, "
      f"{len(all_features)} numeric features.")

# container for summary
results_summary = []

# ===============================================================
# 2. Model 1: L1-Logistic (26 features) + Repeated CV + Bootstrap
# ===============================================================
print("\n" + "=" * 70)
print("Model 1: L1-Logistic (26 features) - Repeated CV & Bootstrap")
print("=" * 70)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rfe_estimator = LogisticRegression(solver='liblinear', penalty='l1', random_state=42)

rfe_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('selector',
        RFECV(
            estimator=rfe_estimator,
            step=1,
            cv=cv_strategy,
            scoring='roc_auc',
            min_features_to_select=10,
            n_jobs=-1
        )
    )
])

print("[L1-Logit] Running RFECV to select features ...")
rfe_pipeline.fit(X_train_full, y_train)
selected_mask_logit = rfe_pipeline.named_steps['selector'].support_
selected_features_logit = X_train_full.columns[selected_mask_logit].tolist()
print(f"RFECV selected {len(selected_features_logit)} features initially.")

# manually keep top 26 (as in your previous setting)
selected_features_logit = selected_features_logit[:26]
print(f"Manually kept top {len(selected_features_logit)} features for L1-Logit:\n{selected_features_logit}")

X_logit = X_train_full[selected_features_logit].copy()

logit_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(
        solver='liblinear',
        penalty='l1',
        random_state=42,
        class_weight='balanced'
    ))
])

param_grid_logit = {
    'model__C': [0.001, 0.01, 0.1, 1, 10]
}

grid_search_logit = GridSearchCV(
    estimator=logit_pipeline,
    param_grid=param_grid_logit,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search_logit.fit(X_logit, y_train)

best_logit_model = grid_search_logit.best_estimator_
print(f"✅ Best params for L1-Logit: {grid_search_logit.best_params_}")
print(f"Best CV AUC (single 5-fold): {grid_search_logit.best_score_:.4f}")

# --- Single 5-fold OOF ---
y_proba_cv_logit = cross_val_predict(
    best_logit_model,
    X_logit,
    y_train,
    cv=cv_strategy,
    method='predict_proba'
)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_cv_logit)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold_idx = np.argmax(f1_scores[:-1])
best_threshold_logit = thresholds[best_threshold_idx]
y_pred_opt_logit = (y_proba_cv_logit >= best_threshold_logit).astype(int)

print("\n--- L1-Logit: Single 5-fold OOF Performance ---")
print(f"Optimal threshold: {best_threshold_logit:.4f}")
print(f"AUC: {roc_auc_score(y_train, y_proba_cv_logit):.4f}")
print(f"F1 (positive class): {f1_score(y_train, y_pred_opt_logit):.4f}")
print("Classification report:")
print(classification_report(y_train, y_pred_opt_logit, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_train, y_pred_opt_logit))

# --- Repeated CV (5x10) ---
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=2025)

print("\n[Robustness] L1-Logit - RepeatedStratifiedKFold (5x10) ...")
auc_scores_logit = cross_val_score(
    best_logit_model,
    X_logit,
    y_train,
    cv=rskf,
    scoring='roc_auc',
    n_jobs=-1
)

mean_auc_logit = auc_scores_logit.mean()
std_auc_logit = auc_scores_logit.std()
n_scores_logit = len(auc_scores_logit)
ci_low_logit = mean_auc_logit - 1.96 * std_auc_logit / np.sqrt(n_scores_logit)
ci_high_logit = mean_auc_logit + 1.96 * std_auc_logit / np.sqrt(n_scores_logit)

print("\n[L1-Logit] Repeated 5x10 CV:")
print(f"  Mean AUC = {mean_auc_logit:.4f}")
print(f"  Std AUC  = {std_auc_logit:.4f}")
print(f"  95% CI   ≈ [{ci_low_logit:.4f}, {ci_high_logit:.4f}]")

# --- Bootstrap (B=200) ---
print("\n[Robustness] L1-Logit - Bootstrap (B=200) ...")
n_bootstrap = 200
rng = np.random.RandomState(2025)
boot_aucs_logit = []

X_logit_boot = X_logit.reset_index(drop=True)
y_boot_base = y_train.reset_index(drop=True)

for b in range(n_bootstrap):
    idx = rng.randint(0, len(y_boot_base), len(y_boot_base))
    oob_mask = np.ones(len(y_boot_base), dtype=bool)
    oob_mask[np.unique(idx)] = False

    if oob_mask.sum() < 15:
        continue

    X_tr = X_logit_boot.iloc[idx]
    y_tr = y_boot_base.iloc[idx]
    X_oob = X_logit_boot[oob_mask]
    y_oob = y_boot_base[oob_mask]

    best_logit_model.fit(X_tr, y_tr)
    y_oob_proba = best_logit_model.predict_proba(X_oob)[:, 1]
    auc_b = roc_auc_score(y_oob, y_oob_proba)
    boot_aucs_logit.append(auc_b)

boot_aucs_logit = np.array(boot_aucs_logit)
mean_boot_logit = boot_aucs_logit.mean()
std_boot_logit = boot_aucs_logit.std()
ci_low_boot_logit = np.percentile(boot_aucs_logit, 2.5)
ci_high_boot_logit = np.percentile(boot_aucs_logit, 97.5)

print(f"\n[L1-Logit] Bootstrap AUC mean = {mean_boot_logit:.4f}, std = {std_boot_logit:.4f}")
print(f"Bootstrap 95% CI = [{ci_low_boot_logit:.4f}, {ci_high_boot_logit:.4f}]")

results_summary.append({
    "Model": "L1-Logit (26 feats)",
    "CV_mean_AUC": mean_auc_logit,
    "CV_std_AUC": std_auc_logit,
    "CV_CI_low": ci_low_logit,
    "CV_CI_high": ci_high_logit,
    "Boot_mean_AUC": mean_boot_logit,
    "Boot_std_AUC": std_boot_logit,
    "Boot_CI_low": ci_low_boot_logit,
    "Boot_CI_high": ci_high_boot_logit,
})
# ===============================================================
# Final Prediction for all patients (L1 Logistic)
# ===============================================================

print("\n[Logit] Training final model on the full selected training set ...")
best_logit_model.fit(X_logit, y_train)

X_all_logit = X_all[selected_features_logit]
logit_pred_prob = best_logit_model.predict_proba(X_all_logit)[:, 1]

df_pred_logit = pd.DataFrame({
    'ID': ids_all,
    'Predicted_Probability_Logit': logit_pred_prob
})
df_pred_logit.to_excel("Table4_Logit_Predictions.xlsx", index=False)
print("Saved: Table4_Logit_Predictions.xlsx")


# ===============================================================
# Visualization 1: Predicted probability distribution
# ===============================================================

import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))

plt.hist(logit_pred_prob[df_full['Hematoma_Expansion']==0], bins=20, alpha=0.6, label='No Expansion', color='blue')
plt.hist(logit_pred_prob[df_full['Hematoma_Expansion']==1], bins=20, alpha=0.6, label='Expansion', color='red')

plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("L1-Logit Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()


# ===============================================================
# Visualization 2: Risk stratification curve
# ===============================================================

plt.figure(figsize=(8,5))
sorted_prob = np.sort(logit_pred_prob)

plt.plot(sorted_prob, marker='.', linestyle='-', linewidth=1)
plt.ylabel("Predicted probability")
plt.xlabel("Patients (sorted)")
plt.title("Risk Stratification Plot - L1 Logistic Regression")
plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Model 2: Extra Trees Classifier (10 fixed features)
#           with Repeated CV + Bootstrap Robustness Analysis
# ===============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score,
    cross_val_predict
)
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.utils import resample

# ===============================================================
# 1. Select the fixed 10 ExtraTrees features
# ===============================================================

selected_features_et = [
    'Hemo_original_shape_Elongation',
    'History_of_Coronary_Artery_Disease',
    'Time_from_Onset_to_First_Imaging',
    'ED_PCA_L_Ratio',
    'Hemo_NCCT_original_firstorder_Variance',
    'History_of_Stroke',
    'Hemo_NCCT_original_firstorder_RobustMeanAbsoluteDeviation',
    'ED_original_shape_MinorAxisLength',
    'Hemo_NCCT_original_firstorder_MeanAbsoluteDeviation',
    'ED_NCCT_original_firstorder_10Percentile'
]

X_et = X_train_full[selected_features_et].copy()
y_et = y_train.copy()

print("Selected ExtraTrees features:")
print(selected_features_et)
print(f"Training set: {X_et.shape[0]} samples, {X_et.shape[1]} features.")

# ===============================================================
# 2. ExtraTrees Pipeline
# ===============================================================

et_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42)),
    ('model', ExtraTreesClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])


# ===============================================================
# 3. Single 5-fold OOF Evaluation
# ===============================================================

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_proba_oof = cross_val_predict(
    et_pipeline,
    X_et,
    y_et,
    cv=cv5,
    method="predict_proba"
)[:, 1]

prec, rec, thr = precision_recall_curve(y_et, y_proba_oof)
f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thr = thr[np.argmax(f1s[:-1])]
y_pred_opt = (y_proba_oof >= best_thr).astype(int)

print("\n================= ExtraTrees: Single 5-fold Results =================")
print(f"Optimal threshold: {best_thr:.4f}")
print(f"AUC: {roc_auc_score(y_et, y_proba_oof):.4f}")
print(f"F1:  {f1_score(y_et, y_pred_opt):.4f}")
print("\nClassification Report:")
print(classification_report(y_et, y_pred_opt, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_et, y_pred_opt))


# ===============================================================
# 4. RepeatedStratifiedKFold (5 × 10 = 50 runs)
# ===============================================================

rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=2025
)

print("\n================= ExtraTrees: Repeated CV (5×10) =================")

auc_scores_cv = cross_val_score(
    et_pipeline,
    X_et,
    y_et,
    cv=rskf,
    scoring="roc_auc",
    n_jobs=-1
)

mean_auc = auc_scores_cv.mean()
std_auc = auc_scores_cv.std()
n_scores = len(auc_scores_cv)
ci_low = mean_auc - 1.96 * std_auc / np.sqrt(n_scores)
ci_high = mean_auc + 1.96 * std_auc / np.sqrt(n_scores)

print(f"Mean AUC: {mean_auc:.4f}")
print(f"Std AUC:  {std_auc:.4f}")
print(f"95% CI:   [{ci_low:.4f}, {ci_high:.4f}]")


# ===============================================================
# 5. Bootstrap Analysis (B = 200)
# ===============================================================

print("\n================= ExtraTrees: Bootstrap (B=200) =================")

B = 200
rng = np.random.RandomState(2025)
boot_aucs = []

Xb = X_et.reset_index(drop=True)
yb = y_et.reset_index(drop=True)

for b in range(B):
    idx = rng.randint(0, len(yb), len(yb))
    oob_mask = np.ones(len(yb), dtype=bool)
    oob_mask[np.unique(idx)] = False

    if oob_mask.sum() < 15:
        continue

    X_train_b = Xb.iloc[idx]
    y_train_b = yb.iloc[idx]
    X_oob = Xb[oob_mask]
    y_oob = yb[oob_mask]

    et_pipeline.fit(X_train_b, y_train_b)
    p_oob = et_pipeline.predict_proba(X_oob)[:, 1]

    boot_auc = roc_auc_score(y_oob, p_oob)
    boot_aucs.append(boot_auc)

boot_aucs = np.array(boot_aucs)
boot_mean = boot_aucs.mean()
boot_std = boot_aucs.std()
boot_low = np.percentile(boot_aucs, 2.5)
boot_high = np.percentile(boot_aucs, 97.5)

print(f"Bootstrap Mean AUC: {boot_mean:.4f}")
print(f"Bootstrap Std AUC:  {boot_std:.4f}")
print(f"95% CI: [{boot_low:.4f}, {boot_high:.4f}]")


# ===============================================================
# 6. Final Model Fit and Prediction for All 160 Patients
# ===============================================================

print("\n================= ExtraTrees: Final Prediction =================")

et_pipeline.fit(X_et, y_et)
X_all_et = X_all[selected_features_et]
final_probs_et = et_pipeline.predict_proba(X_all_et)[:, 1]

final_output_et = pd.DataFrame({
    "ID": df_full["ID"].unique(),
    "ExtraTrees_Probability": final_probs_et
})

final_output_et.to_excel("Table4_ExtraTrees_Final_Predictions.xlsx", index=False)

print("Final predictions saved to: Table4_ExtraTrees_Final_Predictions.xlsx")

# ===============================================================
# 7. Visualization: Predicted Probability Distribution
# ===============================================================

import matplotlib.pyplot as plt

# Identify labels for plotting
y_all = df_full['Hematoma_Expansion']
valid_mask = ~y_all.isna()

plt.figure(figsize=(8, 5))
plt.hist(final_probs_et[(y_all == 0) & valid_mask], bins=20, alpha=0.6, label='No Expansion', color='blue')
plt.hist(final_probs_et[(y_all == 1) & valid_mask], bins=20, alpha=0.6, label='Expansion', color='red')

plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("ExtraTrees: Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()


# ===============================================================
# 8. Visualization: Risk Stratification Curve
# ===============================================================

plt.figure(figsize=(8, 5))
sorted_probs = np.sort(final_probs_et)

plt.plot(sorted_probs, marker='.', linestyle='-', linewidth=1, color='purple')
plt.ylabel("Predicted probability")
plt.xlabel("Patients (sorted)")
plt.title("ExtraTrees: Risk Stratification Curve")
plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Model: CatBoost (20 selected features)
#        with Repeated CV + Bootstrap Robustness Analysis
# ===============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score,
    cross_val_predict
)
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    classification_report,
    confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.utils import resample

from catboost import CatBoostClassifier


# ===============================================================
# 1. FIXED SELECTED CATBOOST FEATURES (from your earlier run)
# ===============================================================

selected_features_cat = [
    'Hemo_original_shape_Elongation',
    'History_of_Coronary_Artery_Disease',
    'Time_from_Onset_to_First_Imaging',
    'ED_PCA_L_Ratio',
    'Hemo_NCCT_original_firstorder_Variance',
    'History_of_Stroke',
    'Hemo_NCCT_original_firstorder_RobustMeanAbsoluteDeviation',
    'ED_original_shape_MinorAxisLength',
    'Hemo_NCCT_original_firstorder_MeanAbsoluteDeviation',
    'ED_NCCT_original_firstorder_10Percentile',
    # Additional 10 features (20 total)
    'HM_volume',
    'ED_volume',
    'ED_original_shape_SurfaceArea',
    'Hemo_original_shape_SurfaceArea',
    'ED_original_shape_MeshVolume',
    'Hemo_original_shape_MeshVolume',
    'Hemo_NCCT_original_firstorder_Energy',
    'ED_NCCT_original_firstorder_Energy',
    'ED_NCCT_original_firstorder_Mean',
    'Hemo_NCCT_original_firstorder_Mean'
]

X_cat = X_train_full[selected_features_cat].copy()
y_cat = y_train.copy()

print("Selected CatBoost features:")
print(selected_features_cat)
print(f"Training samples = {X_cat.shape[0]}, features = {X_cat.shape[1]}")


# ===============================================================
# 2. CatBoost Pipeline
# ===============================================================

cat_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42)),
    ('model', CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=0.05,
        l2_leaf_reg=3,
        random_seed=42,
        verbose=0,
        auto_class_weights='Balanced'
    ))
])


# ===============================================================
# 3. Single 5-fold OOF Evaluation
# ===============================================================

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_proba_oof = cross_val_predict(
    cat_pipeline,
    X_cat,
    y_cat,
    cv=cv5,
    method="predict_proba"
)[:, 1]

prec, rec, thr = precision_recall_curve(y_cat, y_proba_oof)
f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thr = thr[np.argmax(f1s[:-1])]
y_pred_opt = (y_proba_oof >= best_thr).astype(int)

print("\n================= CatBoost: Single 5-fold Results =================")
print(f"Optimal threshold = {best_thr:.4f}")
print(f"AUC = {roc_auc_score(y_cat, y_proba_oof):.4f}")
print(f"F1  = {f1_score(y_cat, y_pred_opt):.4f}")
print("\nClassification Report:")
print(classification_report(y_cat, y_pred_opt, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_cat, y_pred_opt))


# ===============================================================
# 4. RepeatedStratifiedKFold (5 × 10 = 50 runs)
# ===============================================================

rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=2025
)

print("\n================= CatBoost: Repeated CV (5×10) =================")

auc_scores_cv = cross_val_score(
    cat_pipeline,
    X_cat,
    y_cat,
    cv=rskf,
    scoring="roc_auc",
    n_jobs=-1
)

mean_auc = auc_scores_cv.mean()
std_auc = auc_scores_cv.std()
n_scores = len(auc_scores_cv)
ci_low = mean_auc - 1.96 * std_auc / np.sqrt(n_scores)
ci_high = mean_auc + 1.96 * std_auc / np.sqrt(n_scores)

print(f"Mean AUC = {mean_auc:.4f}")
print(f"Std AUC  = {std_auc:.4f}")
print(f"95% CI   = [{ci_low:.4f}, {ci_high:.4f}]")


# ===============================================================
# 5. Bootstrap Analysis (B = 200)
# ===============================================================

print("\n================= CatBoost: Bootstrap (B=200) =================")

B = 200
rng = np.random.RandomState(2025)
boot_aucs = []

Xb = X_cat.reset_index(drop=True)
yb = y_cat.reset_index(drop=True)

for b in range(B):
    idx = rng.randint(0, len(yb), len(yb))
    oob_mask = np.ones(len(yb), dtype=bool)
    oob_mask[np.unique(idx)] = False

    if oob_mask.sum() < 15:
        continue

    X_train_b = Xb.iloc[idx]
    y_train_b = yb.iloc[idx]
    X_oob = Xb[oob_mask]
    y_oob = yb[oob_mask]

    cat_pipeline.fit(X_train_b, y_train_b)
    p_oob = cat_pipeline.predict_proba(X_oob)[:, 1]

    boot_auc = roc_auc_score(y_oob, p_oob)
    boot_aucs.append(boot_auc)

boot_aucs = np.array(boot_aucs)
boot_mean = boot_aucs.mean()
boot_std = boot_aucs.std()
boot_low = np.percentile(boot_aucs, 2.5)
boot_high = np.percentile(boot_aucs, 97.5)

print(f"Bootstrap Mean AUC = {boot_mean:.4f}")
print(f"Bootstrap Std AUC  = {boot_std:.4f}")
print(f"95% CI = [{boot_low:.4f}, {boot_high:.4f}]")


# ===============================================================
# 6. Final Prediction for All 160 Patients
# ===============================================================

print("\n================= CatBoost: Final Predictions =================")

cat_pipeline.fit(X_cat, y_cat)

X_all_cat = X_all[selected_features_cat]
final_probs = cat_pipeline.predict_proba(X_all_cat)[:, 1]

output_cat = pd.DataFrame({
    "ID": df_full["ID"].unique(),
    "CatBoost_Probability": final_probs
})

output_cat.to_excel("Table4_CatBoost_Final_Predictions.xlsx", index=False)

print("Final predictions saved to: Table4_CatBoost_Final_Predictions.xlsx")

# ===============================================================
# 7. Visualization: Predicted Probability Distribution
# ===============================================================

import matplotlib.pyplot as plt

# Extract ground-truth labels for plotting
y_all = df_full['Hematoma_Expansion']
valid_mask = ~y_all.isna()

plt.figure(figsize=(8, 5))
plt.hist(final_probs[(y_all == 0) & valid_mask], bins=20, alpha=0.6, 
         label='No Expansion', color='blue')
plt.hist(final_probs[(y_all == 1) & valid_mask], bins=20, alpha=0.6, 
         label='Expansion', color='red')

plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("CatBoost: Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()


# ===============================================================
# 8. Visualization: Risk Stratification Curve
# ===============================================================

plt.figure(figsize=(8, 5))
sorted_probs = np.sort(final_probs)

plt.plot(sorted_probs, marker='.', linestyle='-', linewidth=1, color='green')
plt.ylabel("Predicted probability")
plt.xlabel("Patients (sorted)")
plt.title("CatBoost: Risk Stratification Curve")
plt.tight_layout()
plt.show()

In [ ]:
# ===============================================================
# 3. Model 2: GAM – with feature selection, CV tuning and robustness
# ===============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pygam import LogisticGAM
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    classification_report,
)

print("\n" + "="*70)
print("Model 2: Generalized Additive Model (GAM)")
print("="*70)

# -----------------------------------------------------------------
# 3.1 Prepare training / full data (reuse df_full, train_df, all_features)
# -----------------------------------------------------------------
# train_df: subset of first 100 patients with non-missing labels
# all_features: numeric features (excluding Hematoma_Expansion)
# df_full: full 160-patient dataframe

X_train_full = train_df[all_features].copy()
y_train = train_df["Hematoma_Expansion"].astype(int).copy()

X_all = df_full[all_features].copy()
ids_all = df_full["ID"].values

print(f"✅ GAM: training on {len(X_train_full)} patients, "
      f"{X_train_full.shape[1]} numeric features available.")

# -----------------------------------------------------------------
# 3.2 Feature selection using Extra Trees (top 20 features)
# -----------------------------------------------------------------
print("\n[GAM] Step 1 – Extra Trees feature selection ...")

# Impute all numeric features (for ExtraTrees only)
fs_imputer = SimpleImputer(strategy="median")
X_train_imputed_fs = fs_imputer.fit_transform(X_train_full)

et_clf = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
et_clf.fit(X_train_imputed_fs, y_train)

importances = et_clf.feature_importances_
feat_imp_df = pd.DataFrame(
    {"feature": X_train_full.columns, "importance": importances}
).sort_values(by="importance", ascending=False)

TOP_N = 20
selected_features = feat_imp_df.head(TOP_N)["feature"].tolist()

print(f"✅ Selected top {TOP_N} features for GAM:")
for i in range(0, len(selected_features), 5):
    print("   " + ", ".join(selected_features[i:i+5]))

# Restrict train / full data to selected features
X_train_sel = X_train_full[selected_features].copy()
X_all_sel = X_all[selected_features].copy()

# -----------------------------------------------------------------
# 3.3 Imputation + Scaling (for GAM)
# -----------------------------------------------------------------
print("\n[GAM] Step 2 – Imputation and scaling ...")

gam_imputer = SimpleImputer(strategy="median")
X_train_imputed = gam_imputer.fit_transform(X_train_sel)
X_all_imputed = gam_imputer.transform(X_all_sel)

gam_scaler = StandardScaler()
X_train_processed = gam_scaler.fit_transform(X_train_imputed)
X_all_processed = gam_scaler.transform(X_all_imputed)

print("✅ Imputation + scaling complete.")

# -----------------------------------------------------------------
# 3.4 Hyperparameter tuning for λ with 5-fold CV (global best λ)
# -----------------------------------------------------------------
print("\n[GAM] Step 3 – Tuning λ with 5-fold stratified CV ...")

lam_values = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_lam = None
best_score = -np.inf

for lam in lam_values:
    cv_scores = []
    for train_idx, val_idx in cv.split(X_train_processed, y_train):
        X_tr = X_train_processed[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_val = X_train_processed[val_idx]
        y_val = y_train.iloc[val_idx]

        try:
            gam = LogisticGAM(lam=lam, max_iter=5000).fit(X_tr, y_tr)
            y_val_proba = gam.predict_proba(X_val)
            auc = roc_auc_score(y_val, y_val_proba)
            cv_scores.append(auc)
        except Exception as e:
            # If optimization fails for a specific λ, record 0 so it won't be chosen
            print(f"   ⚠️  Optimization failed for lam={lam}: {e}")
            cv_scores.append(0.0)

    mean_auc = float(np.mean(cv_scores))
    print(f"   lam={lam:.2f} | mean CV AUC={mean_auc:.4f}")

    if mean_auc > best_score:
        best_score = mean_auc
        best_lam = lam

print(f"\n✅ Global best λ for GAM: {best_lam} "
      f"(5-fold CV AUC={best_score:.4f})")

# -----------------------------------------------------------------
# 3.5 Single 5-fold OOF prediction using best λ (for main results)
# -----------------------------------------------------------------
print("\n[GAM] Step 4 – Single 5-fold OOF predictions with best λ ...")

y_probas_oof = np.zeros(len(y_train))

for train_idx, val_idx in cv.split(X_train_processed, y_train):
    X_tr = X_train_processed[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train_processed[val_idx]

    gam = LogisticGAM(lam=best_lam, max_iter=5000).fit(X_tr, y_tr)
    y_probas_oof[val_idx] = gam.predict_proba(X_val)

# Choose optimal threshold using F1 on OOF probabilities
precisions, recalls, thresholds = precision_recall_curve(y_train, y_probas_oof)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

if len(thresholds) > 0:
    best_idx = np.argmax(f1_scores[:-1])  # last F1 corresponds to no threshold
    best_threshold = float(thresholds[best_idx])
else:
    best_threshold = 0.5

y_pred_oof = (y_probas_oof >= best_threshold).astype(int)

print("\n--- GAM: Single 5-fold OOF Performance ---")
print(f"Optimal probability threshold: {best_threshold:.4f}")
print(f"AUC: {roc_auc_score(y_train, y_probas_oof):.4f}")
print(f"F1 (positive class): {f1_score(y_train, y_pred_oof):.4f}")
print("\nClassification Report:")
print(classification_report(y_train, y_pred_oof, digits=4))

# -----------------------------------------------------------------
# 3.6 Robustness check: Repeated CV + λ stability (5×5)
# -----------------------------------------------------------------
print("\n[GAM] Step 5 – Robustness: RepeatedStratifiedKFold (5×5) and λ stability ...")

rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=2025
)

auc_scores_rep = []
best_lams_per_split = []

for fold_id, (train_idx, val_idx) in enumerate(rskf.split(X_train_processed, y_train), start=1):
    X_tr = X_train_processed[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train_processed[val_idx]
    y_val = y_train.iloc[val_idx]

    # inner tuning of λ on current training fold
    inner_best_lam = None
    inner_best_auc = -np.inf

    for lam in lam_values:
        try:
            gam_inner = LogisticGAM(lam=lam, max_iter=5000).fit(X_tr, y_tr)
            y_val_proba = gam_inner.predict_proba(X_val)
            auc_lam = roc_auc_score(y_val, y_val_proba)
        except Exception as e:
            auc_lam = 0.0

        if auc_lam > inner_best_auc:
            inner_best_auc = auc_lam
            inner_best_lam = lam

    best_lams_per_split.append(inner_best_lam)

    # evaluate using best λ for this split
    gam_best = LogisticGAM(lam=inner_best_lam, max_iter=5000).fit(X_tr, y_tr)
    y_val_proba_best = gam_best.predict_proba(X_val)
    auc_best = roc_auc_score(y_val, y_val_proba_best)
    auc_scores_rep.append(auc_best)

# Repeated CV AUC statistics
auc_scores_rep = np.array(auc_scores_rep)
mean_auc_rep = float(auc_scores_rep.mean())
std_auc_rep = float(auc_scores_rep.std())
n_rep = len(auc_scores_rep)
ci_low_rep = mean_auc_rep - 1.96 * std_auc_rep / np.sqrt(n_rep)
ci_high_rep = mean_auc_rep + 1.96 * std_auc_rep / np.sqrt(n_rep)

print("\n[GAM] Repeated 5×5 CV results:")
print(f"  Mean AUC = {mean_auc_rep:.4f}")
print(f"  Std  AUC = {std_auc_rep:.4f}")
print(f"  95% CI   ≈ [{ci_low_rep:.4f}, {ci_high_rep:.4f}]")

# λ stability summary
best_lams_per_split = np.array(best_lams_per_split)
unique_lams, lam_counts = np.unique(best_lams_per_split, return_counts=True)

print("\n[GAM] λ stability across all repeated CV splits:")
for lam_val, cnt in zip(unique_lams, lam_counts):
    print(f"  λ = {lam_val:.2f}  → selected {int(cnt)} times")

# -----------------------------------------------------------------
# 3.7 Fit final GAM on full training set and predict all 160 patients
# -----------------------------------------------------------------
print("\n[GAM] Step 6 – Train final GAM on full training set and predict all patients ...")

final_gam = LogisticGAM(lam=best_lam, max_iter=5000).fit(X_train_processed, y_train)

final_probs_all = final_gam.predict_proba(X_all_processed)

gam_pred_df = pd.DataFrame({
    "ID": ids_all,
    "GAM_probability_of_hematoma_expansion": final_probs_all
})

output_file_gam = "GAM_Hematoma_Expansion_Predictions_Robust.xlsx"
gam_pred_df.to_excel(output_file_gam, index=False)

print(f"✅ Final GAM predictions saved to '{output_file_gam}'.")
print("✅ GAM model (with CV tuning + robustness check) completed.")

In [ ]:
#!/usr/bin/env python
"""
Moudel2.1 Edema Progression: Seven-Model Joint Pipeline (Final, 6-Column Data In)
=============================================================================

**Data expectation** (already validated & cleaned by you): a *long-format* CSV/Excel with
columns exactly:

    ID, SerialNumber, VisitIndex, ED_volume, Time_from_Onset_hours, log_time

- `Time_from_Onset_hours` is numeric (float hours from onset to scan).
- `log_time` = log(Time_from_Onset_hours + LOG_OFFSET) already present (we will recheck & recompute if needed).
- One row per *patient-visit*.
- Example shape: (450, 6).

**Preprocessing strategies**
  *MethodA* – Outlier filtering (hard caps from paper):
      Time_from_Onset_hours <= OUTLIER_X_CAP  &  ED_volume <= OUTLIER_Y_CAP.
      (Optional IQR refinement hook left in place but default False.)
  *MethodB* – Log-time transform (retain all points):
      Keep rows with Time_from_Onset_hours > -LOG_OFFSET, then compute/recompute log_time.

**Model families (7)** evaluated under both preprocessing strategies:
  1. Poly(d)            – global polynomial regression, d ∈ POLY_DEGREES.
  2. Spline(df)         – cubic B-spline regression, df ∈ SPLINE_DFS (patsy if available; fallback manual basis).
  3. Kernel(h_frac)     – Gaussian Nadaraya–Watson smoother; bandwidth = frac * scale(x) (IQR or SD).
  4. LOESS(frac)        – statsmodels LOWESS local regression; frac ∈ LOESS_FRACS.
  5. GAM(lam)           – pygam.LinearGAM(s(0)) smoothing splines; smoothing λ ∈ GAM_LAMS.
  6. GPR(ls)            – Gaussian Process Regression (RBF length-scale = ls * scale(x) + White noise).
  7. Logistic(4p)       – 4‑parameter logistic growth curve fit via non-linear least squares.

For each (Preprocess × ModelHyper) combination we compute:
  - CV_RMSE (K-fold, default 5) – *model selection metric*
  - In-sample metrics on full preprocessed data: RMSE, R2, AIC, BIC

Best model per preprocessing = row with minimum CV_RMSE (tie → smaller Hyper).

**Outputs** (in OUTDIR):
  - q2a_seven_xy_all_clean.csv                (cleaned input)
  - q2a_seven_xy_methodA.csv / _removed.csv   (after outlier filtering)
  - q2a_seven_xy_methodB.csv                   (after log-time)
  - q2a_seven_metrics.csv                      (full metrics table)
  - q2a_seven_best_models_summary.csv          (best per preprocess)
  - q2a_seven_methodA_poly_coefs.csv           (if bestA is Poly)
  - q2a_seven_methodB_poly_coefs.csv           (if bestB is Poly)
  - q2a_seven_predictions_all.csv              (prediction & residuals, both best models)
  - q2a_seven_residual_by_patient.csv          (per-patient mean residual from MethodA best)
  - Plots: raw scatter, MethodA scatter, MethodB scatter (log + orig), compare best, curves, residual hist.

Axis policy for raw-time plots:
  *Start slightly below 0* so dense early-time points are visible.
  Enforce at least 0→X_CAP_MIN (default 5,000 h) horizontally and 0→Y_CAP_MIN (default 175,000 ml) vertically.
  Automatically extend if data exceed those caps.

---
**Quick start**
```bash
pip install numpy pandas matplotlib patsy statsmodels pygam scikit-learn scipy
python q2a_seven_models.py  # if you save this file under that name
```
Optionally specify a custom input: `python q2a_seven_models.py --in myfile.csv`.
---
"""

from __future__ import annotations

import os
import re
import argparse
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Optional dependencies (soft imports with fallbacks)
# ------------------------------------------------------------------
_HAS_PATSY = False
_HAS_STATS_LOWESS = False
_HAS_PYGAM = False
_HAS_SK_GPR = False
_HAS_SCIPY = False

try:
    import patsy  # type: ignore
    _HAS_PATSY = True
except Exception:  # pragma: no cover
    pass

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess  # type: ignore
    _HAS_STATS_LOWESS = True
except Exception:  # pragma: no cover
    pass

try:
    from pygam import LinearGAM, s  # type: ignore
    _HAS_PYGAM = True
except Exception:  # pragma: no cover
    pass

try:
    from sklearn.gaussian_process import GaussianProcessRegressor  # type: ignore
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel  # type: ignore
    _HAS_SK_GPR = True
except Exception:  # pragma: no cover
    pass

try:
    from scipy.optimize import curve_fit  # type: ignore
    _HAS_SCIPY = True
except Exception:  # pragma: no cover
    pass


# =============================================================================
# CONFIG
# =============================================================================
PATH_XY_IN = "q2a_joint_outputs/q2a_joint_xy_all.csv"  # change if needed
OUTDIR = "q2a_seven_models_outputs"

# preprocess params
LOG_OFFSET = 1.0
OUTLIER_X_CAP = 2000.0     # hours
OUTLIER_Y_CAP = 100000.0   # ml
USE_IQR_REFINE = False     # optional Tukey after caps (not used by default)

# model grids
POLY_DEGREES   = [2, 3, 4, 5, 6]
SPLINE_DFS     = [3, 4, 5, 6]
KERNEL_FRACS   = [0.1, 0.3, 0.5, 0.8, 1.0]
LOESS_FRACS    = [0.25, 0.5, 0.75]
GAM_LAMS       = [0.1, 1.0, 10.0]   # smoothing lambda grid
GPR_LS_FRACS   = [0.1, 0.3, 0.5, 1.0]  # length-scale multipliers
LOGISTIC_TAG   = "4p"  # single hyper choice (no grid)

# CV
N_SPLITS = 5
RANDOM_STATE = 42
SHUFFLE_KFOLD = True

# plotting appearance
POINT_SIZE = 32
ALPHA_ALL = 0.5
ALPHA_REMOVED = 0.75
ALPHA_KEPT = 0.65

# axis policy
X_CAP_MIN = 5000.0
Y_CAP_MIN = 175000.0
NEG_MARGIN_PCT = 0.05  # extend below 0 by 5% of data-range (or at least NEG_MARGIN_MIN)
NEG_MARGIN_MIN_X = 50.0
NEG_MARGIN_MIN_Y = 50.0
PAD_POS_FRAC = 0.05

# =============================================================================
# IO helpers
# =============================================================================

def _ensure_outdir(path: str = OUTDIR):
    os.makedirs(path, exist_ok=True)
    return path


def _join_out(name: str) -> str:
    return os.path.join(OUTDIR, name)


# =============================================================================
# Data ingestion (6-column long format)
# =============================================================================

def read_long_xy(path_xy: str, log_offset: float = LOG_OFFSET) -> pd.DataFrame:
    """Read the validated long-format dataset and coerce dtypes.
    Adds / re-computes log_time if missing or stale.
    """
    if not os.path.exists(path_xy):
        raise FileNotFoundError(f"File not found: {path_xy}")
    if path_xy.lower().endswith(".csv"):
        df = pd.read_csv(path_xy)
    else:
        df = pd.read_excel(path_xy)

    req = ["ID", "SerialNumber", "VisitIndex", "ED_volume", "Time_from_Onset_hours"]
    for c in req:
        if c not in df.columns:
            raise ValueError(f"Required column missing: {c}")

    df = df.copy()
    df["ED_volume"] = pd.to_numeric(df["ED_volume"], errors="coerce")
    df["Time_from_Onset_hours"] = pd.to_numeric(df["Time_from_Onset_hours"], errors="coerce")
    df["VisitIndex"] = pd.to_numeric(df["VisitIndex"], errors="coerce")

    df = df.dropna(subset=["ED_volume", "Time_from_Onset_hours", "VisitIndex"])  # drop rows missing core data
    df["VisitIndex"] = df["VisitIndex"].astype(int)

    # ensure strictly finite
    df = df[np.isfinite(df["ED_volume"]) & np.isfinite(df["Time_from_Onset_hours"])]

    # (Re)compute log_time to guarantee consistency
    df["log_time"] = np.log(np.maximum(df["Time_from_Onset_hours"].values, 0.0) + log_offset)

    return df.reset_index(drop=True)


# =============================================================================
# Preprocessing variants
# =============================================================================

def methodA_outlier(df: pd.DataFrame,
                    x_cap: float = OUTLIER_X_CAP,
                    y_cap: float = OUTLIER_Y_CAP,
                    use_iqr: bool = USE_IQR_REFINE) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Hard-cap outlier filter (paper). Optional Tukey IQR refine."""
    mask_cap = (df["Time_from_Onset_hours"] <= x_cap) & (df["ED_volume"] <= y_cap)
    kept = df[mask_cap].copy()
    removed = df[~mask_cap].copy()
    removed["drop_reason"] = "cap"
    if use_iqr and not kept.empty:
        q1x, q3x = np.percentile(kept["Time_from_Onset_hours"], [25, 75])
        q1y, q3y = np.percentile(kept["ED_volume"], [25, 75])
        mask_iqr = (
            kept["Time_from_Onset_hours"].between(q1x - 1.5*(q3x-q1x), q3x + 1.5*(q3x-q1x)) &
            kept["ED_volume"].between(q1y - 1.5*(q3y-q1y), q3y + 1.5*(q3y-q1y))
        )
        removed_iqr = kept[~mask_iqr].copy(); removed_iqr["drop_reason"] = "iqr"
        removed = pd.concat([removed, removed_iqr], ignore_index=True)
        kept = kept[mask_iqr].copy()
    return kept.reset_index(drop=True), removed.reset_index(drop=True)


def methodB_log(df: pd.DataFrame, offset: float = LOG_OFFSET) -> pd.DataFrame:
    """Retain all rows with valid (x + offset) > 0 and compute log_time."""
    dfc = df[df["Time_from_Onset_hours"] > -offset].copy()
    dfc["log_time"] = np.log(np.maximum(dfc["Time_from_Onset_hours"].values, 0.0) + offset)
    dfc = dfc[np.isfinite(dfc["log_time"])]
    return dfc.reset_index(drop=True)


# =============================================================================
# Metrics helper
# =============================================================================

def _insample_metrics(y_true: np.ndarray, y_pred: np.ndarray, n_params: int) -> Dict[str, float]:
    resid = y_true - y_pred
    n = len(y_true)
    sse = float(np.sum(resid**2))
    rmse = float(np.sqrt(sse / n))
    sst = float(np.sum((y_true - np.mean(y_true))**2))
    r2 = float(1 - sse / sst) if sst > 0 else np.nan
    aic = float(n * np.log(sse / n) + 2 * n_params) if n > n_params and sse > 0 else np.nan
    bic = float(n * np.log(sse / n) + n_params * np.log(n)) if n > n_params and sse > 0 else np.nan
    return {"RMSE": rmse, "R2": r2, "AIC": aic, "BIC": bic}


# =============================================================================
# Modeling primitives: POLY / SPLINE / KERNEL
# =============================================================================

def _poly_design(x: np.ndarray, deg: int) -> np.ndarray:
    return np.column_stack([x**i for i in range(1, deg+1)])


def _fit_poly(x: np.ndarray, y: np.ndarray, deg: int):
    from sklearn.linear_model import LinearRegression
    X = _poly_design(x, deg)
    m = LinearRegression().fit(X, y)
    return m


def _pred_poly(model, x: np.ndarray, deg: int) -> np.ndarray:
    X = _poly_design(x, deg)
    return model.predict(X)


def _poly_nparams(deg: int) -> int:
    return deg + 1  # intercept + d terms


# ---- Cubic B-spline regression ----

def _spline_design(x: np.ndarray, df: int, degree: int = 3) -> np.ndarray:
    if _HAS_PATSY:
        X = patsy.dmatrix(f"bs(x, df={df}, degree={degree}, include_intercept=False)", {"x": x}, return_type="dataframe")
        return np.asarray(X)
    # fallback: simple truncated-power basis
    xs = np.asarray(x, dtype=float)
    knots = np.quantile(xs, np.linspace(0, 1, df+2)[1:-1])  # interior df knots
    X = [xs, xs**2, xs**3]
    for k in knots:
        X.append(np.maximum(0, xs - k)**3)
    return np.column_stack(X)


def _fit_spline(x: np.ndarray, y: np.ndarray, df: int, degree: int = 3):
    from sklearn.linear_model import LinearRegression
    X = _spline_design(x, df, degree=degree)
    m = LinearRegression().fit(X, y)
    return m


def _pred_spline(model, x: np.ndarray, df: int, degree: int = 3) -> np.ndarray:
    X = _spline_design(x, df, degree=degree)
    return model.predict(X)


def _spline_nparams(df: int) -> int:
    if _HAS_PATSY:
        return df + 1
    return 1 + 3 + df  # intercept + poly + knots


# ---- Gaussian KERNEL (Nadaraya–Watson) ----

def _kernel_scale(x: np.ndarray, mode: str = "iqr") -> float:
    if mode == "std":
        return float(np.std(x))
    q1, q3 = np.percentile(x, [25, 75])
    return float(q3 - q1)


def _nw_gaussian_predict(x_train: np.ndarray, y_train: np.ndarray,
                         x_eval: np.ndarray, h: float) -> np.ndarray:
    diff = (x_eval[:, None] - x_train[None, :]) / h
    w = np.exp(-0.5 * diff**2)
    w_sum = w.sum(axis=1, keepdims=True) + 1e-12
    return (w / w_sum).dot(y_train)


def _fit_kernel(x: np.ndarray, y: np.ndarray, h_frac: float, mode: str = "iqr"):
    scale = _kernel_scale(x, mode=mode)
    h = max(scale * h_frac, 1e-6)
    return {"x": x.copy(), "y": y.copy(), "h": h}


def _pred_kernel(model: Dict[str, Any], x_eval: np.ndarray) -> np.ndarray:
    return _nw_gaussian_predict(model["x"], model["y"], x_eval, model["h"])


def _kernel_nparams() -> int:  # effective param ~ bandwidth
    return 1


# =============================================================================
# Additional modeling: LOESS / GAM / GPR / LOGISTIC
# =============================================================================

# ---- LOESS ----

def _fit_loess(x: np.ndarray, y: np.ndarray, frac: float):
    if not _HAS_STATS_LOWESS:
        # fallback: use kernel fit with h chosen by frac of range
        rng = np.max(x) - np.min(x)
        h = max(frac * rng, 1e-6)
        return {"_fallback_kernel": True, **_fit_kernel(x, y, h_frac=h / _kernel_scale(x), mode="iqr")}
    # statsmodels.lowess returns sorted results; we just store them
    res = lowess(endog=y, exog=x, frac=frac, it=0, return_sorted=True)
    return {"x": res[:,0], "y": res[:,1], "frac": frac, "_lowess": True}


def _pred_loess(model: Dict[str, Any], x_eval: np.ndarray) -> np.ndarray:
    if model.get("_lowess"):
        # linear interpolation over LOWESS grid
        return np.interp(x_eval, model["x"], model["y"], left=model["y"][0], right=model["y"][-1])
    # fallback kernel
    return _pred_kernel(model, x_eval)


def _loess_nparams() -> int:  # effective param ~ smoothing frac
    return 1


# ---- GAM ----

def _fit_gam(x: np.ndarray, y: np.ndarray, lam: float):
    """Fit a 1D GAM.

    *Fix for pygam gridsearch error*: pygam's .gridsearch() requires a grid of
    length >1. We instead instantiate with the desired `lam` and .fit() directly.
    """
    if _HAS_PYGAM:
        gam = LinearGAM(s(0, n_splines=20, spline_order=3), lam=lam)
        gam.fit(x[:,None], y)
        return gam
    # fallback: spline regression approx (treat lam as df proxy)
    df_proxy = 4 if lam < 1 else 5 if lam < 5 else 6
    return _fit_spline(x, y, df=df_proxy)


def _pred_gam(model, x_eval: np.ndarray, lam: float):
    if _HAS_PYGAM and hasattr(model, 'predict'):
        return model.predict(x_eval[:,None])
    # fallback: we used spline model
    df_proxy = 4 if lam < 1 else 5 if lam < 5 else 6
    return _pred_spline(model, x_eval, df=df_proxy)


def _gam_nparams(lam: float) -> int:
    return 20 + 1 if _HAS_PYGAM else _spline_nparams(6)


# ---- GPR ----

def _fit_gpr(x: np.ndarray, y: np.ndarray, ls_frac: float, mode: str = "iqr"):
    if not _HAS_SK_GPR:
        # fallback: kernel smoother
        return {"_fallback_kernel": True, **_fit_kernel(x, y, h_frac=ls_frac, mode=mode)}
    scale = _kernel_scale(x, mode=mode)
    length_scale = max(scale * ls_frac, 1e-6)
    kernel = 1.0 * RBF(length_scale=length_scale, length_scale_bounds="fixed") + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e3))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=0.0, normalize_y=True)
    gp.fit(x[:,None], y)
    return gp


def _pred_gpr(model, x_eval: np.ndarray):
    if _HAS_SK_GPR and hasattr(model, 'predict'):
        return model.predict(x_eval[:,None])
    # fallback kernel
    return _pred_kernel(model, x_eval)


def _gpr_nparams() -> int:
    return 3  # rough: amplitude, length, noise


# ---- LOGISTIC 4-parameter ----

def _logistic4(x, A, K, B, M):
    # A=lower asymptote, K=upper, B=growth rate, M=midpoint
    return A + (K - A) / (1.0 + np.exp(-B * (x - M)))


def _fit_logistic4(x: np.ndarray, y: np.ndarray):
    if not _HAS_SCIPY:
        # fallback: polynomial
        return {"_fallback_poly": True, "model": _fit_poly(x, y, deg=3), "deg": 3}
    # robust initial guesses
    ymin, ymax = np.min(y), np.max(y)
    A0, K0 = float(ymin), float(ymax)
    # midpoint by 50% volume (or median x)
    try:
        half = A0 + 0.5*(K0-A0)
        M0 = float(np.median(x[np.argsort(np.abs(y-half))[:max(1,len(y)//5)]]))
    except Exception:
        M0 = float(np.median(x))
    B0 = 0.01 if (np.max(x)-np.min(x))>0 else 1.0
    p0 = [A0, K0, B0, M0]
    try:
        popt, _ = curve_fit(_logistic4, x, y, p0=p0, maxfev=20000)
        return tuple(popt)
    except Exception:
        # fallback poly if fit fails
        return {"_fallback_poly": True, "model": _fit_poly(x, y, deg=3), "deg": 3}


def _pred_logistic4(model, x_eval: np.ndarray) -> np.ndarray:
    if isinstance(model, tuple) and len(model) == 4:
        return _logistic4(x_eval, *model)
    # fallback poly
    return _pred_poly(model["model"], x_eval, deg=model["deg"])


def _logistic_nparams() -> int:
    return 4  # A,K,B,M


# =============================================================================
# Cross-validation wrapper (generic)
# =============================================================================
from sklearn.model_selection import KFold


def _cv_model(x: np.ndarray,
              y: np.ndarray,
              family: str,
              hyper: Any,
              n_splits: int = N_SPLITS,
              seed: int = RANDOM_STATE,
              shuffle: bool = SHUFFLE_KFOLD) -> float:
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=seed)
    rmses: List[float] = []
    for tr, te in kf.split(x):
        x_tr, x_te = x[tr], x[te]
        y_tr, y_te = y[tr], y[te]
        if family == "Poly":
            m = _fit_poly(x_tr, y_tr, deg=hyper)
            pred = _pred_poly(m, x_te, deg=hyper)
        elif family == "Spline":
            m = _fit_spline(x_tr, y_tr, df=hyper)
            pred = _pred_spline(m, x_te, df=hyper)
        elif family == "Kernel":
            m = _fit_kernel(x_tr, y_tr, h_frac=hyper)
            pred = _pred_kernel(m, x_te)
        elif family == "LOESS":
            m = _fit_loess(x_tr, y_tr, frac=hyper)
            pred = _pred_loess(m, x_te)
        elif family == "GAM":
            m = _fit_gam(x_tr, y_tr, lam=hyper)
            pred = _pred_gam(m, x_te, lam=hyper)
        elif family == "GPR":
            m = _fit_gpr(x_tr, y_tr, ls_frac=hyper)
            pred = _pred_gpr(m, x_te)
        elif family == "Logistic":
            m = _fit_logistic4(x_tr, y_tr)
            pred = _pred_logistic4(m, x_te)
        else:
            raise ValueError(f"Unknown family {family}")
        rmses.append(float(np.sqrt(np.mean((y_te - pred)**2))))
    return float(np.mean(rmses))


# =============================================================================
# Fit full model + metrics
# =============================================================================

def _fit_full_model(x: np.ndarray, y: np.ndarray, family: str, hyper: Any):
    if family == "Poly":
        m = _fit_poly(x, y, deg=hyper); pred = _pred_poly(m, x, deg=hyper); n_params = _poly_nparams(hyper)
    elif family == "Spline":
        m = _fit_spline(x, y, df=hyper); pred = _pred_spline(m, x, df=hyper); n_params = _spline_nparams(hyper)
    elif family == "Kernel":
        m = _fit_kernel(x, y, h_frac=hyper); pred = _pred_kernel(m, x); n_params = _kernel_nparams()
    elif family == "LOESS":
        m = _fit_loess(x, y, frac=hyper); pred = _pred_loess(m, x); n_params = _loess_nparams()
    elif family == "GAM":
        m = _fit_gam(x, y, lam=hyper); pred = _pred_gam(m, x, lam=hyper); n_params = _gam_nparams(hyper)
    elif family == "GPR":
        m = _fit_gpr(x, y, ls_frac=hyper); pred = _pred_gpr(m, x); n_params = _gpr_nparams()
    elif family == "Logistic":
        m = _fit_logistic4(x, y); pred = _pred_logistic4(m, x); n_params = _logistic_nparams()
    else:
        raise ValueError(f"Unknown family {family}")
    ins = _insample_metrics(y, pred, n_params=n_params)
    return m, ins


# =============================================================================
# Model grid evaluation for given preprocessed dataset
# =============================================================================

def _coerce_hyper(family: str, hyper: Any):
    if family in ("Poly", "Spline"):
        return int(round(float(hyper)))
    if family in ("Kernel", "LOESS", "GAM", "GPR"):
        return float(hyper)
    return hyper  # Logistic uses string tag


def evaluate_model_grid(df: pd.DataFrame,
                        x_col: str,
                        y_col: str = "ED_volume",
                        n_splits: int = N_SPLITS) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    x = df[x_col].values.astype(float)
    y = df[y_col].values.astype(float)

    # Poly
    for d in POLY_DEGREES:
        cv_rmse = _cv_model(x, y, "Poly", d, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "Poly", d)
        rows.append({"Family":"Poly","Hyper":d,"CV_RMSE":cv_rmse,**ins})

    # Spline
    for dfv in SPLINE_DFS:
        cv_rmse = _cv_model(x, y, "Spline", dfv, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "Spline", dfv)
        rows.append({"Family":"Spline","Hyper":dfv,"CV_RMSE":cv_rmse,**ins})

    # Kernel
    for frac in KERNEL_FRACS:
        cv_rmse = _cv_model(x, y, "Kernel", frac, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "Kernel", frac)
        rows.append({"Family":"Kernel","Hyper":frac,"CV_RMSE":cv_rmse,**ins})

    # LOESS
    for frac in LOESS_FRACS:
        cv_rmse = _cv_model(x, y, "LOESS", frac, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "LOESS", frac)
        rows.append({"Family":"LOESS","Hyper":frac,"CV_RMSE":cv_rmse,**ins})

    # GAM
    for lam in GAM_LAMS:
        cv_rmse = _cv_model(x, y, "GAM", lam, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "GAM", lam)
        rows.append({"Family":"GAM","Hyper":lam,"CV_RMSE":cv_rmse,**ins})

    # GPR
    for ls in GPR_LS_FRACS:
        cv_rmse = _cv_model(x, y, "GPR", ls, n_splits=n_splits)
        _, ins = _fit_full_model(x, y, "GPR", ls)
        rows.append({"Family":"GPR","Hyper":ls,"CV_RMSE":cv_rmse,**ins})

    # Logistic (single)
    cv_rmse = _cv_model(x, y, "Logistic", LOGISTIC_TAG, n_splits=n_splits)
    _, ins = _fit_full_model(x, y, "Logistic", LOGISTIC_TAG)
    rows.append({"Family":"Logistic","Hyper":LOGISTIC_TAG,"CV_RMSE":cv_rmse,**ins})

    return pd.DataFrame(rows)


def select_best_model(metrics_df: pd.DataFrame) -> Tuple[Optional[str], Optional[Any]]:
    if metrics_df.empty:
        return None, None
    m_sorted = metrics_df.sort_values(["CV_RMSE", "Hyper"], ascending=[True, True])
    top = m_sorted.iloc[0]
    fam = str(top["Family"])
    hyp = _coerce_hyper(fam, top["Hyper"])
    return fam, hyp


def fit_best_model(df: pd.DataFrame, x_col: str, family: str, hyper: Any):
    x = df[x_col].values.astype(float)
    y = df["ED_volume"].values.astype(float)
    model, ins = _fit_full_model(x, y, family, hyper)
    return model, ins


# =============================================================================
# Plotting helpers
# =============================================================================

def _axis_caps(x: np.ndarray, y: np.ndarray) -> Tuple[Tuple[float,float],Tuple[float,float]]:
    if len(x) == 0:
        return (0, X_CAP_MIN), (0, Y_CAP_MIN)
    x_min, x_max = float(np.nanmin(x)), float(np.nanmax(x))
    y_min, y_max = float(np.nanmin(y)), float(np.nanmax(y))
    xr = max(x_max - x_min, 1.0)
    yr = max(y_max - y_min, 1.0)
    # negative margins below 0
    x_lo = min(-NEG_MARGIN_MIN_X, 0 - NEG_MARGIN_PCT * xr)
    y_lo = min(-NEG_MARGIN_MIN_Y, 0 - NEG_MARGIN_PCT * yr)
    # positive sides with pad & min caps
    x_hi = max(X_CAP_MIN, x_max * (1 + PAD_POS_FRAC))
    y_hi = max(Y_CAP_MIN, y_max * (1 + PAD_POS_FRAC))
    return (x_lo, x_hi), (y_lo, y_hi)


def _plot_scatter(df: pd.DataFrame, out_png: str, title: str,
                  x_col: str = "Time_from_Onset_hours", y_col: str = "ED_volume",
                  removed_df: Optional[pd.DataFrame] = None):
    x = df[x_col].values; y = df[y_col].values
    (xlim, ylim) = _axis_caps(df["Time_from_Onset_hours"].values, df["ED_volume"].values)
    plt.figure(figsize=(8,6))
    if removed_df is not None and not removed_df.empty:
        plt.scatter(removed_df["Time_from_Onset_hours"], removed_df["ED_volume"],
                    s=POINT_SIZE, alpha=ALPHA_REMOVED, color="red", label="Removed")
    plt.scatter(x, y, s=POINT_SIZE, alpha=ALPHA_KEPT if removed_df is not None else ALPHA_ALL,
                edgecolors="none", label="Kept" if removed_df is not None else None)
    plt.xlabel("Onset-to-Scan Time (hours)" if x_col=="Time_from_Onset_hours" else x_col)
    plt.ylabel("ED Volume (ml)")
    plt.title(title)
    plt.xlim(*xlim); plt.ylim(*ylim); plt.grid(True, alpha=0.25)
    if removed_df is not None: plt.legend()
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()
    print(f"[plot] saved: {out_png}")


def plot_scatter_all(df_all: pd.DataFrame, out_png: str):
    _plot_scatter(df_all, out_png, title="All Patients (sub001-sub100)")


def plot_scatter_methodA(df_all: pd.DataFrame, df_kept: pd.DataFrame, df_removed: pd.DataFrame, out_png: str):
    _plot_scatter(df_kept, out_png, title="Method A: After Outlier Filtering", removed_df=df_removed)


def plot_scatter_methodB_log(df_log: pd.DataFrame, out_png: str, offset: float = LOG_OFFSET):
    # y-lim in raw units to match other plots
    (xlim_raw, ylim_raw) = _axis_caps(df_log["Time_from_Onset_hours"].values, df_log["ED_volume"].values)
    plt.figure(figsize=(8,6))
    plt.scatter(df_log["log_time"], df_log["ED_volume"], s=POINT_SIZE, alpha=ALPHA_ALL, edgecolors="none")
    plt.xlabel(f"log(Time_from_Onset_hours + {offset})")
    plt.ylabel("ED Volume (ml)")
    plt.title("Method B: Log-Time Scatter")
    plt.ylim(*ylim_raw); plt.grid(True, alpha=0.25)
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()
    print(f"[plot] saved: {out_png}")


def plot_scatter_methodB_in_orig_time(df_log: pd.DataFrame, out_png: str):
    _plot_scatter(df_log, out_png, title="Method B Data (Original Time Axis)")


def plot_compare_best(df_all: pd.DataFrame,
                      dfA: pd.DataFrame,
                      famA: str, hypA: Any, modelA: Any,
                      dfB: pd.DataFrame,
                      famB: str, hypB: Any, modelB: Any,
                      offset: float,
                      out_png: str):
    # raw axis
    x_all = df_all["Time_from_Onset_hours"].values
    y_all = df_all["ED_volume"].values
    (xlim, ylim) = _axis_caps(x_all, y_all)
    plt.figure(figsize=(9,6))
    plt.scatter(x_all, y_all, s=16, alpha=0.3, edgecolors="none", label="All points")

    # MethodA fit on raw x
    xsA = np.linspace(max(0,xlim[0]), xlim[1], 400)
    xsA_model = xsA.copy()  # x passed directly
    if famA == "Poly":
        ysA = _pred_poly(modelA, xsA_model, deg=hypA)
    elif famA == "Spline":
        ysA = _pred_spline(modelA, xsA_model, df=hypA)
    elif famA == "Kernel":
        mA = _fit_kernel(dfA["Time_from_Onset_hours"].values, dfA["ED_volume"].values, h_frac=hypA)
        ysA = _pred_kernel(mA, xsA_model)
    elif famA == "LOESS":
        mA = _fit_loess(dfA["Time_from_Onset_hours"].values, dfA["ED_volume"].values, frac=hypA)
        ysA = _pred_loess(mA, xsA_model)
    elif famA == "GAM":
        mA = _fit_gam(dfA["Time_from_Onset_hours"].values, dfA["ED_volume"].values, lam=hypA)
        ysA = _pred_gam(mA, xsA_model, lam=hypA)
    elif famA == "GPR":
        mA = _fit_gpr(dfA["Time_from_Onset_hours"].values, dfA["ED_volume"].values, ls_frac=hypA)
        ysA = _pred_gpr(mA, xsA_model)
    elif famA == "Logistic":
        mA = _fit_logistic4(dfA["Time_from_Onset_hours"].values, dfA["ED_volume"].values)
        ysA = _pred_logistic4(mA, xsA_model)
    else:
        ysA = None
    if ysA is not None:
        plt.plot(xsA_model, ysA, lw=2, label=f"MethodA {famA}({hypA})")

    # MethodB fit is in log space; evaluate over raw space by logging xs
    xsB = np.linspace(max(0,xlim[0]), xlim[1], 400)
    log_xsB = np.log(np.maximum(xsB,0) + offset)
    if famB == "Poly":
        ysB = _pred_poly(modelB, log_xsB, deg=hypB)
    elif famB == "Spline":
        ysB = _pred_spline(modelB, log_xsB, df=hypB)
    elif famB == "Kernel":
        mB = _fit_kernel(dfB["log_time"].values, dfB["ED_volume"].values, h_frac=hypB)
        ysB = _pred_kernel(mB, log_xsB)
    elif famB == "LOESS":
        mB = _fit_loess(dfB["log_time"].values, dfB["ED_volume"].values, frac=hypB)
        ysB = _pred_loess(mB, log_xsB)
    elif famB == "GAM":
        mB = _fit_gam(dfB["log_time"].values, dfB["ED_volume"].values, lam=hypB)
        ysB = _pred_gam(mB, log_xsB, lam=hypB)
    elif famB == "GPR":
        mB = _fit_gpr(dfB["log_time"].values, dfB["ED_volume"].values, ls_frac=hypB)
        ysB = _pred_gpr(mB, log_xsB)
    elif famB == "Logistic":
        # logistic fit in log space
        mB = _fit_logistic4(dfB["log_time"].values, dfB["ED_volume"].values)
        ysB = _pred_logistic4(mB, log_xsB)
    else:
        ysB = None
    if ysB is not None:
        plt.plot(xsB, ysB, lw=2, label=f"MethodB {famB}({hypB})")

    plt.xlabel("Onset-to-Scan Time (hours)")
    plt.ylabel("ED Volume (ml)")
    plt.title("Best Models: MethodA vs MethodB")
    plt.xlim(*xlim); plt.ylim(*ylim); plt.grid(True, alpha=0.25); plt.legend()
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()
    print(f"[plot] saved: {out_png}")


# =============================================================================
# Export utilities
# =============================================================================

def _poly_coef_vector(model, deg: int) -> np.ndarray:
    # intercept + slopes (aligned powers 1..deg)
    return np.concatenate(([model.intercept_], model.coef_))


def _poly_formula_text(coefs: np.ndarray, varname="t") -> str:
    terms = []
    for p in range(1, len(coefs)):
        c = coefs[p]; terms.append(f"{c:+.6g}*{varname}^{p}")
    return f"y = {coefs[0]:.6g} " + " ".join(terms)


# =============================================================================
# Main driver
# =============================================================================

def run_q2a_seven_models(path_xy_in: str = PATH_XY_IN,
                         outdir: str = OUTDIR,
                         log_offset: float = LOG_OFFSET) -> Dict[str, Any]:
    _ensure_outdir(outdir)

    # --------------------------------------------------------------
    # Load 6-col dataset
    # --------------------------------------------------------------
    df_all = read_long_xy(path_xy_in, log_offset=log_offset)
    print(f"[data] loaded dataset: {df_all.shape}")
    df_all.to_csv(_join_out("q2a_seven_xy_all_clean.csv"), index=False)

    # --------------------------------------------------------------
    # Preprocess MethodA & MethodB
    # --------------------------------------------------------------
    dfA, dfA_rm = methodA_outlier(df_all)
    dfA.to_csv(_join_out("q2a_seven_xy_methodA.csv"), index=False)
    dfA_rm.to_csv(_join_out("q2a_seven_xy_methodA_removed.csv"), index=False)
    print(f"[data] MethodA kept={len(dfA)} removed={len(dfA_rm)}")

    dfB = methodB_log(df_all, offset=log_offset)
    dfB.to_csv(_join_out("q2a_seven_xy_methodB.csv"), index=False)
    print(f"[data] MethodB rows={len(dfB)}")

    # --------------------------------------------------------------
    # Model evaluation grids
    # --------------------------------------------------------------
    print("[model] evaluating MethodA...")
    metricsA = evaluate_model_grid(dfA, x_col="Time_from_Onset_hours", n_splits=N_SPLITS)
    metricsA["Preprocess"] = "MethodA"

    print("[model] evaluating MethodB...")
    metricsB = evaluate_model_grid(dfB, x_col="log_time", n_splits=N_SPLITS)
    metricsB["Preprocess"] = "MethodB"

    metrics_all = pd.concat([metricsA, metricsB], ignore_index=True)
    metrics_all.to_csv(_join_out("q2a_seven_metrics.csv"), index=False)
    print(f"[output] metrics saved: q2a_seven_metrics.csv")

    # --------------------------------------------------------------
    # Select best per preprocess
    # --------------------------------------------------------------
    famA, hypA = select_best_model(metricsA)
    famB, hypB = select_best_model(metricsB)
    print(f"[select] MethodA best: {famA}({hypA})")
    print(f"[select] MethodB best: {famB}({hypB})")

    # Fit best full models
    mA, insA = fit_best_model(dfA, "Time_from_Onset_hours", famA, hypA) if famA else (None, {})
    mB, insB = fit_best_model(dfB, "log_time", famB, hypB) if famB else (None, {})

    # Save small summary
    pd.DataFrame([
        {"Method":"MethodA","Family":famA,"Hyper":hypA,"CV_RMSE":metricsA.sort_values('CV_RMSE').iloc[0]['CV_RMSE']},
        {"Method":"MethodB","Family":famB,"Hyper":hypB,"CV_RMSE":metricsB.sort_values('CV_RMSE').iloc[0]['CV_RMSE']},
    ]).to_csv(_join_out("q2a_seven_best_models_summary.csv"), index=False)

    # Export polynomial coefs if applicable
    if famA == "Poly":
        coefsA = _poly_coef_vector(mA, hypA)
        pd.DataFrame({"term":["Intercept"]+[f"t^{i}" for i in range(1,hypA+1)],"coef":coefsA}).to_csv(_join_out("q2a_seven_methodA_poly_coefs.csv"), index=False)
    if famB == "Poly":
        coefsB = _poly_coef_vector(mB, hypB)
        pd.DataFrame({"term":["Intercept"]+[f"logt^{i}" for i in range(1,hypB+1)],"coef":coefsB}).to_csv(_join_out("q2a_seven_methodB_poly_coefs.csv"), index=False)

    # --------------------------------------------------------------
    # Predictions (best models) over *all* original points
    # --------------------------------------------------------------
    t_all = df_all["Time_from_Onset_hours"].values
    y_all = df_all["ED_volume"].values

    # MethodA predictions (raw)
    if famA == "Poly":
        yhatA = _pred_poly(mA, t_all, deg=hypA)
    elif famA == "Spline":
        yhatA = _pred_spline(mA, t_all, df=hypA)
    elif famA == "Kernel":
        yhatA = _pred_kernel(mA, t_all)
    elif famA == "LOESS":
        yhatA = _pred_loess(mA, t_all)
    elif famA == "GAM":
        yhatA = _pred_gam(mA, t_all, lam=hypA)
    elif famA == "GPR":
        yhatA = _pred_gpr(mA, t_all)
    elif famA == "Logistic":
        yhatA = _pred_logistic4(mA, t_all)
    else:
        yhatA = np.full_like(t_all, np.nan, dtype=float)

    # MethodB predictions (map via log)
    log_all = np.log(np.maximum(t_all,0) + log_offset)
    if famB == "Poly":
        yhatB = _pred_poly(mB, log_all, deg=hypB)
    elif famB == "Spline":
        yhatB = _pred_spline(mB, log_all, df=hypB)
    elif famB == "Kernel":
        yhatB = _pred_kernel(mB, log_all)
    elif famB == "LOESS":
        yhatB = _pred_loess(mB, log_all)
    elif famB == "GAM":
        yhatB = _pred_gam(mB, log_all, lam=hypB)
    elif famB == "GPR":
        yhatB = _pred_gpr(mB, log_all)
    elif famB == "Logistic":
        yhatB = _pred_logistic4(mB, log_all)
    else:
        yhatB = np.full_like(t_all, np.nan, dtype=float)

    df_pred_all = df_all.copy()
    df_pred_all["Pred_MethodA"] = yhatA
    df_pred_all["Resid_MethodA"] = df_pred_all["ED_volume"] - df_pred_all["Pred_MethodA"]
    df_pred_all["Pred_MethodB"] = yhatB
    df_pred_all["Resid_MethodB"] = df_pred_all["ED_volume"] - df_pred_all["Pred_MethodB"]
    df_pred_all.to_csv(_join_out("q2a_seven_predictions_all.csv"), index=False)
    print(f"[export] predictions written: q2a_seven_predictions_all.csv")

    # per-patient mean residual (MethodA primary)
    df_resid = (
        df_pred_all.groupby("ID")["Resid_MethodA"].mean().reset_index().rename(columns={"Resid_MethodA":"Residual_Mean"})
    )
    df_resid.to_csv(_join_out("q2a_seven_residual_by_patient.csv"), index=False)
    print(f"[export] per-patient residuals: q2a_seven_residual_by_patient.csv")

    # --------------------------------------------------------------
    # Plots
    # --------------------------------------------------------------
    plot_scatter_all(df_all, _join_out("plot_all_patients_scatter.png"))
    plot_scatter_methodA(df_all, dfA, dfA_rm, _join_out("plot_methodA_outlier_scatter.png"))
    plot_scatter_methodB_log(dfB, _join_out("plot_methodB_log_scatter.png"), offset=log_offset)
    plot_scatter_methodB_in_orig_time(dfB, _join_out("plot_methodB_log_data_in_original_time.png"))
    plot_compare_best(df_all, dfA, famA, hypA, mA, dfB, famB, hypB, mB, offset=log_offset,
                      out_png=_join_out("plot_compare_best.png"))

    # MethodA curve plot
    xlim, ylim = _axis_caps(t_all, y_all)
    xs = np.linspace(max(0,xlim[0]), xlim[1], 400)
    if famA == "Poly":
        ysA = _pred_poly(mA, xs, deg=hypA)
    elif famA == "Spline":
        ysA = _pred_spline(mA, xs, df=hypA)
    elif famA == "Kernel":
        ysA = _pred_kernel(mA, xs)
    elif famA == "LOESS":
        ysA = _pred_loess(mA, xs)
    elif famA == "GAM":
        ysA = _pred_gam(mA, xs, lam=hypA)
    elif famA == "GPR":
        ysA = _pred_gpr(mA, xs)
    elif famA == "Logistic":
        ysA = _pred_logistic4(mA, xs)
    else:
        ysA = None
    if ysA is not None:
        plt.figure(figsize=(8,6))
        plt.scatter(t_all, y_all, s=18, alpha=0.35)
        plt.plot(xs, ysA, lw=2, label=f"MethodA {famA}({hypA})")
        plt.xlabel("Onset-to-Scan Time (hours)"); plt.ylabel("ED Volume (ml)")
        plt.xlim(*xlim); plt.ylim(*ylim); plt.grid(True, alpha=0.25); plt.legend()
        plt.tight_layout(); plt.savefig(_join_out("plot_methodA_curve.png"), dpi=150); plt.close()
        print("[plot] plot_methodA_curve.png")

    # MethodB curve (back-transform to raw time)
    xsB = np.linspace(max(0,xlim[0]), xlim[1], 400)
    log_xsB = np.log(np.maximum(xsB,0) + log_offset)
    if famB == "Poly":
        ysB = _pred_poly(mB, log_xsB, deg=hypB)
    elif famB == "Spline":
        ysB = _pred_spline(mB, log_xsB, df=hypB)
    elif famB == "Kernel":
        ysB = _pred_kernel(mB, log_xsB)
    elif famB == "LOESS":
        ysB = _pred_loess(mB, log_xsB)
    elif famB == "GAM":
        ysB = _pred_gam(mB, log_xsB, lam=hypB)
    elif famB == "GPR":
        ysB = _pred_gpr(mB, log_xsB)
    elif famB == "Logistic":
        ysB = _pred_logistic4(mB, log_xsB)
    else:
        ysB = None
    if ysB is not None:
        plt.figure(figsize=(8,6))
        plt.scatter(t_all, y_all, s=18, alpha=0.35)
        plt.plot(xsB, ysB, lw=2, label=f"MethodB {famB}({hypB})")
        plt.xlabel("Onset-to-Scan Time (hours)"); plt.ylabel("ED Volume (ml)")
        plt.xlim(*xlim); plt.ylim(*ylim); plt.grid(True, alpha=0.25); plt.legend()
        plt.tight_layout(); plt.savefig(_join_out("plot_methodB_curve.png"), dpi=150); plt.close()
        print("[plot] plot_methodB_curve.png")

    # Residual histogram (MethodA best)
    plt.figure(figsize=(7,5))
    plt.hist(df_pred_all["Resid_MethodA"], bins=40, alpha=0.8)
    plt.axvline(0, color="k", lw=1)
    plt.xlabel("Residual (Observed - Predicted, ml) [MethodA]")
    plt.ylabel("Count"); plt.title("Residual Distribution (MethodA best)")
    plt.tight_layout(); plt.savefig(_join_out("plot_residual_hist_methodA.png"), dpi=150); plt.close()
    print("[plot] plot_residual_hist_methodA.png")

    return {
        "xy_all": df_all,
        "xy_methodA": dfA,
        "xy_methodA_removed": dfA_rm,
        "xy_methodB": dfB,
        "metrics": metrics_all,
        "methodA_best": (famA, hypA, insA),
        "methodB_best": (famB, hypB, insB),
    }


# =============================================================================
# CLI (ipykernel‑safe)
# =============================================================================
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Q2(a) Seven-Model Edema Progression Pipeline")
    parser.add_argument("--in", dest="infile", default=PATH_XY_IN, help="Input long-format CSV/Excel path")
    parser.add_argument("--offset", dest="offset", type=float, default=LOG_OFFSET, help="Log offset")
    # parse_known_args prevents Jupyter auto -f crash
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[warn] Ignoring unknown CLI args: {unknown}")

    PATH_XY_IN = args.infile
    LOG_OFFSET = args.offset

    res = run_q2a_seven_models(PATH_XY_IN, outdir=OUTDIR, log_offset=LOG_OFFSET)
    print("Done.")


In [ ]:
#!/usr/bin/env python
"""
Moudel 2.1Select Global Best Model (Supports LOESS & GAM)
-----------------------------------------------------
"""

DATA_PATH    = "q2a_seven_models_outputs/q2a_seven_xy_all_clean.csv"
METRICS_PATH = "q2a_seven_models_outputs/q2a_seven_metrics.csv"
OUTDIR       = "q2a_best_model_outputs"
# =======================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    _HAS_LOWESS = True
except Exception:
    _HAS_LOWESS = False

try:
    from pygam import LinearGAM, s
    _HAS_PYGAM = True
except Exception:
    _HAS_PYGAM = False

# ------------------------------------------
# Config
# ------------------------------------------
LOG_OFFSET      = 1.0
OUTLIER_X_CAP   = 2000.0
OUTLIER_Y_CAP   = 100000.0
X_CAP_MIN       = 5000.0
Y_CAP_MIN       = 175000.0
POINT_SIZE      = 32
ALPHA_ALL       = 0.5
ALPHA_REMOVED   = 0.75
ALPHA_KEPT      = 0.65

# ------------------------------------------
def _ensure_outdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def read_long_xy(path_xy: str, log_offset: float = LOG_OFFSET) -> pd.DataFrame:
    df = pd.read_csv(path_xy) if path_xy.lower().endswith(".csv") else pd.read_excel(path_xy)
    df["ED_volume"] = pd.to_numeric(df["ED_volume"], errors="coerce")
    df["Time_from_Onset_hours"] = pd.to_numeric(df["Time_from_Onset_hours"], errors="coerce")
    df = df.dropna(subset=["ED_volume","Time_from_Onset_hours"])
    df["log_time"] = np.log(np.maximum(df["Time_from_Onset_hours"],0.0) + log_offset)
    return df.reset_index(drop=True)

def read_metrics(path_metrics: str) -> pd.DataFrame:
    m = pd.read_csv(path_metrics) if path_metrics.lower().endswith(".csv") else pd.read_excel(path_metrics)
    m["CV_RMSE"] = pd.to_numeric(m["CV_RMSE"], errors="coerce")
    return m

def methodA_outlier(df: pd.DataFrame) -> (pd.DataFrame, pd.DataFrame):
    mask = (df["Time_from_Onset_hours"] <= OUTLIER_X_CAP) & (df["ED_volume"] <= OUTLIER_Y_CAP)
    return df[mask].copy().reset_index(drop=True), df[~mask].copy().reset_index(drop=True)

def pick_global_best(metrics: pd.DataFrame):
    row = metrics.sort_values("CV_RMSE").iloc[0]
    return str(row["Family"]), float(row["Hyper"]) if str(row["Family"])!="Logistic" else row["Hyper"], str(row["Preprocess"]), float(row["CV_RMSE"])

# ------------------------------------------
# Fit GAM
# ------------------------------------------
def _fit_gam(x: np.ndarray, y: np.ndarray, lam: float):
    if not _HAS_PYGAM:
        raise RuntimeError("pygam is not installed.")
    gam = LinearGAM(s(0), lam=lam)
    gam.fit(x[:, None], y)
    return gam

def _pred_gam(gam_model, x_eval: np.ndarray):
    return gam_model.predict(x_eval[:, None])

# ------------------------------------------
def run_q2a_best_model(data_path=DATA_PATH, metrics_path=METRICS_PATH, outdir=OUTDIR):
    _ensure_outdir(outdir)
    df_all = read_long_xy(data_path)
    metrics = read_metrics(metrics_path)

    fam, hyp, pre, cv = pick_global_best(metrics)
    print(f"[Best] {fam}({hyp}) from {pre}, CV_RMSE={cv:.2f}")

    if pre == "MethodA":
        dfA, dfA_rm = methodA_outlier(df_all)
        df_train = dfA
        x_train = df_train["Time_from_Onset_hours"].values
    else:
        df_train = df_all
        x_train = df_train["log_time"].values
    y_train = df_train["ED_volume"].values

    if fam == "GAM":
        model = _fit_gam(x_train, y_train, hyp)
        predict_func = lambda x: _pred_gam(model, x)
        print("[Info] GAM coefficients saved to gam_coefs.csv")
        pd.DataFrame({"coef": model.coef_}).to_csv(os.path.join(outdir,"gam_coefs.csv"), index=False)
    elif fam == "LOESS":
        model = lowess(endog=y_train, exog=x_train, frac=hyp, it=0, return_sorted=True)
        predict_func = lambda x: np.interp(x, model[:,0], model[:,1])
    else:
        raise NotImplementedError(f"Best model {fam} not yet implemented.")

    t_all = df_all["Time_from_Onset_hours"].values
    x_eval = t_all if pre == "MethodA" else df_all["log_time"].values
    y_pred = predict_func(x_eval)
    df_all["Pred_Best"] = y_pred
    df_all["Resid_Best"] = df_all["ED_volume"] - y_pred
    df_all.to_csv(os.path.join(outdir,"predictions_all_points.csv"), index=False)

    df_resid = df_all.groupby("ID")["Resid_Best"].agg(["mean","count","max","min"]).reset_index()
    df_resid.to_csv(os.path.join(outdir,"residual_by_patient.csv"), index=False)

    print("[Done] Outputs in:", outdir)
    return model, df_all, df_resid

if __name__ == "__main__":
    run_q2a_best_model()


In [ ]:
#!/usr/bin/env python
"""
Moudel 2.1Select Global Best Model (Supports LOESS & GAM)
-----------------------------------------------------
"""

DATA_PATH    = "q2a_seven_models_outputs/q2a_seven_xy_all_clean.csv"
METRICS_PATH = "q2a_seven_models_outputs/q2a_seven_metrics.csv"
OUTDIR       = "q2a_best_model_outputs"
# =======================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    _HAS_LOWESS = True
except Exception:
    _HAS_LOWESS = False

try:
    from pygam import LinearGAM, s
    _HAS_PYGAM = True
except Exception:
    _HAS_PYGAM = False

# ------------------------------------------
# Config
# ------------------------------------------
LOG_OFFSET      = 1.0
OUTLIER_X_CAP   = 2000.0
OUTLIER_Y_CAP   = 100000.0
X_CAP_MIN       = 5000.0
Y_CAP_MIN       = 175000.0
POINT_SIZE      = 32
ALPHA_ALL       = 0.5
ALPHA_REMOVED   = 0.75
ALPHA_KEPT      = 0.65

# ------------------------------------------
def _ensure_outdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def read_long_xy(path_xy: str, log_offset: float = LOG_OFFSET) -> pd.DataFrame:
    df = pd.read_csv(path_xy) if path_xy.lower().endswith(".csv") else pd.read_excel(path_xy)
    df["ED_volume"] = pd.to_numeric(df["ED_volume"], errors="coerce")
    df["Time_from_Onset_hours"] = pd.to_numeric(df["Time_from_Onset_hours"], errors="coerce")
    df = df.dropna(subset=["ED_volume","Time_from_Onset_hours"])
    df["log_time"] = np.log(np.maximum(df["Time_from_Onset_hours"],0.0) + log_offset)
    return df.reset_index(drop=True)

def read_metrics(path_metrics: str) -> pd.DataFrame:
    m = pd.read_csv(path_metrics) if path_metrics.lower().endswith(".csv") else pd.read_excel(path_metrics)
    m["CV_RMSE"] = pd.to_numeric(m["CV_RMSE"], errors="coerce")
    return m

def methodA_outlier(df: pd.DataFrame) -> (pd.DataFrame, pd.DataFrame):
    mask = (df["Time_from_Onset_hours"] <= OUTLIER_X_CAP) & (df["ED_volume"] <= OUTLIER_Y_CAP)
    return df[mask].copy().reset_index(drop=True), df[~mask].copy().reset_index(drop=True)

def pick_global_best(metrics: pd.DataFrame):
    row = metrics.sort_values("CV_RMSE").iloc[0]
    return str(row["Family"]), float(row["Hyper"]) if str(row["Family"])!="Logistic" else row["Hyper"], str(row["Preprocess"]), float(row["CV_RMSE"])

# ------------------------------------------
# Fit GAM
# ------------------------------------------
def _fit_gam(x: np.ndarray, y: np.ndarray, lam: float):
    if not _HAS_PYGAM:
        raise RuntimeError("pygam is not installed.")
    gam = LinearGAM(s(0), lam=lam)
    gam.fit(x[:, None], y)
    return gam

def _pred_gam(gam_model, x_eval: np.ndarray):
    return gam_model.predict(x_eval[:, None])

# ------------------------------------------
def run_q2a_best_model(data_path=DATA_PATH, metrics_path=METRICS_PATH, outdir=OUTDIR):
    _ensure_outdir(outdir)
    df_all = read_long_xy(data_path)
    metrics = read_metrics(metrics_path)

    fam, hyp, pre, cv = pick_global_best(metrics)
    print(f"[Best] {fam}({hyp}) from {pre}, CV_RMSE={cv:.2f}")

    if pre == "MethodA":
        dfA, dfA_rm = methodA_outlier(df_all)
        df_train = dfA
        x_train = df_train["Time_from_Onset_hours"].values
    else:
        df_train = df_all
        x_train = df_train["log_time"].values
    y_train = df_train["ED_volume"].values

    if fam == "GAM":
        model = _fit_gam(x_train, y_train, hyp)
        predict_func = lambda x: _pred_gam(model, x)
        print("[Info] GAM coefficients saved to gam_coefs.csv")
        pd.DataFrame({"coef": model.coef_}).to_csv(os.path.join(outdir,"gam_coefs.csv"), index=False)
    elif fam == "LOESS":
        model = lowess(endog=y_train, exog=x_train, frac=hyp, it=0, return_sorted=True)
        predict_func = lambda x: np.interp(x, model[:,0], model[:,1])
    else:
        raise NotImplementedError(f"Best model {fam} not yet implemented.")

    t_all = df_all["Time_from_Onset_hours"].values
    x_eval = t_all if pre == "MethodA" else df_all["log_time"].values
    y_pred = predict_func(x_eval)
    df_all["Pred_Best"] = y_pred
    df_all["Resid_Best"] = df_all["ED_volume"] - y_pred
    df_all.to_csv(os.path.join(outdir,"predictions_all_points.csv"), index=False)

    df_resid = df_all.groupby("ID")["Resid_Best"].agg(["mean","count","max","min"]).reset_index()
    df_resid.to_csv(os.path.join(outdir,"residual_by_patient.csv"), index=False)

    print("[Done] Outputs in:", outdir)
    return model, df_all, df_resid

if __name__ == "__main__":
    run_q2a_best_model()


In [ ]:
#!/usr/bin/env python
"""
moudel2.1- Compute GAM Expression and Residuals for First 100 Patients
===================================================================

1. Load cleaned long-format data (sub001–sub100).
2. Apply MethodA filtering.
3. Fit (or load) GAM model.
4. Output the expression with coefficients.
5. Compute mean absolute residuals (MAR) for each patient and save.

Requirements:
    pip install numpy pandas matplotlib pygam
"""

import os
import numpy as np
import pandas as pd
from pygam import LinearGAM, s

# -----------------------
# Config
# -----------------------
DATA_PATH = "q2a_seven_models_outputs/q2a_seven_xy_all_clean.csv"
OUTDIR = "q2a_gam_final_outputs"
LAM = 1.0  # Best lambda from your metrics
X_CAP = 2000.0
Y_CAP = 100000.0

def ensure_outdir(path):
    os.makedirs(path, exist_ok=True)
    return path

def methodA_outlier(df, x_cap=X_CAP, y_cap=Y_CAP):
    mask = (df["Time_from_Onset_hours"] <= x_cap) & (df["ED_volume"] <= y_cap)
    return df[mask].copy(), df[~mask].copy()

def build_gam_expression(model):
    coefs = model.coef_
    expr = f"ED_volume(t) = {coefs[0]:.6f}"
    for i, c in enumerate(coefs[1:], 1):
        expr += f" + ({c:.6f}) * B_{i}(t)"
    return expr, coefs

def run_gam_residuals():
    ensure_outdir(OUTDIR)
    df = pd.read_csv(DATA_PATH)
    dfA, df_rm = methodA_outlier(df)

    x = dfA["Time_from_Onset_hours"].values
    y = dfA["ED_volume"].values

    # Fit GAM
    gam = LinearGAM(s(0, n_splines=20, spline_order=3), lam=LAM).fit(x[:, None], y)

    # Expression
    expr, coefs = build_gam_expression(gam)
    with open(os.path.join(OUTDIR, "gam_expression.txt"), "w") as f:
        f.write(expr)
    pd.DataFrame({"coef": coefs}).to_csv(os.path.join(OUTDIR, "gam_coefs.csv"), index=False)
    print("[Expression Saved]")

    # Predict and residuals for first 100 patients
    df_first100 = df[df["ID"].str.startswith("sub") & df["ID"].str[3:6].astype(int).between(1,100)]
    pred = gam.predict(df_first100["Time_from_Onset_hours"].values[:, None])
    df_first100["Pred_ED"] = pred
    df_first100["Resid"] = df_first100["ED_volume"] - df_first100["Pred_ED"]
    df_first100.to_csv(os.path.join(OUTDIR, "first100_residuals.csv"), index=False)

    # Mean absolute residual per patient
    df_mar = df_first100.groupby("ID")["Resid"].apply(lambda r: np.mean(np.abs(r))).reset_index()
    df_mar.rename(columns={"Resid": "MAR"}, inplace=True)
    df_mar.to_csv(os.path.join(OUTDIR, "first100_patient_MAR.csv"), index=False)
    print("[Residuals Saved]")

    return expr, df_mar

if __name__ == "__main__":
    expression, df_mar = run_gam_residuals()
    print("GAM Expression:")
    print(expression)
    print("Sample MAR:")
    print(df_mar.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tslearn.clustering import TimeSeriesKMeans


df_norm = pd.read_csv("Q2b_Normalized_XY.csv")

# Step 1: For each patient, interpolate a fixed-length y_norm sequence of 20 points.
fixed_length = 20
patient_series = {}
for pid, group in df_norm.groupby("ID"):
    group_sorted = group.sort_values("x_norm")
    x = group_sorted["x_norm"].values
    y = group_sorted["y_norm"].values
    if len(x) < 2:
        continue
    x_new = np.linspace(x.min(), x.max(), fixed_length)
    y_new = np.interp(x_new, x, y)
    patient_series[pid] = y_new

# Step 2: Constructing a time series array
ts_array = np.array(list(patient_series.values()))
ids = list(patient_series.keys())

# Step 3：Elbow method for plotting clustering error for K=2 to 7
sse_list = []
K_range = range(2, 8)
for k in K_range:
    model = TimeSeriesKMeans(n_clusters=k, metric="dtw", random_state=0)
    labels = model.fit_predict(ts_array)
    sse_list.append(model.inertia_)

# Step 4: Save Drawing
plt.figure(figsize=(8, 5))
plt.plot(K_range, sse_list, marker="o")
plt.title("Elbow Method for Optimal K (DTW-KMeans)")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Sum of DTW Distances)")
plt.grid(True)
plt.tight_layout()
plt.savefig("Q2b_ElbowPlot_DTWKMeans_Updated.png", dpi=300)
plt.show()

# Step5: Save the interpolation result matrix 
pd.DataFrame(ts_array, index=ids).to_csv("Q2b_Interpolated_Ynorm_Matrix.csv")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tslearn.clustering import TimeSeriesKMeans
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# ============ Step 1: Load standardised data ============
df = pd.read_csv("Q2b_Normalized_XY.csv")

# ============ Step 2: Interpolation to a uniform-length sequence ============
fixed_length = 20
patient_series = {}
for pid, group in df.groupby("ID"):
    group_sorted = group.sort_values("x_norm")
    x = group_sorted["x_norm"].values
    y = group_sorted["y_norm"].values
    if len(x) < 2:
        continue
    x_new = np.linspace(x.min(), x.max(), fixed_length)
    y_new = np.interp(x_new, x, y)
    patient_series[pid] = y_new

ts_array = np.array(list(patient_series.values()))
ids = list(patient_series.keys())

# ============ Step 3: K=5 ============
model = TimeSeriesKMeans(n_clusters=5, metric="dtw", random_state=0)
labels = model.fit_predict(ts_array)
group_map = dict(zip(ids, labels))
df["Group"] = df["ID"].map(group_map)

poly_models = {}
group_coefs = {}

for g in range(5):
    df_g = df[df["Group"] == g]
    if len(df_g) < 3:
        continue
    X = df_g["x_norm"].values.reshape(-1, 1)
    y = df_g["y_norm"].values
    X_poly = np.hstack([X**3, X**2, X, np.ones_like(X)])

    
    if g == 2:
        model = Ridge(alpha=8)
    elif g == 3:
        model = Lasso(alpha=0.048)
    else:
        model = LinearRegression()

    model.fit(X_poly, y)
    poly_models[g] = model
    group_coefs[g] = model.coef_.tolist() + [model.intercept_]

# ============ Step 5: Visualisation of Fitting Diagram ============
plt.figure(figsize=(14, 10))
colors = sns.color_palette("tab10", 5)

for g in range(5):
    plt.subplot(2, 3, g + 1)
    df_g = df[df["Group"] == g]
    for pid, group_data in df_g.groupby("ID"):
        plt.plot(group_data["x_norm"], group_data["y_norm"], 'o', alpha=0.3)

    if g in poly_models:
        x_fit = np.linspace(-2.5, 2.5, 200).reshape(-1, 1)
        x_poly = np.hstack([x_fit**3, x_fit**2, x_fit, np.ones_like(x_fit)])
        y_pred = poly_models[g].predict(x_poly)
        plt.plot(x_fit, y_pred, color=colors[g], lw=3, label=f"Group {g} Fit")
        plt.title(f"Group {g}")
        plt.xlabel("x_norm (Standardized Time)")
        plt.ylabel("y_norm (Standardized Volume)")
        plt.legend()

plt.tight_layout()
plt.savefig("Q2b_K5_GroupPolyFit_Optimized.png", dpi=300)
plt.show()

plt.figure(figsize=(14, 10))
colors = sns.color_palette("tab10", 5)
rng = np.random.default_rng(0)

for g in range(5):
    plt.subplot(2, 3, g + 1)
    df_g = df[df["Group"] == g]
    for pid, group_data in df_g.groupby("ID"):
        plt.plot(group_data["x_norm"], group_data["y_norm"], 'o', alpha=0.25, color='gray')

    if g in poly_models and len(df_g) >= 3:
        
        X_all = df_g["x_norm"].values.reshape(-1, 1)
        y_all = df_g["y_norm"].values
        X_all_poly = np.hstack([X_all**3, X_all**2, X_all, np.ones_like(X_all)])

        # 
        x_fit = np.linspace(-2.5, 2.5, 200).reshape(-1, 1)
        x_poly = np.hstack([x_fit**3, x_fit**2, x_fit, np.ones_like(x_fit)])

        model = poly_models[g]
        y_pred = model.predict(x_poly)

        #  -> 95% CI
        resid = y_all - model.predict(X_all_poly)
        stderr = resid.std(ddof=1)  
        ci_upper = y_pred + 1.96 * stderr
        ci_lower = y_pred - 1.96 * stderr

        plt.plot(x_fit, y_pred, color=colors[g], lw=3, label=f"Group {g} Fit")
        plt.fill_between(x_fit.ravel(), ci_lower, ci_upper, color=colors[g], alpha=0.18, label="95% CI (RSE)")
        plt.title(f"Group {g}")
        plt.xlabel("x_norm (Standardized Time)")
        plt.ylabel("y_norm (Standardized Volume)")
        plt.legend()

plt.tight_layout()
plt.savefig("Q2b_K5_GroupPolyFit_withCI_RSE.png", dpi=300)
plt.show()

# ============ Step 5B:Visualisation + 95% CI (Bootstrap percentile method)  ============
plt.figure(figsize=(14, 10))
colors = sns.color_palette("tab10", 5)

B = 1000  
rng = np.random.default_rng(2025)

def make_model_for_group(g):
    
    if g == 2:
        return Ridge(alpha=8)
    elif g == 3:
        return Lasso(alpha=0.048)
    else:
        return LinearRegression()

for g in range(5):
    plt.subplot(2, 3, g + 1)
    df_g = df[df["Group"] == g]
    for pid, group_data in df_g.groupby("ID"):
        plt.plot(group_data["x_norm"], group_data["y_norm"], 'o', alpha=0.25, color='gray')

    if g in poly_models and len(df_g) >= 3:
        x_all = df_g["x_norm"].values.reshape(-1, 1)
        y_all = df_g["y_norm"].values
        X_all_poly = np.hstack([x_all**3, x_all**2, x_all, np.ones_like(x_all)])

        x_fit = np.linspace(-2.5, 2.5, 200).reshape(-1, 1)
        X_fit_poly = np.hstack([x_fit**3, x_fit**2, x_fit, np.ones_like(x_fit)])

        base_pred = poly_models[g].predict(X_fit_poly)

        preds = np.empty((B, X_fit_poly.shape[0]), dtype=float)
        n = len(y_all)
        for b in range(B):
            idx = rng.integers(0, n, size=n)           
            xb = x_all[idx]
            yb = y_all[idx]
            Xb_poly = np.hstack([xb**3, xb**2, xb, np.ones_like(xb)])

            mdl = make_model_for_group(g)
            try:
                mdl.fit(Xb_poly, yb)
                preds[b, :] = mdl.predict(X_fit_poly)
            except Exception:
                
                preds[b, :] = base_pred

        ci_lower = np.quantile(preds, 0.025, axis=0)
        ci_upper = np.quantile(preds, 0.975, axis=0)

        plt.plot(x_fit, base_pred, color=colors[g], lw=3, label=f"Group {g} Fit")
        plt.fill_between(x_fit.ravel(), ci_lower, ci_upper, color=colors[g], alpha=0.18, label="95% CI (Bootstrap)")
        plt.title(f"Group {g}")
        plt.xlabel("x_norm (Standardized Time)")
        plt.ylabel("y_norm (Standardized Volume)")
        plt.legend()

plt.tight_layout()
plt.savefig("Q2b_K5_GroupPolyFit_withCI_Bootstrap.png", dpi=300)
plt.show()

# ============ Step 6:Residual calculation (Root Mean Square Error)  ============
residuals = []

for pid, group_data in df.groupby("ID"):
    group_id = group_data["Group"].iloc[0]
    if group_id not in poly_models:
        continue
    model = poly_models[group_id]
    x = group_data["x_norm"].values.reshape(-1, 1)
    y_true = group_data["y_norm"].values
    x_poly = np.hstack([x**3, x**2, x, np.ones_like(x)])
    y_pred = model.predict(x_poly)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    residuals.append({"ID": pid, "Group": group_id, "RMSE": rmse})

df_residual = pd.DataFrame(residuals)
df_residual.to_csv("Q2b_Patient_Residuals_and_Group.csv", index=False)

# ============ Step 7:Regression coefficient output  ============
coef_df = pd.DataFrame.from_dict(group_coefs, orient="index",
    columns=["x^3", "x^2", "x", "const", "Intercept"])
coef_df.index.name = "Group"
coef_df.to_csv("Q2b_PolyFit_Coefficients_K5.csv")


In [ ]:
# -*- coding: utf-8 -*-
"""
Refine clustering:
1) GMM soft clustering on interpolated yf(t) vectors (K=5)
   -> per-patient probabilities, ambiguous list, suspects 5->4
2) Extract curve features (peak/time, early/mid/late slope, AUC, monotonicity, etc.)
   -> KMeans & Hierarchical re-clustering
   -> silhouette scores, crosstabs vs original labels (if provided) and GMM soft labels
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter

warnings.filterwarnings("ignore")
plt.ion()

# ------------------- Config -------------------
FILE_INTERP = "Q2b_Interpolated_Ynorm_Matrix.csv"
FILE_ORIG_GROUP = "Q2b_Patient_Residuals_and_Group.csv"  # optional
K = 5                           # keep same number of groups
AMBI_THRESH = 0.60              # ambiguous if max prob < 0.60
DELTA_54 = 0.15                 # suggest 5->4 if prob4 - prob5 >= 0.15
GRID_MIN, GRID_MAX = -2.5, 2.5  # standardized time grid range
OUTDIR = Path("./refine_clustering_outputs")
FIGDIR = OUTDIR / "figs"
OUTDIR.mkdir(exist_ok=True, parents=True)
FIGDIR.mkdir(exist_ok=True, parents=True)
# ----------------------------------------------

def ensure_matrix(df):
    """Ensure the matrix columns are only the time-grid values (drop 'ID' if present)."""
    df = df.copy()
    # If there is an 'ID' column (as in your plotting code), drop it
    id_like = [c for c in df.columns if str(c).lower() in ["id","patient_id","sub"]]
    if id_like:
        df = df.drop(columns=id_like)
    # Keep numeric columns only
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return df[num_cols]

def extract_features(y: np.ndarray, x: np.ndarray) -> dict:
    """Extract robust curve features from a single standardized curve y(x)."""
    # Smooth a bit to stabilize derivatives (optional)
    if len(y) >= 7:
        y_s = savgol_filter(y, window_length=7 if len(y)>=7 else len(y)-(len(y)%2==0), polyorder=2, mode="interp")
    else:
        y_s = y

    # Peak & valley
    peak_idx = int(np.argmax(y_s))
    peak_val = float(y_s[peak_idx])
    peak_time = float(x[peak_idx])
    valley_idx = int(np.argmin(y_s))
    valley_val = float(y_s[valley_idx])
    valley_time = float(x[valley_idx])

    # Early/Mid/Late slopes via simple linear fit
    def linfit(t, v):
        # slope only
        A = np.vstack([t, np.ones_like(t)]).T
        coef, *_ = np.linalg.lstsq(A, v, rcond=None)
        return float(coef[0])

    n = len(x)
    seg = max(5, n//5)
    early = slice(0, seg)
    mid   = slice(n//2 - seg//2, n//2 + seg//2)
    late  = slice(n-seg, n)

    early_slope = linfit(x[early], y_s[early])
    mid_slope   = linfit(x[mid],   y_s[mid])
    late_slope  = linfit(x[late],  y_s[late])
    overall_slope = linfit(x, y_s)

    # AUC (trapezoidal)
    auc = float(np.trapz(y_s, x))

    # Monotonicity & shape
    dy = np.gradient(y_s, x)
    pos_ratio = float(np.mean(dy > 0))
    sign_changes = int(np.sum(np.diff(np.sign(dy)) != 0))

    return dict(peak_val=peak_val, peak_time=peak_time,
                valley_val=valley_val, valley_time=valley_time,
                early_slope=early_slope, mid_slope=mid_slope, late_slope=late_slope,
                overall_slope=overall_slope, auc=auc,
                pos_ratio=pos_ratio, sign_changes=sign_changes)

# ------------------- Load data -------------------
df_interp = pd.read_csv(FILE_INTERP, index_col=0)
ids = df_interp.index.astype(str)

# matrix of y_norm (N x L)
Y = ensure_matrix(df_interp).to_numpy(dtype=float)
L = Y.shape[1]
x_grid = np.linspace(GRID_MIN, GRID_MAX, L)

# Try to load original groups (optional)
orig = None
if os.path.exists(FILE_ORIG_GROUP):
    orig = pd.read_csv(FILE_ORIG_GROUP)
    # unify id col name
    cand = [c for c in orig.columns if c.lower() in ["id","sub","patient_id"]]
    if cand:
        orig = orig.rename(columns={cand[0]:"ID"})
        orig["ID"] = orig["ID"].astype(str)
        # ensure one record per ID (take first if duplicates)
        orig_grp = orig.drop_duplicates("ID")[["ID","Group"]]
    else:
        orig = None

# ------------------- GMM soft clustering -------------------
# 1) scale & reduce dim by PCA (avoid high-dim overfit), keep 0.95 variance
scaler = StandardScaler(with_mean=True, with_std=True)
Yz = scaler.fit_transform(Y)

pca = PCA(n_components=0.95, random_state=42)
Z = pca.fit_transform(Yz)     # N x d (d usually << L)

gmm = GaussianMixture(n_components=K, covariance_type="full", random_state=42,
                      n_init=10, reg_covar=1e-6)
gmm.fit(Z)
proba = gmm.predict_proba(Z)       # N x K
gmm_lab = proba.argmax(axis=1) + 1 # 1..K
max_prob = proba.max(axis=1)

# Build output table
cols = [f"prob_G{k}" for k in range(1, K+1)]
df_prob = pd.DataFrame(proba, columns=cols, index=ids).reset_index().rename(columns={"index":"ID"})
df_prob["GMM_Label"] = gmm_lab
df_prob["Max_Prob"] = max_prob
df_prob["Ambiguous"] = (df_prob["Max_Prob"] < AMBI_THRESH).astype(int)

# If original grouping exists, attach and make 'suspect 5->4'
if orig is not None:
    df_prob = df_prob.merge(orig_grp, on="ID", how="left", suffixes=("", "_Orig"))
    df_prob["Suggest_5_to_4"] = (
        (df_prob.get("Group").values == 5) &
        ((df_prob["prob_G4"] - df_prob["prob_G5"]) >= DELTA_54)
    ).astype(int)

df_prob.to_csv(OUTDIR/"gmm_probabilities.csv", index=False)
print(f"[save] gmm_probabilities.csv with {len(df_prob)} rows")

if orig is not None:
    sus = df_prob[(df_prob["Suggest_5_to_4"] == 1)]
    sus.to_csv(OUTDIR/"gmm_suspect_move_5_to_4.csv", index=False)
    print(f"[save] gmm_suspect_move_5_to_4.csv: {len(sus)} candidates")

# Plots: PC1-PC2 scatter & prob-heatmap
if Z.shape[1] >= 2:
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=Z[:,0], y=Z[:,1], hue=gmm_lab, palette="tab10", s=40, edgecolor="none")
    plt.title("GMM soft clustering on PCA space")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.legend(title="GMM label", bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIGDIR/"gmm_pca_scatter.png", dpi=150)
    plt.show()

plt.figure(figsize=(8,6))
# reorder by hard label for visual grouping
order = np.argsort(gmm_lab)
sns.heatmap(proba[order], cmap="viridis", cbar=True, vmin=0, vmax=1)
plt.title("GMM responsibilities (sorted by argmax label)")
plt.xlabel("Group"); plt.ylabel("Patients (sorted)")
plt.xticks(np.arange(K)+0.5, [f"G{k}" for k in range(1, K+1)])
plt.tight_layout()
plt.savefig(FIGDIR/"gmm_prob_heatmap.png", dpi=150)
plt.show()

# ------------------- Feature extraction -------------------
features = []
for i, pid in enumerate(ids):
    feats = extract_features(Y[i, :], x_grid)
    feats["ID"] = pid
    features.append(feats)

df_feat = pd.DataFrame(features).set_index("ID").reset_index()
df_feat.to_csv(OUTDIR/"curve_features.csv", index=False)
print(f"[save] curve_features.csv with {len(df_feat)} rows and {df_feat.shape[1]-1} features")

# ------------------- Re-clustering on features -------------------
# scale features
feat_cols = [c for c in df_feat.columns if c != "ID"]
Xs = StandardScaler().fit_transform(df_feat[feat_cols])

# KMeans
km = KMeans(n_clusters=K, n_init=20, random_state=42)
lab_km = km.fit_predict(Xs) + 1
sil_km = silhouette_score(Xs, lab_km)
pd.DataFrame({"ID":df_feat["ID"], "KMeans_Label":lab_km}).to_csv(OUTDIR/"kmeans_labels.csv", index=False)
print(f"[KMeans] silhouette = {sil_km:.3f}")

# Hierarchical (Ward)
hier = AgglomerativeClustering(n_clusters=K, linkage="ward")
lab_h = hier.fit_predict(Xs) + 1
sil_h = silhouette_score(Xs, lab_h)
pd.DataFrame({"ID":df_feat["ID"], "HIER_Label":lab_h}).to_csv(OUTDIR/"hier_labels.csv", index=False)
print(f"[Hierarchical] silhouette = {sil_h:.3f}")

# Crosstabs vs original and vs GMM
df_all = pd.DataFrame({"ID":df_feat["ID"], "GMM_Label":gmm_lab, "KMeans_Label":lab_km, "HIER_Label":lab_h})
if orig is not None:
    df_all = df_all.merge(orig_grp, on="ID", how="left")
    ct1 = pd.crosstab(df_all["Group"], df_all["GMM_Label"])
    ct2 = pd.crosstab(df_all["Group"], df_all["KMeans_Label"])
    ct3 = pd.crosstab(df_all["Group"], df_all["HIER_Label"])
    ct1.to_csv(OUTDIR/"crosstab_orig_vs_gmm.csv")
    ct2.to_csv(OUTDIR/"crosstab_orig_vs_kmeans.csv")
    ct3.to_csv(OUTDIR/"crosstab_orig_vs_hier.csv")
    print("[save] crosstabs vs original labels")

ct4 = pd.crosstab(df_all["GMM_Label"], df_all["KMeans_Label"])
ct5 = pd.crosstab(df_all["GMM_Label"], df_all["HIER_Label"])
ct4.to_csv(OUTDIR/"crosstab_gmm_vs_kmeans.csv")
ct5.to_csv(OUTDIR/"crosstab_gmm_vs_hier.csv")
print("[save] crosstabs among new labels")

# ------------------- Focus on Groups 4 & 5 -------------------
# List ambiguous and potential 5->4 candidates
if orig is not None:
    amb = df_prob[(df_prob["Ambiguous"]==1)]
    amb.to_csv(OUTDIR/"gmm_ambiguous_patients.csv", index=False)
    print(f"[save] gmm_ambiguous_patients.csv: {len(amb)} rows")

    # plot curves of suspected 5->4
    sus = df_prob[(df_prob.get("Suggest_5_to_4",0)==1)]
    if len(sus) > 0:
        plt.figure(figsize=(7.5,5.2))
        for pid in sus["ID"].values:
            y = Y[ids.tolist().index(pid)]
            plt.plot(x_grid, y, alpha=0.5)
        plt.title("Suspected moves 5→4 (curves)")
        plt.xlabel("x_norm"); plt.ylabel("y_norm")
        plt.grid(True, alpha=.2)
        plt.tight_layout()
        plt.savefig(FIGDIR/"suspect_5_to_4_curves.png", dpi=150)
        plt.show()

# Features 2D visualization (KMeans)
if df_feat.shape[0] >= 10:
    plt.figure(figsize=(6.8,5.2))
    # choose 2 informative features
    f1, f2 = "peak_time", "late_slope"
    sns.scatterplot(data=df_feat, x=f1, y=f2, hue=lab_km, palette="tab10", s=50)
    plt.title(f"Features scatter ({f1} vs {f2}) - KMeans labels")
    plt.legend(title="KMeans")
    plt.tight_layout()
    plt.savefig(FIGDIR/"features_scatter_kmeans.png", dpi=150)
    plt.show()

print(f"\n[Done] Outputs saved in {OUTDIR.resolve()}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Based on existing DTW+GAM results: Plot "per-patient curves" by group, 
and overlay Group GAM Mean + 95% CI.

Required files (default in the same directory as script):
- Q2b_Normalized_XY.csv                  # Original normalized points: ID, x_norm, y_norm
- residuals_gam.csv                      # Per-patient residuals & Group (from previous script)
- gam_outputs/gam_group_curves.csv       # Group means & CI (from previous script; if missing, will fit on the fly)

Output:
- gam_outputs/figs/per_patient_curves_by_group.png   # Panel plot (one subplot per group)
- gam_outputs/figs/per_group_patients/G{g}_patients.png  # Individual plot per group
"""

import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

def _ensure(pkgs):
    for p in pkgs:
        try:
            __import__(p.split("==")[0])
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])

_ensure(["numpy","pandas","matplotlib","seaborn","pygam","scikit-learn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pygam import LinearGAM, s
from sklearn.metrics import mean_squared_error

# ================== Configuration ==================
FN_XY      = "Q2b_Normalized_XY.csv"
FN_RES     = "residuals_gam.csv"                # Select ID, Group
FN_GCURVE  = "gam_outputs/gam_group_curves.csv" # Select group means/CI; if missing, fit on the fly
OUT_DIR    = Path("gam_outputs")
FIG_DIR    = OUT_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
(P := FIG_DIR / "per_group_patients").mkdir(exist_ok=True)

# Whether to fit a small GAM for each patient and connect lines (smoother), 
# otherwise connect raw points directly
SMOOTH_EACH_PATIENT = True
PATIENT_SPLINES = 6      # Max splines for per-patient small GAM (automatically capped at points-1)
PATIENT_LAM     = 0.4    # Smoothing strength for small GAM

# Group GAM (used only when gam_group_curves.csv is missing)
GROUP_SPLINES = 12
GROUP_LAM     = 0.6

# ================== Helper Functions ==================
def read_xy(fn=FN_XY):
    df = pd.read_csv(fn)
    col_id = [c for c in df.columns if c.lower() in ["id","patient","patient_id","sub"]][0]
    col_x  = [c for c in df.columns if c.lower().startswith("x")][0]
    col_y  = [c for c in df.columns if c.lower().startswith("y")][0]
    df = df.rename(columns={col_id:"ID", col_x:"x_norm", col_y:"y_norm"})
    df["ID"] = df["ID"].astype(str)
    return df[["ID","x_norm","y_norm"]].dropna()

def fit_group_gam_and_curve(df_g):
    """Fit GAM for the group on the fly and return curve & CI when gam_group_curves.csv is missing"""
    X = df_g[["x_norm"]].values
    y = df_g["y_norm"].values
    gam = LinearGAM(s(0, n_splines=GROUP_SPLINES), lam=GROUP_LAM).fit(X, y)
    x_fit = np.linspace(df_g["x_norm"].min(), df_g["x_norm"].max(), 200)
    y_fit = gam.predict(x_fit)
    ci    = gam.confidence_intervals(x_fit, width=0.95)
    return pd.DataFrame({"Group": df_g["Group"].iloc[0],
                         "x_fit": x_fit, "y_fit": y_fit,
                         "ci_low": ci[:,0], "ci_high": ci[:,1]})

def plot_one_group(ax, df_g, curve_g=None, color="C0"):
    """Plot all patient curves for the group on the given subplot, overlaying group mean + CI (if provided)"""
    # One line per patient
    if SMOOTH_EACH_PATIENT:
        for pid, g in df_g.groupby("ID"):
            g = g.sort_values("x_norm")
            if len(g) >= 3:
                # Small GAM smoothing
                ns = int(min(PATIENT_SPLINES, len(g)-1))
                ns = max(ns, 3)
                try:
                    gam_i = LinearGAM(s(0, n_splines=ns), lam=PATIENT_LAM).fit(g[["x_norm"]].values, g["y_norm"].values)
                    xx = np.linspace(g["x_norm"].min(), g["x_norm"].max(), 40)
                    yy = gam_i.predict(xx)
                    ax.plot(xx, yy, lw=1.3, alpha=0.35, color=color)
                except Exception:
                    ax.plot(g["x_norm"], g["y_norm"], lw=1.0, alpha=0.35, color=color)
            else:
                ax.plot(g["x_norm"], g["y_norm"], lw=1.0, alpha=0.35, color=color)
    else:
        for pid, g in df_g.groupby("ID"):
            g = g.sort_values("x_norm")
            ax.plot(g["x_norm"], g["y_norm"], lw=1.0, alpha=0.35, color=color)

    # Group GAM Curve + 95% CI
    if curve_g is not None and len(curve_g):
        ax.plot(curve_g["x_fit"], curve_g["y_fit"], color="k", lw=3, label="Group mean (GAM)")
        ax.fill_between(curve_g["x_fit"], curve_g["ci_low"], curve_g["ci_high"],
                        color="k", alpha=0.12, label="95% CI")

    ax.set_xlabel("The standardized time")
    ax.set_ylabel("The standardized volume")
    ax.legend(loc="upper left")

# ================== Data Loading & Merging ==================
df_xy  = read_xy(FN_XY)
df_res = pd.read_csv(FN_RES)   # Must contain ID, Group
df_res["ID"] = df_res["ID"].astype(str)
df = df_xy.merge(df_res[["ID","Group"]], on="ID", how="inner")

# Group GAM curves (if available)
if os.path.exists(FN_GCURVE):
    gcurve = pd.read_csv(FN_GCURVE)
else:
    gcurve = None

# ================== Plot Panel Figure ==================
sns.set_theme(style="whitegrid")
groups = sorted(df["Group"].dropna().unique().tolist())
K = len(groups)
ncols = 3
nrows = int(np.ceil(K / ncols))
fig = plt.figure(figsize=(20, 8) if nrows==1 else (20, 10))
palette = sns.color_palette("tab10", K)

for i, g in enumerate(groups, start=1):
    ax = plt.subplot(nrows, ncols, i)
    df_g = df[df["Group"] == g].copy()
    df_g = df_g.sort_values(["ID","x_norm"])
    color = palette[i-1]

    # Group mean line + CI
    if gcurve is not None:
        curve_g = gcurve[gcurve["Group"] == g]
        if len(curve_g) == 0:
            curve_g = fit_group_gam_and_curve(df_g)
    else:
        curve_g = fit_group_gam_and_curve(df_g)

    plot_one_group(ax, df_g, curve_g, color=color)
    n_pat = df_g["ID"].nunique()
    ax.set_title(f"Group {g} Cluster subgroups after standardization {g}\n(n={n_pat})")

plt.tight_layout()
out_panel = FIG_DIR / "per_patient_curves_by_group.png"
plt.savefig(out_panel, dpi=200)
plt.show()
print(f"[save] {out_panel}")

# ================== Also save individual plots for each group ==================
for g in groups:
    df_g = df[df["Group"] == g].copy()
    if gcurve is not None:
        curve_g = gcurve[gcurve["Group"] == g]
        if len(curve_g) == 0:
            curve_g = fit_group_gam_and_curve(df_g)
    else:
        curve_g = fit_group_gam_and_curve(df_g)

    plt.figure(figsize=(9,4.5))
    plot_one_group(plt.gca(), df_g, curve_g, color=palette[groups.index(g)])
    n_pat = df_g["ID"].nunique()
    plt.title(f"Group {g} Cluster subgroups after standardization {g}\n(n={n_pat})")
    out_one = P / f"G{g}_patients.png"
    plt.tight_layout()
    plt.savefig(out_one, dpi=220)
    plt.close()
    print(f"[save] {out_one}")

In [ ]:
# -*- coding: utf-8 -*-
"""
DTW+GAM Grouping -> Group GAM Curves -> Denormalized Residuals (RMSE/MAE, original units)
New in this version:
- Intra-group GroupKFold CV to select/evaluate GAM smoothing (lam)
- Intra-group Sum of Squares (WSS) (original units)
- Comparison of two strategies for sparse patients (2/3 points):
  A) linear_for_sparse=True: Use linear regression for sparse patients; not used in Group GAM training but included in evaluation.
  B) drop_sparse=True      : Exclude sparse patients; not used for training nor evaluation.
- Visualization: Output 95% CI for each group (both normalized space & original units), plotted independently for A/B.
"""

import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

def _ensure(pkgs):
    for p in pkgs:
        try:
            __import__(p)
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])
_ensure(["numpy","pandas","matplotlib","seaborn","pygam","scikit-learn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pygam import LinearGAM, s
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression

# ----------------- Configuration -----------------
XY_CSV      = "Q2b_Normalized_XY.csv"                  # Must contain: ID, x_norm, y_norm
NORMINFO    = "Q2b_Patient_Normalization_Info.csv"     # Must contain: (y_mu/y_mean), (y_sigma/y_std)
LABELS_CSV  = "gam_outputs/residuals_gam.csv"          # Must contain: ID, Group (from DTW+GAM)

LAM_GRID    = [0.1, 0.3, 0.6, 1.0, 3.0, 10.0]          # Candidate smoothing parameters for intra-group CV
N_SPLINES   = 12                                       # Number of splines for Group GAM
N_FOLDS     = 5                                        # GroupKFold folds (grouped by patient ID)
N_CI        = 200                                      # Number of points for plotting x_grid
ORIG_CI_Q   = (0.025, 0.975)                           # CI quantiles in original units
LINE_ALPHA  = 0.22                                     # Transparency for background individual lines
CI_ALPHA    = 0.16                                     # Transparency for CI

OUTDIR = Path("./gam_cv_wss_outputs")
FIGDIR = OUTDIR / "figs"
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

# ----------------- Helper Functions -----------------
def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Column name not found (candidates {candidates})")
    return None

def _read_xy(csv):
    df = pd.read_csv(csv)
    idc = _pick_col(df, ["ID","id","Sub","sub","patient_id"])
    xc  = _pick_col(df, ["x_norm","x","X_norm","X"])
    yc  = _pick_col(df, ["y_norm","y","Y_norm","Y"])
    return df.rename(columns={idc:"ID", xc:"x_norm", yc:"y_norm"})[["ID","x_norm","y_norm"]].dropna()

def _read_norm(csv):
    nf = pd.read_csv(csv)
    idc = _pick_col(nf, ["ID","id","Sub","sub","patient_id"])
    ymu = _pick_col(nf, ["y_mu","y_mean","mu_y"])
    ysg = _pick_col(nf, ["y_sigma","y_std","sigma_y"])
    return nf.rename(columns={idc:"ID", ymu:"y_mu", ysg:"y_sigma"})[["ID","y_mu","y_sigma"]]

def _read_labels(csv):
    lf = pd.read_csv(csv)
    idc = _pick_col(lf, ["ID","id","Sub","sub","patient_id"])
    grp = _pick_col(lf, ["Group","group","Label","label"])
    lf = lf.rename(columns={idc:"ID", grp:"Group"})[["ID","Group"]].drop_duplicates("ID")
    # Compatible with 0/1-based indexing
    if lf["Group"].min() == 1:
        lf["Group"] = lf["Group"] - 1
    return lf

def _rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def _mae(a, b):  return float(mean_absolute_error(a, b))

def _fit_gam_cv_return_best(df_g, lam_grid=LAM_GRID, n_splines=N_SPLINES, n_folds=N_FOLDS):
    """Perform GroupKFold by patient ID, selecting the model with minimal CV RMSE (normalized space) from lam_grid."""
    X = df_g[["x_norm"]].to_numpy()
    y = df_g["y_norm"].to_numpy()
    groups = df_g["ID"].to_numpy()
    n_uni = df_g["ID"].nunique()
    if n_uni < 2:
        # Fallback: Train with default lam
        gam_best = LinearGAM(s(0, n_splines=n_splines), lam=lam_grid[0]).fit(X, y)
        return gam_best, lam_grid[0], np.nan, pd.DataFrame([{"lam":lam_grid[0],"CV_RMSE":np.nan}])

    gkf = GroupKFold(n_splits=min(n_folds, n_uni))
    cv_summary, best_lam, best_rmse = [], None, np.inf
    for lam in lam_grid:
        rmses = []
        for tr_idx, va_idx in gkf.split(X, y, groups):
            gam = LinearGAM(s(0, n_splines=n_splines), lam=lam).fit(X[tr_idx], y[tr_idx])
            yhat = gam.predict(X[va_idx])
            rmses.append(_rmse(y[va_idx], yhat))
        rm = float(np.mean(rmses))
        cv_summary.append({"lam": lam, "CV_RMSE": rm})
        if rm < best_rmse:
            best_rmse, best_lam = rm, lam

    gam_best = LinearGAM(s(0, n_splines=n_splines), lam=best_lam).fit(X, y)
    return gam_best, best_lam, best_rmse, pd.DataFrame(cv_summary)

def _linear_fit_predict(x, y, x_pred):
    """Linear regression for sparse patients (<=3 points) (normalized space)."""
    lr = LinearRegression().fit(x.reshape(-1,1), y)
    return lr.predict(x_pred.reshape(-1,1))

def _plot_group_figs(dfg_all, dfg_ns, dfg_sp, gam, scenario, g, figdir=FIGDIR, n_ci=N_CI,
                     ci_alpha=CI_ALPHA, line_alpha=LINE_ALPHA, q=ORIG_CI_Q):
    """
    Output two figures:
    1) Normalized space: Patient points/lines + Group GAM + 95% CI (pyGAM)
    2) Original units: Patient lines in original units + "Approximate 95% CI" of Group GAM in original units (mapped by patient μ/σ and taking quantiles across patients)
    """
    # --- Prepare grid (based on x-range of non-sparse data) ---
    xmin = float(dfg_ns["x_norm"].min()) if len(dfg_ns) else float(dfg_all["x_norm"].min())
    xmax = float(dfg_ns["x_norm"].max()) if len(dfg_ns) else float(dfg_all["x_norm"].max())
    x_grid = np.linspace(xmin, xmax, n_ci)
    y_fit_norm = gam.predict(x_grid)

    # --- Normalized space CI ---
    ci_norm = gam.confidence_intervals(x_grid, width=0.95)  # (n,2)

    # --- Original units "Approximate CI": Map via each patient's μ/σ, then take quantiles across patient dimension ---
    # Use μ/σ of non-sparse patients (consistent with training)
    mu_sigma = dfg_ns[["ID","y_mu","y_sigma"]].drop_duplicates("ID")
    if mu_sigma.empty:  # Fallback
        mu_sigma = dfg_all[["ID","y_mu","y_sigma"]].drop_duplicates("ID")
    y_hat_org_all = []
    for _, r in mu_sigma.iterrows():
        mu, sg = r["y_mu"], r["y_sigma"]
        y_hat_org_all.append(y_fit_norm * sg + mu)
    if len(y_hat_org_all) >= 1:
        M = np.vstack(y_hat_org_all)  # shape: [n_patients_in_group, n_ci]
        lo_org = np.quantile(M, q[0], axis=0)
        hi_org = np.quantile(M, q[1], axis=0)
        y_fit_org = np.mean(M, axis=0)  # Use a "representative" mean for visualization
    else:
        lo_org = hi_org = y_fit_org = None

    # --- Colors and directories ---
    pal = sns.color_palette("tab10")
    col  = pal[(g % len(pal))]
    scen = "A_linear_for_sparse" if scenario=="linear_for_sparse" else "B_drop_sparse"
    scen_dir = figdir / scen
    scen_dir.mkdir(parents=True, exist_ok=True)

    # ========== Fig 1: Normalized Space ==========
    plt.figure(figsize=(6.8,4.6))
    # Background individual lines: Non-sparse (solid light color)
    for pid, dfi in dfg_ns.groupby("ID"):
        plt.plot(dfi["x_norm"], dfi["y_norm"], '-', color='gray', alpha=line_alpha, lw=1.4)
    # Sparse (drawn only in strategy A, dashed)
    if scenario == "linear_for_sparse" and len(dfg_sp):
        for pid, dfi in dfg_sp.groupby("ID"):
            plt.plot(dfi["x_norm"], dfi["y_norm"], '--', color='red', alpha=line_alpha, lw=1.2)

    # Group GAM + CI
    plt.plot(x_grid, y_fit_norm, color=col, lw=3, label="Group GAM (norm)")
    plt.fill_between(x_grid, ci_norm[:,0], ci_norm[:,1], color=col, alpha=ci_alpha, label="95% CI (norm)")
    plt.title(f"Group {g} ({scen}) – normalized")
    plt.xlabel("x_norm"); plt.ylabel("y_norm")
    plt.legend()
    plt.tight_layout()
    plt.savefig(scen_dir / f"group_{g}_norm.png", dpi=180)
    plt.show()
    plt.close()

    # ========== Fig 2: Original Units ==========
    plt.figure(figsize=(6.8,4.6))
    # Background: Convert individual points back to original units and plot
    # Non-sparse
    for pid, dfi in dfg_ns.groupby("ID"):
        mu = dfi["y_mu"].iloc[0]; sg = dfi["y_sigma"].iloc[0]
        y_org = dfi["y_norm"].values * sg + mu
        plt.plot(dfi["x_norm"].values, y_org, '-', color='gray', alpha=line_alpha, lw=1.4)
    # Sparse (Strategy A)
    if scenario == "linear_for_sparse" and len(dfg_sp):
        for pid, dfi in dfg_sp.groupby("ID"):
            mu = dfi["y_mu"].iloc[0]; sg = dfi["y_sigma"].iloc[0]
            y_org = dfi["y_norm"].values * sg + mu
            plt.plot(dfi["x_norm"].values, y_org, '--', color='blue', alpha=line_alpha, lw=1.2)

    # Group GAM curve (mean in original units) + Approximate CI
    if y_fit_org is not None:
        plt.plot(x_grid, y_fit_org, color=col, lw=3, label="Group GAM (orig, mean over patients)")
        plt.fill_between(x_grid, lo_org, hi_org, color=col, alpha=ci_alpha, label="~95% band (orig)")
    plt.title(f"Group {g} ({scen}) – original units")
    plt.xlabel("x_norm"); plt.ylabel("volume (ml)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(scen_dir / f"group_{g}_orig.png", dpi=180)
    plt.close()

def _eval_scenario(df_merge, scenario="linear_for_sparse"):
    """
    Unified entry point for two scenarios:
    - linear_for_sparse: Linear prediction for sparse patients (<=3 points); Group GAM training/eval uses only non-sparse (>=4 points).
    - drop_sparse:      Sparse patients are excluded directly, neither trained nor evaluated.
    Returns: per-patient residuals (original units), group-level stats table, CV summary, WSS, etc., and outputs visualization per group.
    """
    assert scenario in ("linear_for_sparse", "drop_sparse")
    # Count points per patient
    pts_count = df_merge.groupby("ID").size().rename("n_points")
    df_merge = df_merge.merge(pts_count, on="ID", how="left")

    # Non-sparse used for Group GAM training/evaluation
    non_sparse = df_merge[df_merge["n_points"] >= 4].copy()
    sparse    = df_merge[df_merge["n_points"] <= 3].copy()

    rows = []
    group_cv_summaries = []
    wss_rows = []

    # Loop by group
    for g, dfg_all in df_merge.groupby("Group"):
        dfg_ns = non_sparse[non_sparse["Group"] == g]
        dfg_sp = sparse[sparse["Group"] == g]
        if len(dfg_ns) < 5 or dfg_ns["ID"].nunique() < 2:
            # Group too small, skip (or use backup linear model here)
            continue

        # 1) Intra-group CV to select best lam + Final Group GAM (trained on non-sparse data)
        gam, best_lam, best_cv_rmse, cv_df = _fit_gam_cv_return_best(dfg_ns)
        group_cv_summaries.append({
            "Group": g, "Best_lam": best_lam, "CV_RMSE": best_cv_rmse,
            "n_patients_non_sparse": int(dfg_ns["ID"].nunique()),
            "n_points_non_sparse":   int(len(dfg_ns))
        })
        cv_df.assign(Group=g).to_csv(OUTDIR/f"cv_curve_group{g}_{scenario}.csv", index=False)

        # 2) Visualization (Normalized/Original units + 95% CI)
        _plot_group_figs(dfg_all, dfg_ns, dfg_sp, gam, scenario=scenario, g=g, figdir=FIGDIR)

        # 3) Per-patient prediction (Residuals in original units)
        #   3.1 Non-sparse: Predict at patient x using Group GAM
        for pid, dfi in dfg_ns.groupby("ID"):
            X = dfi[["x_norm"]].to_numpy()
            y_norm_true = dfi["y_norm"].to_numpy()
            y_norm_pred = gam.predict(X)
            mu = dfi["y_mu"].to_numpy()
            sg = dfi["y_sigma"].to_numpy()
            y_true = y_norm_true * sg + mu
            y_hat  = y_norm_pred * sg + mu
            rows.append({
                "ID": pid, "Group": g, "RMSE": _rmse(y_true, y_hat), "MAE": _mae(y_true, y_hat),
                "n_points": int(dfi["n_points"].iloc[0]), "scenario": scenario
            })
            # Accumulate WSS (per point)
            wss_rows.append({"Group": g, "sse": float(np.sum((y_true - y_hat)**2)), "n_points": len(y_true)})

        #   3.2 Sparse: Two strategies
        if scenario == "linear_for_sparse":
            for pid, dfi in dfg_sp.groupby("ID"):
                x = dfi["x_norm"].to_numpy()
                y = dfi["y_norm"].to_numpy()
                y_norm_pred = _linear_fit_predict(x, y, x_pred=x)
                mu = dfi["y_mu"].to_numpy()
                sg = dfi["y_sigma"].to_numpy()
                y_true = y * sg + mu
                y_hat  = y_norm_pred * sg + mu
                rows.append({
                    "ID": pid, "Group": g, "RMSE": _rmse(y_true, y_hat), "MAE": _mae(y_true, y_hat),
                    "n_points": int(dfi["n_points"].iloc[0]), "scenario": scenario
                })
                wss_rows.append({"Group": g, "sse": float(np.sum((y_true - y_hat)**2)), "n_points": len(y_true)})

        # drop_sparse case: Do not process sparse (naturally excluded)

    # Group-level statistics (Original units)
    res = pd.DataFrame(rows)
    stats = res.groupby("Group").agg(
        Count=("ID","count"),
        RMSE_mean=("RMSE","mean"),
        RMSE_median=("RMSE","median"),
        RMSE_std=("RMSE","std"),
        MAE_mean=("MAE","mean"),
        MAE_median=("MAE","median"),
        MAE_std=("MAE","std"),
    ).reset_index()

    # WSS (Original units)
    if wss_rows:
        wss_df = pd.DataFrame(wss_rows).groupby("Group").agg(
            WSS=("sse","sum"), n_points=("n_points","sum")
        ).reset_index()
        wss_df["WSS_per_point"] = wss_df["WSS"] / wss_df["n_points"].clip(lower=1)
        stats = stats.merge(wss_df, on="Group", how="left")
    else:
        stats["WSS"] = np.nan
        stats["WSS_per_point"] = np.nan

    # CV Summary
    cv_sum = pd.DataFrame(group_cv_summaries)
    return res, stats, cv_sum

# ----------------- 1) Load and Merge -----------------
xy   = _read_xy(XY_CSV)
norm = _read_norm(NORMINFO)
lab  = _read_labels(LABELS_CSV)

df = xy.merge(lab, on="ID", how="inner").merge(norm, on="ID", how="left")
df["Group"] = df["Group"].astype(int)

# ----------------- 2) Evaluate both strategies + Visualization -----------------
res_A, stats_A, cv_A = _eval_scenario(df, scenario="linear_for_sparse")
res_B, stats_B, cv_B = _eval_scenario(df, scenario="drop_sparse")

# Save
OUTDIR.mkdir(exist_ok=True, parents=True)
res_A.to_csv(OUTDIR/"patient_residuals_denorm_linear.csv", index=False)
res_B.to_csv(OUTDIR/"patient_residuals_denorm_drop.csv",   index=False)
stats_A.round(3).to_csv(OUTDIR/"group_stats_linear.csv", index=False)
stats_B.round(3).to_csv(OUTDIR/"group_stats_drop.csv",   index=False)
cv_A.round(4).to_csv(OUTDIR/"cv_summary_linear.csv", index=False)
cv_B.round(4).to_csv(OUTDIR/"cv_summary_drop.csv",   index=False)

# ----------------- 3) Side-by-side Comparison Table -----------------
cmp = stats_A.merge(stats_B, on="Group", suffixes=("_linear","_drop"))
order_cols = [
    "Group",
    "Count_linear","Count_drop",
    "RMSE_mean_linear","RMSE_mean_drop",
    "MAE_mean_linear","MAE_mean_drop",
    "WSS_per_point_linear","WSS_per_point_drop"
]
for c in order_cols:
    if c not in cmp.columns:  # Avoid error if group is missing
        cmp[c] = np.nan
cmp = cmp[order_cols].sort_values("Group")
cmp.to_csv(OUTDIR/"compare_linear_vs_drop.csv", index=False)

# ----------------- 4) Extra Boxplots (Original Units, one for A/B each) -----------------
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8.5,4.6))
sns.boxplot(x="Group", y="RMSE", data=res_A, color="#8fd3fe")
sns.stripplot(x="Group", y="RMSE", data=res_A, color="k", alpha=0.4, size=3)
plt.title("RMSE by Group (A: linear for sparse) - original units")
plt.tight_layout()
plt.savefig(FIGDIR/"rmse_boxplot_linear.png", dpi=180)
plt.close()

plt.figure(figsize=(8.5,4.6))
sns.boxplot(x="Group", y="RMSE", data=res_B, color="#bee9a8")
sns.stripplot(x="Group", y="RMSE", data=res_B, color="k", alpha=0.4, size=3)
plt.title("RMSE by Group (B: drop sparse) - original units")
plt.tight_layout()
plt.savefig(FIGDIR/"rmse_boxplot_drop.png", dpi=180)
plt.show()
plt.close()

# ----------------- Summary Print -----------------
print("\n[Summary] Group Sizes (Original):")
print(df.drop_duplicates("ID")["Group"].value_counts().sort_index())
print("\n=== Strategy A: Linear for sparse patients (included in eval) ===")
print(stats_A.round(3))
print("\n=== Strategy B: Drop sparse patients (excluded from eval) ===")
print(stats_B.round(3))
print("\n=== Comparison (Key Metrics) ===")
print(cmp.round(3))
print(f"\n[Done] Outputs in: {OUTDIR.resolve()}")

In [ ]:
# -*- coding: utf-8 -*-
"""
Compare three evaluation strategies (Baseline / Strategy A Linear Inclusion / Strategy B Exclusion)
on "Denormalized Residuals" + CV + WSS.

Dependencies: Q2b_Normalized_XY.csv, Q2b_Patient_Normalization_Info.csv, gam_outputs/residuals_gam.csv
"""

import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

def _ensure(pkgs):
    for p in pkgs:
        try:
            __import__(p)
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])
_ensure(["numpy","pandas","matplotlib","seaborn","pygam","scikit-learn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pygam import LinearGAM, s
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------- Configuration ----------------
XY_CSV      = "Q2b_Normalized_XY.csv"                  # Must contain: ID, x_norm, y_norm
NORMINFO    = "Q2b_Patient_Normalization_Info.csv"     # Must contain: y_mean/y_mu, y_std/y_sigma
LABELS_CSV  = "gam_outputs/residuals_gam.csv"          # Must contain: ID, Group (from DTW+GAM)

GAM_SPLINES = 12
GAM_LAM     = 0.6
SPARSE_MAX_POINTS = 3  # Threshold for "sparse patients" (<= 3 points)

OUTDIR = Path("./denorm_compare_strategies_outputs")
FIGDIR = OUTDIR / "figs"
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---------------- Helper Functions ----------------
def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Column name not found, candidates: {candidates}")
    return None

def _read_xy(csv):
    df = pd.read_csv(csv)
    idc = _pick_col(df, ["ID","id","Sub","sub","patient_id"])
    xc  = _pick_col(df, ["x_norm","x","X_norm","X"])
    yc  = _pick_col(df, ["y_norm","y","Y_norm","Y"])
    return df.rename(columns={idc:"ID", xc:"x_norm", yc:"y_norm"})[["ID","x_norm","y_norm"]].dropna()

def _read_norm(csv):
    nf = pd.read_csv(csv)
    idc = _pick_col(nf, ["ID","id","Sub","sub","patient_id"])
    ymu = _pick_col(nf, ["y_mu","y_mean","mu_y"])
    ysg = _pick_col(nf, ["y_sigma","y_std","sigma_y"])
    return nf.rename(columns={idc:"ID", ymu:"y_mu", ysg:"y_sigma"})[["ID","y_mu","y_sigma"]]

def _read_labels(csv):
    lf = pd.read_csv(csv)
    idc = _pick_col(lf, ["ID","id","Sub","sub","patient_id"])
    grp = _pick_col(lf, ["Group","group","Label","label"])
    return lf.rename(columns={idc:"ID", grp:"Group"})[["ID","Group"]]

def _fit_group_gam(df_g, n_splines=GAM_SPLINES, lam=GAM_LAM):
    X = df_g[["x_norm"]].values
    y = df_g["y_norm"].values
    gam = LinearGAM(s(0, n_splines=n_splines), lam=lam).fit(X, y)
    return gam

def _rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def _mae(a, b):  return float(mean_absolute_error(a, b))

# "Reference scale" of true y within group, for CV calculation (avoiding zero-division)
def _group_true_mean(df_group):
    # Mean absolute value of all points (denormalized) in the group, 
    # avoiding cancellation of positive/negative values.
    return float(np.mean(np.abs(df_group["y_true_org"].values)))

# Within-group WSS (in original units) -- sum of (y_true - y_pred)^2 for all evaluated points
def _group_wss(df_group):
    resid = df_group["y_true_org"].values - df_group["y_hat_org"].values
    return float(np.sum(resid**2))

# Count points per patient
def _count_points(dfg_by_id):
    return int(len(dfg_by_id))

# ---------------- 1) Load Data & Merge ----------------
xy   = _read_xy(XY_CSV)
norm = _read_norm(NORMINFO)
lab  = _read_labels(LABELS_CSV)

df = xy.merge(lab, on="ID", how="inner").merge(norm, on="ID", how="left")
df["Group"] = df["Group"].astype(int)

# ---------------- 2) Intra-group GAM models (in normalized space) ----------------
gam_models = {}
for g, dfg in df.groupby("Group"):
    if len(dfg) < 5:
        continue
    gam_models[g] = _fit_group_gam(dfg)

# ---------------- 3) Prepare a "points table" shared by three strategies ----------------
# For easier WSS/CV calculation, we prepare y_true_org and all three predictions for every observation.
rows_points = []
for pid, dfi in df.groupby("ID"):
    g = int(dfi["Group"].iloc[0])
    # True y in normalized space
    y_norm_true = dfi["y_norm"].values
    x = dfi[["x_norm"]].values

    # Denormalize (using each record's own mean/variance)
    mu = dfi["y_mu"].values
    sg = dfi["y_sigma"].values
    y_true_org = y_norm_true * sg + mu

    # Baseline: Group GAM prediction -> Denormalize
    if g in gam_models:
        y_norm_pred_base = gam_models[g].predict(x)
    else:
        # Very few groups might lack GAM (too few points), fallback to simple linear regression
        lr = LinearRegression().fit(x, y_norm_true)
        y_norm_pred_base = lr.predict(x)
    y_hat_org_base = y_norm_pred_base * sg + mu

    # Strategy A: If patient points <= SPARSE_MAX_POINTS, use individual linear regression; otherwise use Group GAM
    if len(dfi) <= SPARSE_MAX_POINTS:
        lr = LinearRegression().fit(x, y_norm_true)
        y_norm_pred_A = lr.predict(x)
    else:
        y_norm_pred_A = y_norm_pred_base  # Same as baseline
    y_hat_org_A = y_norm_pred_A * sg + mu

    # Strategy B: Exclude sparse patients (<=3 points): put prediction here first, filter during statistics
    y_hat_org_B = y_norm_pred_base * sg + mu

    # Record detailed points
    for (xt, yt_org, yb, ya, ybB) in zip(x.ravel(), y_true_org, y_hat_org_base, y_hat_org_A, y_hat_org_B):
        rows_points.append({
            "ID": pid,
            "Group": g,
            "x_norm": xt,
            "y_true_org": yt_org,
            "y_hat_org_baseline": yb,
            "y_hat_org_A": ya,
            "y_hat_org_B": ybB,
        })

pts = pd.DataFrame(rows_points)

# Append point count per patient, for Strategy B filtering
npts = df.groupby("ID").size().rename("n_points").reset_index()
pts = pts.merge(npts, on="ID", how="left")

# ---------------- 4) "Per-patient residuals" and "Per-group statistics" for three strategies ----------------
def per_patient_residuals(pts_table, method: str):
    """
    method in ["baseline", "A", "B"]:
      - baseline: All patients, predicted using y_hat_org_baseline
      - A: All patients, sparse (<=3) predicted using individual linear regression (already in y_hat_org_A)
      - B: Exclude sparse patients (<=3)
    """
    if method == "baseline":
        df_use = pts_table.copy()
        df_use["y_pred"] = df_use["y_hat_org_baseline"]
    elif method == "A":
        df_use = pts_table.copy()
        df_use["y_pred"] = df_use["y_hat_org_A"]
    elif method == "B":
        df_use = pts_table[pts_table["n_points"] > SPARSE_MAX_POINTS].copy()
        df_use["y_pred"] = df_use["y_hat_org_B"]
    else:
        raise ValueError("method must be baseline/A/B")

    # Per-patient RMSE/MAE (original units)
    rows = []
    for pid, dfi in df_use.groupby("ID"):
        g = int(dfi["Group"].iloc[0])
        yt = dfi["y_true_org"].values
        yp = dfi["y_pred"].values
        rows.append({
            "ID": pid,
            "Group": g,
            "RMSE": _rmse(yt, yp),
            "MAE":  _mae(yt, yp),
            "n_points": int(dfi["n_points"].iloc[0])
        })
    per_pat = pd.DataFrame(rows)

    # Per-group statistics: RMSE/MAE (mean/median/std), WSS, count, WSS_per_point, CV
    # Average magnitude of true y within group (for CV) -- calculated using the method's point set
    per_group_stats = []
    for g, dfg in df_use.groupby("Group"):
        # All true points & predicted points in this group
        g_true = dfg["y_true_org"].values
        g_pred = dfg["y_pred"].values
        # WSS
        wss = float(np.sum((g_true - g_pred)**2))
        # "Scale" of true values within group
        scale = float(np.mean(np.abs(g_true))) if len(g_true) > 0 else np.nan

        # Get patient RMSE/MAE for this group from per_pat
        grp_pat = per_pat[per_pat["Group"] == g]
        rmse_mean  = grp_pat["RMSE"].mean()
        rmse_median= grp_pat["RMSE"].median()
        rmse_std   = grp_pat["RMSE"].std()
        mae_mean   = grp_pat["MAE"].mean()
        mae_median = grp_pat["MAE"].median()
        mae_std    = grp_pat["MAE"].std()

        per_group_stats.append({
            "Group": g,
            "Count": int(grp_pat["ID"].nunique()),
            "RMSE_mean": rmse_mean,
            "RMSE_median": rmse_median,
            "RMSE_std": rmse_std,
            "MAE_mean": mae_mean,
            "MAE_median": mae_median,
            "MAE_std": mae_std,
            # CV (use "mean absolute value" of true values in group as scale, avoiding denominator close to 0)
            "CV_RMSE_mean": (rmse_mean / scale) if scale > 1e-9 else np.nan,
            "CV_RMSE_median": (rmse_median / scale) if scale > 1e-9 else np.nan,
            "CV_RMSE_std": (rmse_std / scale) if scale > 1e-9 else np.nan,
            "CV_MAE_mean": (mae_mean / scale) if scale > 1e-9 else np.nan,
            "CV_MAE_median": (mae_median / scale) if scale > 1e-9 else np.nan,
            "CV_MAE_std": (mae_std / scale) if scale > 1e-9 else np.nan,
            "Mean_points_per_patient": float(dfg.groupby("ID").size().mean()),
            "WSS": wss,
            "n_points": int(len(dfg)),
            "WSS_per_point": (wss / len(dfg)) if len(dfg) > 0 else np.nan
        })

    return per_pat, pd.DataFrame(per_group_stats).sort_values("Group")

# Calculate three methods
perpat_base, stats_base = per_patient_residuals(pts, "baseline")
perpat_A,    stats_A    = per_patient_residuals(pts, "A")
perpat_B,    stats_B    = per_patient_residuals(pts, "B")

# Save
perpat_base.to_csv(OUTDIR/"patient_residuals_baseline.csv", index=False)
perpat_A.to_csv(OUTDIR/"patient_residuals_strategyA_linear.csv", index=False)
perpat_B.to_csv(OUTDIR/"patient_residuals_strategyB_drop.csv", index=False)
stats_base.round(3).to_csv(OUTDIR/"group_stats_baseline.csv", index=False)
stats_A.round(3).to_csv(OUTDIR/"group_stats_strategyA_linear.csv", index=False)
stats_B.round(3).to_csv(OUTDIR/"group_stats_strategyB_drop.csv", index=False)

print("\n[Summary] Group Sizes (Original):")
print(df.drop_duplicates("ID")["Group"].value_counts().sort_index().rename("count"))

print("\n=== Baseline (No Strategy) ===")
print(stats_base.round(3))
print("\n=== Strategy A: Linear for sparse patients (included in evaluation) ===")
print(stats_A.round(3))
print("\n=== Strategy B: Drop sparse patients (excluded from evaluation) ===")
print(stats_B.round(3))

# ---------------- 5) "Comparison Table" aligned with your existing A/B results ----------------
# Concatenate three tables: Add Method label to columns
sb = stats_base.copy(); sb["Method"] = "Baseline"
sa = stats_A.copy();    sa["Method"] = "StrategyA_linear"
sd = stats_B.copy();    sd["Method"] = "StrategyB_drop"
all_stats = pd.concat([sb, sa, sd], ignore_index=True)

# Key metric comparison (one row per group, columns are three methods)
def _wide_metric(df_metric, metric_col):
    wide = df_metric.pivot(index="Group", columns="Method", values=metric_col).reset_index()
    wide.columns.name = None
    return wide

cmp_rmse_mean = _wide_metric(all_stats, "RMSE_mean")
cmp_mae_mean  = _wide_metric(all_stats, "MAE_mean")
cmp_wsspp     = _wide_metric(all_stats, "WSS_per_point")
cmp_cv_rmse   = _wide_metric(all_stats, "CV_RMSE_mean")

cmp_rmse_mean.to_csv(OUTDIR/"compare_RMSE_mean.csv", index=False)
cmp_mae_mean.to_csv(OUTDIR/"compare_MAE_mean.csv", index=False)
cmp_wsspp.to_csv(OUTDIR/"compare_WSS_per_point.csv", index=False)
cmp_cv_rmse.to_csv(OUTDIR/"compare_CV_RMSE_mean.csv", index=False)

print("\n=== Comparison (Key metrics: RMSE_mean / MAE_mean / WSS_per_point / CV_RMSE_mean) saved ===")

# ---------------- 6) Visualization Comparison Plots ----------------
sns.set_theme(style="whitegrid")

def bar_compare(metric_name, ylabel, fname):
    plt.figure(figsize=(9,4.8))
    plot_df = all_stats[["Group","Method",metric_name]].copy()
    sns.barplot(data=plot_df, x="Group", y=metric_name, hue="Method")
    plt.title(f"{metric_name} by Group (Baseline vs StrategyA vs StrategyB)")
    plt.ylabel(ylabel); plt.xlabel("Group")
    plt.tight_layout()
    plt.savefig(FIGDIR/f"{fname}", dpi=200)
    plt.show()

bar_compare("RMSE_mean", "RMSE_mean (original units)", "cmp_RMSE_mean.png")
bar_compare("WSS_per_point", "WSS per point (original units^2)", "cmp_WSS_per_point.png")
bar_compare("CV_RMSE_mean", "CV of RMSE_mean (unitless)", "cmp_CV_RMSE_mean.png")

print(f"\n[Done] Outputs in: {OUTDIR.resolve()}")

In [ ]:
# =============================================================================
# PART 5: Robustness Checks (Bootstrap & Sensitivity)
# =============================================================================
# Instructions: Append this entire block to the end of the previous script.
# It requires the 'pts' DataFrame and 'METHODS' generated in previous steps.
# =============================================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Configuration
BOOT_B = 1000           # Number of bootstrap iterations
RNG_SEED = 2025         # Random seed for reproducibility
ROBUST_DIR = OUTDIR / "robustness"
ROBUST_FIG = ROBUST_DIR / "figs"
ROBUST_DIR.mkdir(parents=True, exist_ok=True)
ROBUST_FIG.mkdir(parents=True, exist_ok=True)

METHODS = ["Baseline", "StrategyA_linear", "StrategyB_drop"]

# ---------------- Helper: Calculate Metrics for a Subset ----------------
def _calc_metrics_subset(df_subset, method_name):
    """
    Calculate WSS_per_point and RMSE_mean for a specific method on a subset of data.
    """
    # 1. Select the prediction column
    if method_name == "Baseline":
        col_pred = "y_hat_org_baseline"
        # Use all points
        df_use = df_subset
    elif method_name == "StrategyA_linear":
        col_pred = "y_hat_org_A"
        # Use all points
        df_use = df_subset
    elif method_name == "StrategyB_drop":
        col_pred = "y_hat_org_B"
        # Use only points > SPARSE_MAX_POINTS
        df_use = df_subset[df_subset["n_points"] > SPARSE_MAX_POINTS]
    else:
        raise ValueError(f"Unknown method: {method_name}")

    if df_use.empty:
        return np.inf, np.inf, 0

    # 2. Calculate residuals
    y_true = df_use["y_true_org"].to_numpy()
    y_pred = df_use[col_pred].to_numpy()
    residuals = y_true - y_pred
    
    # 3. Aggregates
    wss = np.sum(residuals**2)
    n = len(y_true)
    wss_pp = wss / n
    
    # RMSE calculation (aggregating per patient first to match main logic)
    # Fast grouping using pandas
    rmse_list = df_use.groupby("ID").apply(
        lambda x: np.sqrt(mean_squared_error(x["y_true_org"], x[col_pred]))
    ).values
    rmse_mean = np.mean(rmse_list)
    
    # Count of valid patients
    n_patients = df_use["ID"].nunique()

    return wss_pp, rmse_mean, n_patients

def _choose_best_strict(metrics_dict):
    """
    Selection Rule:
    1. Lowest WSS_per_point
    2. Lowest RMSE_mean
    3. Highest Patient Count (represented as negative for minimization)
    """
    # dict structure: {MethodName: (wss_pp, rmse_mean, count)}
    # We want to minimize (WSS, RMSE, -Count)
    candidates = []
    for m, (w, r, c) in metrics_dict.items():
        candidates.append((w, r, -c, m)) # -c because sorting is ascending
    
    # Sort and pick first
    candidates.sort() 
    return candidates[0][3]

# ---------------- A) Bootstrap Selection Frequency ----------------
print(f"\n[Robustness] Running Bootstrap Analysis (B={BOOT_B})...")
rng = np.random.default_rng(RNG_SEED)

boot_results = []

# Iterate over each group
for g, df_group in pts.groupby("Group"):
    unique_ids = df_group["ID"].unique()
    n_ids = len(unique_ids)
    
    counts = {m: 0 for m in METHODS}
    
    for _ in range(BOOT_B):
        # 1. Resample patients with replacement
        boot_ids = rng.choice(unique_ids, size=n_ids, replace=True)
        
        # 2. Filter data (vectorized isin is faster)
        # Note: A patient selected twice must appear twice. 
        # Using merge/join is better than isin for duplicates.
        boot_id_df = pd.DataFrame({"ID": boot_ids})
        df_boot = boot_id_df.merge(df_group, on="ID", how="left")
        
        # 3. Calculate metrics for all 3 methods on this bootstrap sample
        curr_metrics = {}
        for m in METHODS:
            curr_metrics[m] = _calc_metrics_subset(df_boot, m)
            
        # 4. Pick winner
        winner = _choose_best_strict(curr_metrics)
        counts[winner] += 1
        
    # Calculate frequencies
    for m in METHODS:
        boot_results.append({
            "Group": g,
            "Method": m,
            "Frequency": counts[m] / BOOT_B
        })

# Plotting Bootstrap Heatmap
boot_df = pd.DataFrame(boot_results)
boot_pivot = boot_df.pivot(index="Group", columns="Method", values="Frequency")

plt.figure(figsize=(8, 5))
sns.heatmap(boot_pivot, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)
plt.title(f"Bootstrap Selection Frequency (B={BOOT_B})")
plt.tight_layout()
plt.savefig(ROBUST_FIG / "bootstrap_selection_frequency.png", dpi=200)
plt.show()
print(f"   Saved: {ROBUST_FIG / 'bootstrap_selection_frequency.png'}")

# ---------------- B) Sensitivity Analysis (Weight Sweep) ----------------
print("\n[Robustness] Running Sensitivity Analysis (Weight Sweep)...")

# We want to blend WSS_per_point and RMSE_mean.
# Since they have different units/scales, we must Z-score normalize them per group first.
# Score = w * Z(WSS) + (1-w) * Z(RMSE)
# Low score wins.

w_grid = np.linspace(0, 1, 21) # 0.0 to 1.0 step 0.05
sens_rows = []

for g, df_group in pts.groupby("Group"):
    # 1. Calculate base metrics for this group (No bootstrap)
    raw_metrics = []
    for m in METHODS:
        wpp, rmm, _ = _calc_metrics_subset(df_group, m)
        raw_metrics.append({"Method": m, "WSS": wpp, "RMSE": rmm})
    
    m_df = pd.DataFrame(raw_metrics)
    
    # 2. Z-score normalization within this group
    # (Add small epsilon to std to avoid division by zero)
    m_df["Z_WSS"] = (m_df["WSS"] - m_df["WSS"].mean()) / (m_df["WSS"].std() + 1e-9)
    m_df["Z_RMSE"] = (m_df["RMSE"] - m_df["RMSE"].mean()) / (m_df["RMSE"].std() + 1e-9)
    
    # 3. Sweep weights
    for w in w_grid:
        # Calculate composite score (Lower is better)
        m_df["Score"] = w * m_df["Z_WSS"] + (1 - w) * m_df["Z_RMSE"]
        
        # Identify best method
        best_row = m_df.sort_values("Score").iloc[0]
        sens_rows.append({
            "Group": g,
            "Weight_on_WSS": w,
            "BestMethod": best_row["Method"]
        })

sens_df = pd.DataFrame(sens_rows)

# Map methods to integers for Heatmap
method_map = {name: i for i, name in enumerate(METHODS)} # 0, 1, 2
sens_df["MethodCode"] = sens_df["BestMethod"].map(method_map)

# Pivot for Heatmap
sens_mat = sens_df.pivot(index="Group", columns="Weight_on_WSS", values="MethodCode")

# Custom Colormap: Baseline(Blue), StrategyA(Green), StrategyB(Red)
# Adjust colors to match standard matplotlib tab10 if desired
my_cmap = ListedColormap(["#1f77b4", "#2ca02c", "#d62728"]) 

plt.figure(figsize=(10, 5))
# Use nearest interpolation to show distinct blocks
plt.imshow(sens_mat, aspect="auto", cmap=my_cmap, vmin=0, vmax=2)

# Axis formatting
plt.yticks(ticks=np.arange(len(sens_mat.index)), labels=sens_mat.index)
# X-ticks: show every 2nd or 4th label to avoid crowding
x_ticks_loc = np.arange(len(w_grid))
x_ticks_lab = [f"{v:.1f}" for v in w_grid]
plt.xticks(ticks=x_ticks_loc[::2], labels=x_ticks_lab[::2])

plt.xlabel("Weight on WSS (Z-score) <--- vs ---> Weight on RMSE (Z-score)")
plt.ylabel("Group ID")
plt.title("Sensitivity of Best Method to Metric Weighting")

# Manual Legend
legend_patches = [Patch(color=c, label=m) for m, c in zip(METHODS, ["#1f77b4", "#2ca02c", "#d62728"])]
plt.legend(handles=legend_patches, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig(ROBUST_FIG / "sensitivity_heatmap.png", dpi=200)
plt.show()
print(f"   Saved: {ROBUST_FIG / 'sensitivity_heatmap.png'}")

print(f"\n[Done] Robustness checks completed. Outputs in: {ROBUST_DIR.resolve()}")

In [ ]:
#Moudel3
# -*- coding: utf-8 -*-
# Step 1-2: Wide to Long + Time Matching + "Hours Since Onset" Conversion + log Volume/Day Indicators
# Please install: pandas, numpy, matplotlib before running

from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

# ------------------------
# Paths
# ------------------------
base_dir = Path(r"D:\FYP2\kelly fyp")
outdir   = base_dir / "Q2d_outputs"
figdir   = outdir / "figs"
q2dma_dir = outdir / "q2dma"   # Output folder (as per your requirement)
outdir.mkdir(parents=True, exist_ok=True)
figdir.mkdir(parents=True, exist_ok=True)
q2dma_dir.mkdir(parents=True, exist_ok=True)

# Input files (adjust here if needed)
path_t1 = base_dir / "Table1-Patient_List_and_Clinical_Information.xlsx"
path_t2 = base_dir / "Table2-Patient_Imaging-Volume_and_Location.xlsx"
path_app = base_dir / "Appendix1-Retrieval_Table-SerialNumber_vs_Time.xlsx"

# ------------------------
# Helper Functions
# ------------------------
def normalize_serial(x):
    """
    Normalize Serial Number:
    - Convert to string and strip whitespace
    - Remove trailing .0 if present
    - Remove non-digit characters (for safety)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    if "." in s:
        left, right = s.split(".", 1)
        if right.strip("0") == "":
            s = left
    s = "".join(ch for ch in s if ch.isdigit())
    return s if s else None

def ensure_cols(df, cols, df_name="DataFrame"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{df_name} Missing necessary columns {missing}")

# ------------------------
# Read Data
# ------------------------
t1  = pd.read_excel(path_t1)
t2  = pd.read_excel(path_t2)
app = pd.read_excel(path_app)

# Standardize ID format
for d, nm in [(t1, "Table1"), (t2, "Table2"), (app, "Appendix1")]:
    if "ID" not in d.columns:
        raise KeyError(f"No 'ID' column found in {nm}")
    d["ID"] = d["ID"].astype(str).str.strip().str.lower()

# Essential column checks
ensure_cols(t1, ["ID", "Time_from_Onset_to_First_Imaging"], "Table1")
ensure_cols(app, ["ID", "First_SerialNumber", "First_check_time_point"], "Appendix1")
# Table2/Appendix1 timepoint naming follows pairs: First_* + Follow_up_k_*
# Scan up to 13 follow-up visits (based on provided column names)
labels = ["First"] + [f"Follow_up_{i}" for i in range(1, 14)]

# ------------------------
# Step 1: Reshape Table2 from wide to long format (one row per scan)
# ------------------------
rows_t2 = []
for _, r in t2.iterrows():
    pid = r["ID"]
    for lab in labels:
        sn_col = f"{lab}_SerialNumber"
        if sn_col not in t2.columns:
            continue
        sn = normalize_serial(r[sn_col])
        if not sn:
            continue

        # Extract HM_volume and ED_volume within the current block (between serial number columns)
        start = t2.columns.get_loc(sn_col)
        end = len(t2.columns)
        for nxt_lab in labels[labels.index(lab)+1:]:
            nxt_sn_col = f"{nxt_lab}_SerialNumber"
            if nxt_sn_col in t2.columns:
                end = t2.columns.get_loc(nxt_sn_col)
                break
        block = t2.columns[start:end]

        hm_col = next((c for c in block if "HM_volume" in str(c)), None)
        ed_col = next((c for c in block if "ED_volume" in str(c)), None)
        hm = pd.to_numeric(r[hm_col], errors="coerce") if hm_col else np.nan
        ed = pd.to_numeric(r[ed_col], errors="coerce") if ed_col else np.nan

        rows_t2.append({
            "ID": pid,
            "label": lab,                   # First / Follow_up_k
            "SerialNumber": sn,
            "HM_volume": hm,
            "ED_volume": ed,
        })
t2_long = pd.DataFrame(rows_t2)

# ------------------------
# Step 1-Addition: Reshape Appendix1 table (SerialNumber + Check Time)
# ------------------------
rows_app = []
for _, r in app.iterrows():
    pid = r["ID"]
    for lab in labels:
        sn_col = f"{lab}_SerialNumber"
        tp_col = f"{lab}_check_time_point"
        if (sn_col not in app.columns) or (tp_col not in app.columns):
            continue
        sn = normalize_serial(r[sn_col])
        if not sn:
            continue
        check_time = pd.to_datetime(r[tp_col], errors="coerce")
        rows_app.append({
            "ID": pid,
            "label": lab,
            "SerialNumber": sn,
            "check_time": check_time
        })
app_long = pd.DataFrame(rows_app)

# ------------------------
# Step 2: Calculate "Hours Since Onset"
# Approach: Use First_check_time_point from Appendix1 and Time_from_Onset_to_First_Imaging from Table1
# Derive onset_time, then calculate (check_time - onset_time) to get hours_since_onset
# ------------------------
first_times = app[["ID", "First_check_time_point"]].copy()
first_times["First_check_time_point"] = pd.to_datetime(first_times["First_check_time_point"], errors="coerce")

t1_int = t1[["ID", "Time_from_Onset_to_First_Imaging"]].copy()
t1_int["Time_from_Onset_to_First_Imaging"] = pd.to_numeric(
    t1_int["Time_from_Onset_to_First_Imaging"], errors="coerce"
)

onset = first_times.merge(t1_int, on="ID", how="inner").rename(
    columns={"Time_from_Onset_to_First_Imaging": "onset_to_first_hours"}
)
# Derive onset time
onset["onset_time"] = onset["First_check_time_point"] - pd.to_timedelta(onset["onset_to_first_hours"], unit="h")

# Merge onset_time back into app_long and calculate hours_since_onset
app_long2 = app_long.merge(onset[["ID", "onset_time"]], on="ID", how="left")
app_long2["hours_since_onset"] = (
    (app_long2["check_time"] - app_long2["onset_time"]).dt.total_seconds() / 3600.0
)

# Merge with t2_long to get volumes and times
df_long = t2_long.merge(
    app_long2[["ID", "SerialNumber", "hours_since_onset"]],
    on=["ID", "SerialNumber"],
    how="inner"
)

# Clean and derive variables
df_long = df_long.dropna(subset=["hours_since_onset", "HM_volume", "ED_volume"]).copy()
df_long["log_Hemo"] = np.log(df_long["HM_volume"] + 1e-3)
df_long["log_ED"]   = np.log(df_long["ED_volume"] + 1e-3)
df_long["time_day"] = df_long["hours_since_onset"] / 24.0

# ------------------------
# Save Results & Quality Control Info
# ------------------------
# Main result: ready for subsequent longitudinal modeling
out_csv = q2dma_dir / "longitudinal_data_with_hours_since_onset.csv"
df_long.to_csv(out_csv, index=False, encoding="utf-8-sig")

# Auxiliary: Count and range of timepoints
stats = {
    "n_patients": int(df_long["ID"].nunique()),
    "n_rows": int(len(df_long)),
    "hours_range": [
        float(df_long["hours_since_onset"].min()) if len(df_long) else None,
        float(df_long["hours_since_onset"].max()) if len(df_long) else None,
    ],
    "median_timepoints_per_patient": float(df_long.groupby("ID")["SerialNumber"].count().median()) if len(df_long) else None
}
with open(q2dma_dir / "step1_2_summary.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

# Simple QC plot: Plot log(ED) over time for a sample of 6 patients
try:
    sample_ids = df_long["ID"].drop_duplicates().sample(min(6, df_long["ID"].nunique()), random_state=42).tolist()
    plt.figure(figsize=(10, 6))
    for sid in sample_ids:
        dfi = df_long[df_long["ID"] == sid].sort_values("time_day")
        plt.plot(dfi["time_day"], dfi["log_ED"], marker="o", alpha=0.8, label=sid)
    plt.xlabel("Time after the onset of the disease (days)")
    plt.ylabel("log(ED_volume)")
    plt.title("Edema volume (logarithm) changes over time: Sample patient")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(figdir / "q2dma_logED_over_time_samples.png", dpi=150)
    plt.show()
    plt.close()
except Exception as e:
    print("Plotting skipped:", e)

print("✅ Step1-2 Complete:")
print("  - Longitudinal data file:", out_csv)
print("  - Summary info:", q2dma_dir / "step1_2_summary.json")
print("  - Example plot:", figdir / "q2dma_logED_over_time_samples.png")
print("  - QC statistics:", stats)

In [ ]:
# Step 3: Longitudinal Regression (Patient Fixed Effects + Cluster-Robust SE)
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

long_path = r"D:\FYP2\kelly fyp\Q2d_outputs\q2dma\longitudinal_data_with_hours_since_onset.csv"
t1_path   = r"D:\FYP2\kelly fyp\Table1-Patient_List_and_Clinical_Information.xlsx"

df = pd.read_csv(long_path, encoding="utf-8-sig")
t1 = pd.read_excel(t1_path)

df["ID"] = df["ID"].astype(str).str.strip().str.lower()
t1["ID"] = t1["ID"].astype(str).str.strip().str.lower()

treat_cols = [
    "Ventricular_drainage_treatment",
    "Hemostatic_treatment",
    "Reducing_intracranial_pressure_treatment",
    "Blood_pressure_lowering_treatment",
    "Sedation_and_analgesic_treatment",
    "Stop_vomiting_and_protect_stomach_treatment",
    "Neurotrophic_treatment",
]
covar_cols = ["Age","Gender","mRS_Score_before_cerebral_hemorrhage"]

t1_sub = t1[["ID"]+treat_cols+covar_cols].copy()
for c in treat_cols:
    t1_sub[c] = pd.to_numeric(t1_sub[c], errors="coerce").fillna(0).astype(int)

t1_sub["Gender"] = t1_sub["Gender"].astype(str).str.strip().str.lower()
gender_dummies = pd.get_dummies(t1_sub["Gender"], prefix="Gender")
t1_sub = pd.concat([t1_sub.drop(columns=["Gender"]), gender_dummies], axis=1)

df = df.merge(t1_sub, on="ID", how="inner")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["mRS_Score_before_cerebral_hemorrhage"] = pd.to_numeric(df["mRS_Score_before_cerebral_hemorrhage"], errors="coerce")

# Baseline controls: initial log volumes for each patient
idx_first = df.groupby("ID")["hours_since_onset"].idxmin()
base = df.loc[idx_first, ["ID","log_Hemo","log_ED"]].rename(columns={"log_Hemo":"baseline_log_Hemo","log_ED":"baseline_log_ED"})
df = df.merge(base, on="ID", how="left")

df = df.dropna(subset=["log_ED","log_Hemo","time_day","Age","baseline_log_ED","baseline_log_Hemo"])

# Select 3 treatments for interaction with time (more stable)
key_ts = ["Reducing_intracranial_pressure_treatment","Hemostatic_treatment","Blood_pressure_lowering_treatment"]
main_part = " + ".join(key_ts)
inter_part = " + ".join([f"time_day:{t}" for t in key_ts])
gender_terms = " + ".join([c for c in df.columns if c.startswith("Gender_")]) or "0"

formula_ED   = f"log_ED ~ time_day + {main_part} + {inter_part} + Age + mRS_Score_before_cerebral_hemorrhage + baseline_log_ED + baseline_log_Hemo + {gender_terms} + C(ID)"
formula_Hemo = f"log_Hemo ~ time_day + {main_part} + {inter_part} + Age + mRS_Score_before_cerebral_hemorrhage + baseline_log_ED + baseline_log_Hemo + {gender_terms} + C(ID)"

model_ED   = smf.ols(formula_ED, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["ID"]})
model_Hemo = smf.ols(formula_Hemo, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["ID"]})

def tidy_ols(m):
    return pd.DataFrame({
        "term": m.params.index,
        "estimate": m.params.values,
        "std_error": m.bse.values,
        "t_value": m.tvalues.values,
        "p_value": m.pvalues.values
    })

sum_ED   = tidy_ols(model_ED)
sum_Hemo = tidy_ols(model_Hemo)

# Save to q2dma
out_dir = r"D:\FYP2\kelly fyp\Q2d_outputs\q2dma"
pd.DataFrame(sum_ED).to_csv(fr"{out_dir}\step3_ED_FE_OLS_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(sum_Hemo).to_csv(fr"{out_dir}\step3_HEMO_FE_OLS_summary.csv", index=False, encoding="utf-8-sig")

print("Key interactions in ED model:")
print(sum_ED[sum_ED["term"].str.startswith("time_day:")][["term","estimate","std_error","p_value"]].to_string(index=False))
print("\nKey interactions in Hemo model:")
print(sum_Hemo[sum_Hemo["term"].str.startswith("time_day:")][["term","estimate","std_error","p_value"]].to_string(index=False))

In [ ]:
# -*- coding: utf-8 -*-
# Step 4~6: Longitudinal FE-OLS Main Results + Time Window Sensitivity + Individual Slope Verification
# Dependencies: pandas, numpy, statsmodels, matplotlib, patsy

from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from patsy import dmatrix

# ======================================
# Path Settings (Adjust to your local directory)
# ======================================
base_dir = Path(r"D:\FYP2\kelly fyp")
indir    = base_dir / "Q2d_outputs" / "q2dma"
outdir   = base_dir / "Q2d_outputs" / "q2dma"   # Consistent with your request: save under q2dma
figdir   = outdir / "figs"
outdir.mkdir(parents=True, exist_ok=True)
figdir.mkdir(parents=True, exist_ok=True)

# Input files
long_path = indir / "longitudinal_data_with_hours_since_onset.csv"
t1_path   = base_dir / "Table1-Patient_List_and_Clinical_Information.xlsx"

# ======================================
# Read Data and Preprocessing
# ======================================
df = pd.read_csv(long_path, encoding="utf-8-sig")
t1 = pd.read_excel(t1_path)

# Standardize ID
df["ID"] = df["ID"].astype(str).str.strip().str.lower()
t1["ID"] = t1["ID"].astype(str).str.strip().str.lower()

# 7 types of treatments + covariates
treat_cols = [
    "Ventricular_drainage_treatment",
    "Hemostatic_treatment",
    "Reducing_intracranial_pressure_treatment",
    "Blood_pressure_lowering_treatment",
    "Sedation_and_analgesic_treatment",
    "Stop_vomiting_and_protect_stomach_treatment",
    "Neurotrophic_treatment",
]
covar_cols = ["Age", "Gender", "mRS_Score_before_cerebral_hemorrhage"]

# Subset and type conversion
t1_sub = t1[["ID"] + treat_cols + covar_cols].copy()
for c in treat_cols:
    t1_sub[c] = pd.to_numeric(t1_sub[c], errors="coerce").fillna(0).astype(int)
t1_sub["Gender"] = t1_sub["Gender"].astype(str).str.strip().str.lower()
gender_dummies = pd.get_dummies(t1_sub["Gender"], prefix="Gender")
t1_sub = pd.concat([t1_sub.drop(columns=["Gender"]), gender_dummies], axis=1)

# Merge into longitudinal panel
df = df.merge(t1_sub, on="ID", how="inner")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["mRS_Score_before_cerebral_hemorrhage"] = pd.to_numeric(df["mRS_Score_before_cerebral_hemorrhage"], errors="coerce")

# Baseline: log volume at the earliest time point for each patient
idx_first = df.groupby("ID")["hours_since_onset"].idxmin()
base = df.loc[idx_first, ["ID", "log_Hemo", "log_ED"]].rename(
    columns={"log_Hemo":"baseline_log_Hemo", "log_ED":"baseline_log_ED"}
)
df = df.merge(base, on="ID", how="left")

# Keep key columns complete (drop NAs)
df = df.dropna(subset=["log_ED","log_Hemo","time_day","Age","baseline_log_ED","baseline_log_Hemo"])

# 3 treatments interacting with time (for robustness; can expand to 7 if needed)
key_ts = ["Reducing_intracranial_pressure_treatment", "Hemostatic_treatment", "Blood_pressure_lowering_treatment"]

# ======================================
# Helper Functions
# ======================================
def fit_fe_ols(data: pd.DataFrame, outcome: str):
    """
    Patient Fixed Effects FE-OLS + Cluster-Robust SE by ID
    Formula: outcome ~ time_day + treatments + time_day:treatments + covariates + baseline + Gender + C(ID)
    """
    main_part = " + ".join(key_ts)
    inter_part = " + ".join([f"time_day:{t}" for t in key_ts])
    gender_terms = " + ".join([c for c in data.columns if c.startswith("Gender_")]) or "0"
    base_term = "baseline_log_ED + baseline_log_Hemo"

    formula = f"{outcome} ~ time_day + {main_part} + {inter_part} + Age + mRS_Score_before_cerebral_hemorrhage + {base_term} + {gender_terms} + C(ID)"
    model = smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": data["ID"]})

    tidy = pd.DataFrame({
        "term": model.params.index,
        "estimate": model.params.values,
        "std_error": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values
    })
    return model, tidy, formula

def plot_marginal(model, data, outcome: str, treat_name: str, out_png: Path):
    """
    Plot marginal effects: Predicted curves for treatment=0 vs 1 
    (other covariates at median, ID uses an observed ID)
    """
    t_min, t_max = data["time_day"].quantile(0.01), data["time_day"].quantile(0.99)
    grid = pd.DataFrame({"time_day": np.linspace(t_min, t_max, 100)})

    # Set covariates to median
    grid["Age"] = data["Age"].median()
    grid["mRS_Score_before_cerebral_hemorrhage"] = data["mRS_Score_before_cerebral_hemorrhage"].median()
    grid["baseline_log_ED"] = data["baseline_log_ED"].median()
    grid["baseline_log_Hemo"] = data["baseline_log_Hemo"].median()
    for c in key_ts:
        grid[c] = 0
    for c in [col for col in data.columns if col.startswith("Gender_")]:
        grid[c] = data[c].median()

    # Keep a real ID for C(ID) (patsy will automatically generate corresponding dummy variables)
    grid["ID"] = data["ID"].iloc[0]

    def predict_curve(treat_on: bool):
        g = grid.copy()
        g[treat_name] = 1 if treat_on else 0
        X = dmatrix(model.model.data.design_info.builder, g, return_type="dataframe")
        # Align columns
        for col in model.model.exog_names:
            if col not in X.columns:
                X[col] = 0.0
        X = X[model.model.exog_names]
        yhat = np.dot(X.values, model.params.values)
        return g["time_day"].values, yhat

    x0, y0 = predict_curve(False)
    x1, y1 = predict_curve(True)

    plt.figure(figsize=(8,5))
    plt.plot(x0, y0, label="Off")
    plt.plot(x1, y1, label="On")
    plt.xlabel("Days since onset")
    plt.ylabel(f"Predicted {outcome}")
    plt.title(f"Marginal effect on {outcome}: {treat_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def write_csv_drop_id_dummies(df_tidy: pd.DataFrame, path: Path):
    """Exclude massive C(ID) dummy variables from the output CSV, keeping only key terms for readability."""
    df_tidy.loc[~df_tidy["term"].str.startswith("C(ID)")].to_csv(path, index=False, encoding="utf-8-sig")

# ======================================
# Step 4: Full Sample (No time window restriction)
# ======================================
model_ED, tidy_ED, formula_ED = fit_fe_ols(df, "log_ED")
model_HE, tidy_HE, formula_HE = fit_fe_ols(df, "log_Hemo")

# Export key terms table (excluding C(ID))
write_csv_drop_id_dummies(tidy_ED, outdir / "step4_ED_FE_OLS_full.csv")
write_csv_drop_id_dummies(tidy_HE, outdir / "step4_HEMO_FE_OLS_full.csv")

# Marginal effect curves (ED/Hemo x 3 treatments)
for tr in key_ts:
    plot_marginal(model_ED, df, "log_ED", tr, figdir / f"step4_ED_marginal_{tr}.png")
    plot_marginal(model_HE, df, "log_Hemo", tr, figdir / f"step4_HEMO_marginal_{tr}.png")

print("Step 4 OK ->", outdir)

# ======================================
# Step 5: Time Window Sensitivity (0–48h / 72h / 96h / 168h)
# ======================================
windows_hours = [48, 72, 96, 168]
for wh in windows_hours:
    sub = df[df["hours_since_onset"] <= wh].copy()
    # Skip if sample size/number of patients is too small, write a note
    if sub["ID"].nunique() < 20 or len(sub) < 60:
        pd.DataFrame({"note":[f"window 0-{wh}h has too few patients/rows for stable FE-OLS"]}) \
          .to_csv(outdir / f"step5_note_window_0_{wh}h.csv", index=False, encoding="utf-8-sig")
        continue

    mED_w, tidyED_w, _ = fit_fe_ols(sub, "log_ED")
    mHE_w, tidyHE_w, _ = fit_fe_ols(sub, "log_Hemo")

    write_csv_drop_id_dummies(tidyED_w, outdir / f"step5_ED_window_0_{wh}h.csv")
    write_csv_drop_id_dummies(tidyHE_w, outdir / f"step5_HEMO_window_0_{wh}h.csv")

print("Step 5 OK ->", outdir)

# ======================================
# Step 6: Patient-level Slopes + Cross-sectional OLS (HC3)
# ======================================
def patient_slopes(df_in: pd.DataFrame, max_hours: int) -> pd.DataFrame:
    """Calculate individual slopes for each patient within the specified time window (hours) (slope of log_ED/log_Hemo vs time_day)"""
    max_days = max_hours / 24.0
    d = df_in[df_in["time_day"] <= max_days].copy()
    rows = []
    for pid, g in d.groupby("ID"):
        g = g.sort_values("time_day")
        if g["time_day"].nunique() < 2:
            continue
        slope_ed = np.polyfit(g["time_day"].values, g["log_ED"].values, deg=1)[0]
        slope_he = np.polyfit(g["time_day"].values, g["log_Hemo"].values, deg=1)[0]
        rows.append({"ID": pid, "ED_slope": slope_ed, "Hemo_slope": slope_he, "n_points": len(g)})
    return pd.DataFrame(rows)

def slope_cross_section_ols(sl_df: pd.DataFrame, wh: int):
    """
    Cross-sectional OLS of Slope ~ 3 Treatments + Age + mRS_before (HC3 Robust SE).
    If sample is too small, only export slope table and note.
    """
    out_sl = outdir / f"step6_patient_slopes_{wh}h.csv"
    if sl_df is None or sl_df.empty:
        pd.DataFrame({"note":[f"window 0-{wh}h: 0 patients with >=2 time points"]}) \
          .to_csv(outdir / f"step6_note_{wh}h.csv", index=False, encoding="utf-8-sig")
        return

    # Merge covariates and treatments (use t1_sub directly to keep encoding consistent with longitudinal model)
    keep = ["ID","Age","mRS_Score_before_cerebral_hemorrhage"] + key_ts
    t1_small = t1_sub[keep].copy()

    X = sl_df.merge(t1_small, on="ID", how="left")
    X.to_csv(out_sl, index=False, encoding="utf-8-sig")

    # Minimum rows for regression: >=20 (adjust as needed)
    if X.dropna().shape[0] < 20:
        pd.DataFrame({"note":[f"window 0-{wh}h: <20 rows for OLS; only patient slopes exported"]}) \
          .to_csv(outdir / f"step6_cs_note_{wh}h.csv", index=False, encoding="utf-8-sig")
        return

    f_ED = "ED_slope ~ " + " + ".join(key_ts) + " + Age + mRS_Score_before_cerebral_hemorrhage"
    f_HE = "Hemo_slope ~ " + " + ".join(key_ts) + " + Age + mRS_Score_before_cerebral_hemorrhage"
    mED = smf.ols(f_ED, data=X).fit(cov_type="HC3")
    mHE = smf.ols(f_HE, data=X).fit(cov_type="HC3")

    pd.DataFrame({
        "term": mED.params.index, "estimate": mED.params.values, "p_value": mED.pvalues.values
    }).to_csv(outdir / f"step6_EDslope_cs_{wh}h.csv", index=False, encoding="utf-8-sig")

    pd.DataFrame({
        "term": mHE.params.index, "estimate": mHE.params.values, "p_value": mHE.pvalues.values
    }).to_csv(outdir / f"step6_HEMOslope_cs_{wh}h.csv", index=False, encoding="utf-8-sig")

# Calculate and export 48h / 72h / 96h / 168h
for wh in [48, 72, 96, 168]:
    sl = patient_slopes(df, wh)
    slope_cross_section_ols(sl, wh)

print("Step 6 OK ->", outdir)

print("\nAll done! Check directory:", str(outdir))
print("Main outputs:")
print(" - step4_ED_FE_OLS_full.csv, step4_HEMO_FE_OLS_full.csv")
print(" - step5_* (Time window sensitivity)")
print(" - step6_patient_slopes_*.csv + step6_*slope_cs_*.csv")
print(" - figs\\step4_* (Marginal effect plots)")

In [ ]:
# -*- coding: utf-8 -*-
# Visualization of Step4+5 Results: Direction/Significance/Stability of Interactions (time_day × Treatment)
# Dependencies: pandas, numpy, matplotlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# 0) Path Setup (Modify as needed)
# =========================
base_dir = Path(r"D:\FYP2\kelly fyp\Q2d_outputs\q2dma")
summary_csv = base_dir / "step4_5_summary_interactions.csv"   # File you generated
figdir = base_dir / "figs"
figdir.mkdir(parents=True, exist_ok=True)

# (Optional) Chinese font setup: Uncomment if you need Chinese titles/labels and have installed Chinese fonts locally
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']  # Or your local Chinese font name
# matplotlib.rcParams['axes.unicode_minus'] = False

# =========================
# 1) Read Data & Prepare
# =========================
df = pd.read_csv(summary_csv, encoding="utf-8-sig")

# Keep only the three interaction terms
term_map = {
    "time_day:Reducing_intracranial_pressure_treatment": "time×Reducing_intracranial_pressure_treatment",
    "time_day:Hemostatic_treatment":                     "time×Hemostatic_treatment",
    "time_day:Blood_pressure_lowering_treatment":        "time×Blood_pressure_lowering_treatment",
}
df = df[df["term"].isin(term_map.keys())].copy()
df["term_cn"] = df["term"].map(term_map)

# Calculate 95% CI
if {"estimate","std_error"}.issubset(df.columns):
    df["ci_low"]  = df["estimate"] - 1.96*df["std_error"]
    df["ci_high"] = df["estimate"] + 1.96*df["std_error"]
else:
    # If std_error is missing, don't plot CI
    df["ci_low"] = np.nan
    df["ci_high"] = np.nan

# Time window order (aligned automatically with actual columns/windows in the file)
all_windows = ["full","0-48h","0-72h","0-96h","0-168h"]
windows = [w for w in all_windows if w in df["window"].unique()]
win_index = {w:i for i,w in enumerate(windows)}
df["win_order"] = df["window"].map(win_index)

# =========================
# 2) Interaction Coefficients by Time Window (with 95% CI)
# =========================
def plot_coefs_by_window(dfo, outcome_name, out_png):
    plt.figure(figsize=(9,6))
    for term_cn, dft in dfo.groupby("term_cn"):
        dft = dft.set_index("window").reindex(windows)
        x = np.arange(len(windows))
        y = dft["estimate"].values
        # If std_error is available, plot error bars
        if "std_error" in dft.columns and dft["std_error"].notna().any():
            yerr = 1.96 * dft["std_error"].values
            plt.errorbar(x, y, yerr=yerr, marker="o", linewidth=2, capsize=3, label=term_cn)
        else:
            plt.plot(x, y, marker="o", linewidth=2, label=term_cn)
    plt.axhline(0, linestyle="--")
    plt.xticks(np.arange(len(windows)), windows)
    plt.xlabel("Time-window ")
    plt.ylabel("Interaction coefficient (estimated value ±95%CI)")
    plt.title(f"{outcome_name}: time* treatment changes")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.show()

plot_coefs_by_window(df[df["outcome"]=="ED"],   "Edema volume(ED)",   figdir / "viz_ED_coef_by_window.png")
plot_coefs_by_window(df[df["outcome"]=="HEMO"], "Hematoma(Hemo)", figdir / "viz_HEMO_coef_by_window.png")

# =========================
# 3) Significance Heatmap (-log10 p; values shown inside cells)
# =========================
def plot_sig_heatmap(dfo, outcome_name, out_png):
    # Matrix: rows=term_cn, columns=window, values=-log10(p)
    row_labels, M = [], []
    for term_cn, dft in dfo.groupby("term_cn"):
        dft = dft.set_index("window").reindex(windows)
        vals = pd.to_numeric(dft["p_value"], errors="coerce")
        M.append(-np.log10(vals.values))
        row_labels.append(term_cn)
    M = np.array(M)

    plt.figure(figsize=(8, 3 + 0.5*len(row_labels)))
    im = plt.imshow(M, aspect="auto")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.yticks(np.arange(len(row_labels)), row_labels)
    plt.xticks(np.arange(len(windows)), windows)

    # Annotate cells with p-values
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            pv = dfo[dfo["term_cn"]==row_labels[i]].set_index("window").reindex(windows)["p_value"].iloc[j]
            txt = f"{pv:.3f}" if pd.notna(pv) else ""
            plt.text(j, i, txt, ha="center", va="center")

    plt.title(f"{outcome_name} Significance heat map (-log10 p)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.show()

plot_sig_heatmap(df[df["outcome"]=="ED"],   "Edema volume(ED)",   figdir / "viz_ED_sig_heatmap.png")
plot_sig_heatmap(df[df["outcome"]=="HEMO"], "Hematoma(Hemo)", figdir / "viz_HEMO_sig_heatmap.png")

# =========================
# 4) Full Sample "Forest Plot" Style (Estimated Values ± 95% CI)
# =========================
def forest_plot(dfi, outcome_name, out_png):
    dfi = dfi[dfi["window"]=="full"].copy()
    if dfi.empty:
        return
    dfi = dfi.sort_values("estimate")
    y = np.arange(len(dfi))
    if "std_error" in dfi.columns and dfi["std_error"].notna().any():
        plt.figure(figsize=(7, 4 + 0.3*len(dfi)))
        plt.errorbar(dfi["estimate"], y, xerr=1.96*dfi["std_error"], fmt="o", capsize=3)
        plt.axvline(0, linestyle="--")
        plt.yticks(y, dfi["term_cn"])
        plt.xlabel("Interaction coefficient (estimated value ±95%CI)")
        plt.title(f"{outcome_name}: Full Sample interaction term forest plot")
        plt.tight_layout()
        plt.savefig(out_png, dpi=150)
        plt.show()

forest_plot(df[df["outcome"]=="ED"],   "Edema volume(ED)",   figdir / "viz_ED_forest_full.png")
forest_plot(df[df["outcome"]=="HEMO"], "Hematoma(Hemo)", figdir / "viz_HEMO_forest_full.png")

# =========================
# 5) (Optional) Bar Chart with Error Bars for One Time Window
#    Change window_to_plot to "full" / "0-72h" / "0-168h" etc.
# =========================
def bar_with_error(dfo, outcome_name, window_to_plot, out_png):
    sub = dfo[dfo["window"]==window_to_plot].copy()
    if sub.empty:
        return
    sub = sub.sort_values("term_cn")
    x = np.arange(len(sub))
    plt.figure(figsize=(7,5))
    if "std_error" in sub.columns and sub["std_error"].notna().any():
        plt.bar(x, sub["estimate"], yerr=1.96*sub["std_error"], capsize=3)
    else:
        plt.bar(x, sub["estimate"])
    plt.axhline(0, linestyle="--")
    plt.xticks(x, sub["term_cn"])
    plt.ylabel("Interaction term coefficient")
    plt.title(f"{outcome_name}: {window_to_plot} Bar chart (±95%CI)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.show()

# Example: Plot full and 0-168h (change here for other time windows)
if "full" in windows:
    bar_with_error(df[df["outcome"]=="ED"],   "Edema volume(ED)",   "full",   figdir / "viz_ED_bar_full.png")
    bar_with_error(df[df["outcome"]=="HEMO"], "Hematoma(Hemo)", "full",   figdir / "viz_HEMO_bar_full.png")
if "0-168h" in windows:
    bar_with_error(df[df["outcome"]=="ED"],   "Edema volume(ED)",   "0-168h", figdir / "viz_ED_bar_0_168h.png")
    bar_with_error(df[df["outcome"]=="HEMO"], "Hematoma(Hemo)", "0-168h", figdir / "viz_HEMO_bar_0_168h.png")

print("Figures saved to:", figdir)

In [ ]:
# -*- coding: utf-8 -*-
# Functionality:
# 1) Refit Step4 FE-OLS (ED/Hemo)
# 2) Plot Marginal Effect Curves (On vs Off) + 95% CI Shading (ED/Hemo × 3 Treatments)
# 3) Synthesize a Single Forest Plot (ED and Hemo stacked vertically)
#
# Dependencies: pandas, numpy, statsmodels, patsy, matplotlib

from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy import dmatrix
import matplotlib.pyplot as plt

# =========================
# 0) Paths (Modify as needed)
# =========================
base_dir = Path(r"D:\FYP2\kelly fyp")                     # Your project root directory
indir    = base_dir / "Q2d_outputs" / "q2dma"             # Output directory from Steps 1-2
t1_path  = base_dir / "Table1-Patient_List_and_Clinical_Information.xlsx"
long_csv = indir / "longitudinal_data_with_hours_since_onset.csv"

out_dir  = indir                                          # Charts and tables still saved in q2dma
fig_dir  = out_dir / "figs"
fig_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Read & Preprocess (Consistent with Step4)
# =========================
df = pd.read_csv(long_csv, encoding="utf-8-sig")
t1 = pd.read_excel(t1_path)

df["ID"] = df["ID"].astype(str).str.strip().str.lower()
t1["ID"] = t1["ID"].astype(str).str.strip().str.lower()

treat_cols = [
    "Ventricular_drainage_treatment",
    "Hemostatic_treatment",
    "Reducing_intracranial_pressure_treatment",
    "Blood_pressure_lowering_treatment",
    "Sedation_and_analgesic_treatment",
    "Stop_vomiting_and_protect_stomach_treatment",
    "Neurotrophic_treatment",
]
covar_cols = ["Age","Gender","mRS_Score_before_cerebral_hemorrhage"]

t1_sub = t1[["ID"] + treat_cols + covar_cols].copy()
for c in treat_cols:
    t1_sub[c] = pd.to_numeric(t1_sub[c], errors="coerce").fillna(0).astype(int)

# Gender dummy variables
t1_sub["Gender"] = t1_sub["Gender"].astype(str).str.strip().str.lower()
gender_dummies = pd.get_dummies(t1_sub["Gender"], prefix="Gender")
t1_sub = pd.concat([t1_sub.drop(columns=["Gender"]), gender_dummies], axis=1)

# Merge
df = df.merge(t1_sub, on="ID", how="inner")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["mRS_Score_before_cerebral_hemorrhage"] = pd.to_numeric(df["mRS_Score_before_cerebral_hemorrhage"], errors="coerce")

# Baseline (log volumes at first time point)
idx_first = df.groupby("ID")["hours_since_onset"].idxmin()
base = df.loc[idx_first, ["ID","log_Hemo","log_ED"]].rename(columns={"log_Hemo":"baseline_log_Hemo","log_ED":"baseline_log_ED"})
df = df.merge(base, on="ID", how="left")

# Ensure necessary columns are complete
df = df.dropna(subset=["log_ED","log_Hemo","time_day","Age","baseline_log_ED","baseline_log_Hemo"])

# Three treatments interacting with time (recommended to keep three to avoid excessive collinearity)
key_ts = [
    "Reducing_intracranial_pressure_treatment",
    "Hemostatic_treatment",
    "Blood_pressure_lowering_treatment"
]

# =========================
# 2) Fit FE-OLS (Patient Fixed Effects + Cluster-Robust SE by ID)
# =========================
def fit_fe_ols(data: pd.DataFrame, outcome: str):
    main_part = " + ".join(key_ts)
    inter_part = " + ".join([f"time_day:{t}" for t in key_ts])
    gender_terms = " + ".join([c for c in data.columns if c.startswith("Gender_")]) or "0"
    base_term = "baseline_log_ED + baseline_log_Hemo"
    formula = f"{outcome} ~ time_day + {main_part} + {inter_part} + Age + mRS_Score_before_cerebral_hemorrhage + {base_term} + {gender_terms} + C(ID)"
    model = smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": data["ID"]})
    return model, formula

m_ED,   f_ED   = fit_fe_ols(df, "log_ED")
m_HEMO, f_HEMO = fit_fe_ols(df, "log_Hemo")

# =========================
# 3) Generate "Marginal Effects + 95% CI Shading" (On vs Off)
#    - Use covariance matrix to calculate prediction variance: Var(xβ)=xΣx'
#    - Other covariates set to median, gender dummies set to median (or proportion)
# =========================
def marginal_with_ci(model, data, outcome_name: str, treat_name: str, out_png: Path, n_grid: int = 120):
    # Time grid
    t_min, t_max = data["time_day"].quantile(0.01), data["time_day"].quantile(0.99)
    grid = pd.DataFrame({"time_day": np.linspace(t_min, t_max, n_grid)})
    # Set covariates to median
    grid["Age"] = data["Age"].median()
    grid["mRS_Score_before_cerebral_hemorrhage"] = data["mRS_Score_before_cerebral_hemorrhage"].median()
    grid["baseline_log_ED"]   = data["baseline_log_ED"].median()
    grid["baseline_log_Hemo"] = data["baseline_log_Hemo"].median()
    # Set other treatments to 0
    for c in key_ts:
        grid[c] = 0
    # Set gender dummies to median (or proportion)
    for c in [col for col in data.columns if col.startswith("Gender_")]:
        grid[c] = data[c].median()

    # Placeholder for fixed effects: use one real ID (other C(ID) dummies will be 0 in design matrix)
    grid["ID"] = data["ID"].iloc[0]

    # Construct design matrix using patsy and align column order with model
    def predict_curve(treat_on: bool):
        g = grid.copy()
        g[treat_name] = 1 if treat_on else 0
        X = dmatrix(model.model.data.design_info.builder, g, return_type="dataframe")
        # Align columns
        for col in model.model.exog_names:
            if col not in X.columns:
                X[col] = 0.0
        X = X[model.model.exog_names]
        beta = model.params.values
        cov  = model.cov_params().values  # Cluster-robust covariance
        yhat = X.values @ beta
        # Variance of prediction (linear combination of point estimates)
        var  = np.einsum("ij,jk,ik->i", X.values, cov, X.values)
        se   = np.sqrt(np.clip(var, 0, np.inf))
        lo   = yhat - 1.96*se
        hi   = yhat + 1.96*se
        return g["time_day"].values, yhat, lo, hi

    x0, y0, lo0, hi0 = predict_curve(False)  # Off
    x1, y1, lo1, hi1 = predict_curve(True)   # On

    plt.figure(figsize=(8.5,5.2))
    # Off
    plt.plot(x0, y0, label="Off")              # Line
    plt.fill_between(x0, lo0, hi0, alpha=0.25) # Shading band
    # On
    plt.plot(x1, y1, label="On")
    plt.fill_between(x1, lo1, hi1, alpha=0.25)

    plt.xlabel("Days since onset")
    plt.ylabel(f"Predicted {outcome_name}")
    plt.title(f"Marginal effect with 95% CI: {treat_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.show()

# Plot for each of the three treatments (three plots each for ED/Hemo)
for tr in key_ts:
    marginal_with_ci(m_ED,   df, "log_ED",   tr, fig_dir / f"ED_marginal_CI_{tr}.png")
    marginal_with_ci(m_HEMO, df, "log_Hemo", tr, fig_dir / f"HEMO_marginal_CI_{tr}.png")

# =========================
# 4) Synthesize a Single Forest Plot (ED and Hemo as two subplots vertically)
#    - Only Full sample (the three interaction terms from this full-sample model)
#    - Coefficients ± 95% CI
# =========================
def extract_interactions(model):
    # Extract coefficients and SE for the three interaction terms
    rows = []
    cov = model.cov_params()
    for tr in key_ts:
        name = f"time_day:{tr}"
        if name in model.params.index:
            est = model.params[name]
            se  = model.bse[name]
            rows.append({"term": name, "estimate": est, "se": se})
    tab = pd.DataFrame(rows)
    # Friendly display names
    cn_map = {
        "time_day:Reducing_intracranial_pressure_treatment": "time×Reducing Intracranial Pressure",
        "time_day:Hemostatic_treatment":                     "time×Hemostatic",
        "time_day:Blood_pressure_lowering_treatment":        "time×Blood Pressure Lowering",
    }
    tab["term_cn"] = tab["term"].map(cn_map)
    tab["lo95"] = tab["estimate"] - 1.96*tab["se"]
    tab["hi95"] = tab["estimate"] + 1.96*tab["se"]
    return tab.sort_values("estimate")

tab_ED   = extract_interactions(m_ED)
tab_HEMO = extract_interactions(m_HEMO)

# Plot both on one page
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
for ax, tab, title in zip(axes, [tab_ED, tab_HEMO], ["ED (Edema Volume)", "Hemo (Hematoma Volume)"]):
    y = np.arange(len(tab))
    ax.errorbar(tab["estimate"], y, xerr=1.96*tab["se"], fmt="o", capsize=3)
    ax.axvline(0, linestyle="--")
    ax.set_yticks(y)
    ax.set_yticklabels(tab["term_cn"])
    ax.set_xlabel("Interaction coefficient (estimate ± 95% CI)")
    ax.set_title(title)
fig.tight_layout()
out_forest = fig_dir / "Forest_Combined_ED_HEMO.png"
fig.savefig(out_forest, dpi=150)
plt.show()

print("✅ Generated marginal effect curves with 95% CI shading (ED/Hemo × 3 treatments)")
print("✅ Generated combined forest plot:", out_forest)
print("Image directory:", fig_dir)

In [ ]:
# M2-3 Analysis (exact highest-frequency combo vs all others)
# Outputs:
#   /mnt/data/Q2_3_results_exact_combo.csv
#   /mnt/data/Q2_3_top_combos_counts.csv
#   /mnt/data/Q2_3_balance_and_sizes.txt

import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from statsmodels.api import Logit, GLM, families

BASE = Path("/Users/xiexintong/Desktop")

# Load
resid = pd.read_excel(BASE/"patient_residuals_final_choice.xlsx")
t1 = pd.read_excel(BASE/"Table1-Patient_List_and_Clinical_Information.xlsx")

ID_COL_RESID = "ID"; GROUP_COL = "Group"; ID_COL_T1 = "ID"

TREAT_COLS = {
    "Hemostatic": "Hemostatic_treatment",
    "ICP lowering": "Reducing_intracranial_pressure_treatment",
    "Neurotrophic": "Neurotrophic_treatment",
    "Sedation/Analgesia": "Sedation_and_analgesic_treatment",
    "EVD": "Ventricular_drainage_treatment",
    "BP lowering": "Blood_pressure_lowering_treatment",
    "Antiemetic/GI-protect": "Stop_vomiting_and_protect_stomach_treatment"
}
TREAT_ORDER = ["Hemostatic","ICP lowering","Neurotrophic","Sedation/Analgesia","EVD","BP lowering","Antiemetic/GI-protect"]
COVARS = ["Age", "Time_from_Onset_to_First_Imaging", "History_of_Hypertension"]

bt = t1[[ID_COL_T1] + list(TREAT_COLS.values()) + COVARS].copy()

def to_bin(x): return pd.to_numeric(x, errors="coerce").fillna(0).astype(int)
for k, col in TREAT_COLS.items(): bt[col] = to_bin(bt[col])

def make_combo(row):
    on = [name for name, col in TREAT_COLS.items() if row[col] == 1]
    on_sorted = [name for name in TREAT_ORDER if name in on]
    return "+".join(on_sorted) if on_sorted else "None"
bt["combo"] = bt.apply(make_combo, axis=1)

# Restrict to IDs in residuals (first 100 cohort)
first_ids = set(resid[ID_COL_RESID].astype(str))
bt_sub = bt[bt[ID_COL_T1].astype(str).isin(first_ids)].copy()

# Highest-frequency combo within this cohort
top_combo = bt_sub["combo"].value_counts().idxmax()

# TreatmentMain by exact string match
bt["treatment_main"] = (bt["combo"] == top_combo).astype(int)

# Merge & define foldings
m = resid[[ID_COL_RESID, GROUP_COL]].merge(
    bt[[ID_COL_T1, "combo", "treatment_main"] + COVARS],
    left_on=ID_COL_RESID, right_on=ID_COL_T1, how="left"
)
m[GROUP_COL] = m[GROUP_COL].astype(str)
m["worsen_A1"] = (m[GROUP_COL].isin(["1"])).astype(int)
m["worsen_A2"] = (m[GROUP_COL].isin(["1","3"])).astype(int)
m["benign_A3"] = (m[GROUP_COL].isin(["0","2","4"])).astype(int)

def dropna_use(df, y):
    return df.dropna(subset=["treatment_main"] + COVARS + [y]).copy()

def overlap_logit(frame, y_col, covar_cols, treat_col="treatment_main", robust_cov="HC3"):
    ps_X = sm.add_constant(frame[covar_cols], has_constant='add')
    ps_y = frame[treat_col]
    ps_fit = Logit(ps_y, ps_X).fit(disp=0, maxiter=200)
    ps = ps_fit.predict(ps_X)
    w = frame[treat_col]*(1-ps) + (1-frame[treat_col])*ps
    WX = sm.add_constant(frame[[treat_col] + covar_cols], has_constant='add')
    Wy = frame[y_col]
    glm_w = GLM(Wy, WX, family=families.Binomial(), freq_weights=w).fit(cov_type=robust_cov)
    coef = glm_w.params[treat_col]; se = glm_w.bse[treat_col]
    OR = float(np.exp(coef)); CI_low = float(np.exp(coef - 1.96*se)); CI_high = float(np.exp(coef + 1.96*se))
    return {"OR": OR, "CI_low": CI_low, "CI_high": CI_high}

def bootstrap_or(frame, y_col, covar_cols, B=200, seed=123):
    rng = np.random.default_rng(seed); ors=[]; n=len(frame)
    for _ in range(B):
        idx = rng.integers(0, n, n)
        bb = frame.iloc[idx].copy()
        try:
            est = overlap_logit(bb, y_col, covar_cols)
            ors.append(est["OR"])
        except Exception:
            pass
    if not ors: return {"median": np.nan, "low": np.nan, "high": np.nan, "N": 0, "pct_gt1": np.nan}
    qs = np.quantile(ors, [0.025,0.5,0.975])
    return {"median": float(qs[1]), "low": float(qs[0]), "high": float(qs[2]), "N": int(len(ors)), "pct_gt1": float((np.array(ors)>1).mean())}

foldings = [
    ("A1_worsen","worsen_A1","<1 is better (less worsen)"),
    ("A2_worsen","worsen_A2","<1 is better (less worsen incl. group 3)"),
    ("A3_benign","benign_A3",">1 is better (more benign groups)")
]

rows = []
for tag, ycol, note in foldings:
    use = dropna_use(m, ycol)
    if use.empty:
        rows.append({"folding": tag, "main_combo": top_combo, "OR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
                     "B_median": np.nan, "B_low": np.nan, "B_high": np.nan, "B_N": 0,
                     "B_pct_OR_gt1": np.nan, "note": note})
        continue
    try:
        est = overlap_logit(use, ycol, COVARS)
        boot = bootstrap_or(use, ycol, COVARS, B=200, seed=123)
        rows.append({"folding": tag, "main_combo": top_combo,
                     "OR": est["OR"], "CI_low": est["CI_low"], "CI_high": est["CI_high"],
                     "B_median": boot["median"], "B_low": boot["low"], "B_high": boot["high"], "B_N": boot["N"],
                     "B_pct_OR_gt1": boot["pct_gt1"], "note": note})
    except Exception as e:
        rows.append({"folding": tag, "main_combo": top_combo, "OR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
                     "B_median": np.nan, "B_low": np.nan, "B_high": np.nan, "B_N": 0,
                     "B_pct_OR_gt1": np.nan, "note": f"{note} | ERROR: {e}"})

res = pd.DataFrame(rows)
res.to_csv(BASE/"Q2_3_results_exact_combo.csv", index=False)

# Save counts for transparency
bt_sub = bt[bt[ID_COL_T1].astype(str).isin(first_ids)].copy()
top_counts = bt_sub["combo"].value_counts().reset_index()
top_counts.columns = ["combo", "count"]
top_counts.to_csv(BASE/"Q2_3_top_combos_counts.csv", index=False)

with open(BASE/"Q2_3_balance_and_sizes.txt","w") as f:
    f.write(f"Top combo (within residuals' IDs): {top_combo}\n")
    f.write("TreatmentMain counts (exact string match) within cohort:\n")
    f.write(str(bt_sub.assign(treatment_main=(bt_sub['combo']==top_combo).astype(int))['treatment_main'].value_counts().to_dict())+"\n")
    f.write("Group counts (0..4):\n")
    f.write(str(resid[GROUP_COL].astype(str).value_counts().to_dict())+"\n")

In [ ]:
# ================================================================
# Stratified feasibility & covariate balance (within-group) analysis
# Data needed (filenames must match):
#   - patient_residuals_final_choice.xlsx  (must contain: ID, Group)
#   - Table1-Patient_List_and_Clinical_Information.xlsx
#       (must contain: ID, 7 treatment indicators, and covariates:
#        Age, Time_from_Onset_to_First_Imaging, History_of_Hypertension)
#
# Outputs (written to BASE):
#   - strata_counts_by_group.csv
#   - strata_covariate_balance_by_group.csv
#   - strata_counts_collapsed_groups.csv
#
# Note: No per-patient outcome is available, so this script focuses on
#       sample sizes, balance diagnostics, and feasibility by Group.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# 1) Paths — set BASE properly
# -----------------------------
# Option A (recommended): place this .py file in the same folder as your Excel files
# and leave BASE = Path(".")
# Option B (Mac Desktop): BASE = Path("/Users/<your_username>/Desktop")
# Option C (Windows Desktop): BASE = Path("C:/Users/<your_username>/Desktop")

BASE = Path("/Users/xiexintong/Desktop")  
RESIDUALS_FILE = BASE / "patient_residuals_final_choice.xlsx"
TABLE1_FILE    = BASE / "Table1-Patient_List_and_Clinical_Information.xlsx"

# -----------------------------
# 2) Load data
# -----------------------------
resid = pd.read_excel(RESIDUALS_FILE)
t1    = pd.read_excel(TABLE1_FILE)

ID_COL_RESID = "ID"
GROUP_COL    = "Group"
ID_COL_T1    = "ID"

# -----------------------------
# 3) Treatment columns & order
# -----------------------------
TREAT_COLS = {
    "Hemostatic":              "Hemostatic_treatment",
    "ICP lowering":            "Reducing_intracranial_pressure_treatment",
    "Neurotrophic":            "Neurotrophic_treatment",
    "Sedation/Analgesia":      "Sedation_and_analgesic_treatment",
    "EVD":                     "Ventricular_drainage_treatment",
    "BP lowering":             "Blood_pressure_lowering_treatment",
    "Antiemetic/GI-protect":   "Stop_vomiting_and_protect_stomach_treatment"
}
TREAT_ORDER = [
    "Hemostatic",
    "ICP lowering",
    "Neurotrophic",
    "Sedation/Analgesia",
    "EVD",
    "BP lowering",
    "Antiemetic/GI-protect"
]

# Baseline covariates for balance diagnostics
COVARS = ["Age", "Time_from_Onset_to_First_Imaging", "History_of_Hypertension"]

# -----------------------------
# 4) Build patient-level combo
# -----------------------------
bt = t1[[ID_COL_T1] + list(TREAT_COLS.values()) + COVARS].copy()

# force 0/1 integers for treatment indicators
for col in TREAT_COLS.values():
    bt[col] = pd.to_numeric(bt[col], errors="coerce").fillna(0).astype(int)

def make_combo(row) -> str:
    """Create a deterministic treatment-combo string from 0/1 flags in TREAT_ORDER."""
    on = [name for name, col in TREAT_COLS.items() if row[col] == 1]
    on_sorted = [name for name in TREAT_ORDER if name in on]
    return "+".join(on_sorted) if on_sorted else "None"

bt["combo"] = bt.apply(make_combo, axis=1)

# -----------------------------
# 5) Restrict to first-100 cohort (IDs present in residuals)
# -----------------------------
cohort_ids = set(resid[ID_COL_RESID].astype(str))
bt_sub = bt[bt[ID_COL_T1].astype(str).isin(cohort_ids)].copy()

# Identify top (highest-frequency) combo within this cohort
top_combo = bt_sub["combo"].value_counts().idxmax()
print("Top combo (highest frequency within cohort):")
print(f"  {top_combo}")
print()

# Define treatment_main by exact string equality to the top combo
bt["treatment_main"] = (bt["combo"] == top_combo).astype(int)

# -----------------------------
# 6) Merge with Group labels
# -----------------------------
m = resid[[ID_COL_RESID, GROUP_COL]].merge(
    bt[[ID_COL_T1, "treatment_main"] + COVARS],
    left_on=ID_COL_RESID, right_on=ID_COL_T1, how="left"
).rename(columns={ID_COL_RESID: "ID"})
m[GROUP_COL] = m[GROUP_COL].astype(str)  # keep as string "0".."4"

# -----------------------------
# 7) Within-group counts
# -----------------------------
counts = (
    pd.crosstab(m[GROUP_COL], m["treatment_main"])
    .rename(columns={0: "others(0)", 1: "main(1)"})
    .reset_index()
    .rename(columns={GROUP_COL: "Group"})
)
counts.loc[len(counts)] = ["Total", counts["others(0)"].sum(), counts["main(1)"].sum()]

print("Within-group counts (Group x treatment_main):")
print(counts)
print()

counts_path = BASE / "strata_counts_by_group.csv"
counts.to_csv(counts_path, index=False)
print(f"Saved: {counts_path}")

# -----------------------------
# 8) Within-group covariate balance
#     - For continuous covariates: mean, sd, SMD
#     - For binary History_of_Hypertension: proportions and difference
# -----------------------------
def smd_two_groups(x: pd.Series, g: pd.Series) -> float:
    """
    Standardized Mean Difference for a continuous variable x between g==1 and g==0.
    Returns NaN if one group is empty or pooled SD is 0.
    """
    a = pd.to_numeric(x[g == 1], errors="coerce").dropna().astype(float)
    b = pd.to_numeric(x[g == 0], errors="coerce").dropna().astype(float)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    ma, mb = a.mean(), b.mean()
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    denom = (len(a) + len(b) - 2)
    if denom <= 0:
        return np.nan
    sp = np.sqrt(((len(a)-1)*sa**2 + (len(b)-1)*sb**2) / denom)
    if sp <= 0 or np.isnan(sp):
        return np.nan
    return (ma - mb) / sp

def prop_and_diff_binary(x: pd.Series, g: pd.Series):
    """
    For a binary variable x (0/1), return:
      p_main, p_others, (p_main - p_others)
    """
    a = pd.to_numeric(x[g == 1], errors="coerce").dropna().astype(int)
    b = pd.to_numeric(x[g == 0], errors="coerce").dropna().astype(int)
    if len(a) == 0 or len(b) == 0:
        return (np.nan, np.nan, np.nan)
    p1 = a.mean()
    p0 = b.mean()
    return (p1, p0, p1 - p0)

rows = []
for grp, df in m.groupby(GROUP_COL):
    g = df["treatment_main"].astype(int)
    # Continuous covariates
    for var in ["Age", "Time_from_Onset_to_First_Imaging"]:
        x = pd.to_numeric(df[var], errors="coerce")
        rows.append({
            "Group": grp,
            "Covariate": var,
            "n_main":   int((g == 1).sum()),
            "n_others": int((g == 0).sum()),
            "mean_main":   float(x[g == 1].mean()) if (g == 1).any() else np.nan,
            "sd_main":     float(x[g == 1].std(ddof=1)) if (g == 1).any() else np.nan,
            "mean_others": float(x[g == 0].mean()) if (g == 0).any() else np.nan,
            "sd_others":   float(x[g == 0].std(ddof=1)) if (g == 0).any() else np.nan,
            "SMD":         float(smd_two_groups(x, g))
        })
    # Binary covariate
    hx = pd.to_numeric(df["History_of_Hypertension"], errors="coerce").fillna(0).astype(int)
    p1, p0, diff = prop_and_diff_binary(hx, g)
    rows.append({
        "Group": grp,
        "Covariate": "History_of_Hypertension",
        "n_main":   int((g == 1).sum()),
        "n_others": int((g == 0).sum()),
        "prop_main":   float(p1) if p1 == p1 else np.nan,
        "prop_others": float(p0) if p0 == p0 else np.nan,
        "Diff(p1-p0)": float(diff) if diff == diff else np.nan,
        "SMD": np.nan  # (we keep diff in proportions for binary)
    })

balance = pd.DataFrame(rows)
print("Within-group covariate balance (first 10 rows):")
print(balance.head(10))
print()

balance_path = BASE / "strata_covariate_balance_by_group.csv"
balance.to_csv(balance_path, index=False)
print(f"Saved: {balance_path}")

# -----------------------------
# 9) Collapsed groups for robustness counts
#    (2,4) => 'SustainedDecline(2+4)'
#    (0,3) => 'RiseThenDownOrStable(0+3)'
#     1    => 'SustainedRise(1)'
# -----------------------------
def collapse_label(g: str) -> str:
    if g in {"2", "4"}:
        return "SustainedDecline(2+4)"
    if g in {"0", "3"}:
        return "RiseThenDownOrStable(0+3)"
    return "SustainedRise(1)"

m["CollapsedGroup"] = m[GROUP_COL].map(collapse_label)

collapsed = (
    pd.crosstab(m["CollapsedGroup"], m["treatment_main"])
    .rename(columns={0: "others(0)", 1: "main(1)"})
    .reset_index()
)
print("Collapsed-group counts:")
print(collapsed)
print()

collapsed_path = BASE / "strata_counts_collapsed_groups.csv"
collapsed.to_csv(collapsed_path, index=False)
print(f"Saved: {collapsed_path}")

print("\nDone. (No per-patient outcome available, so this script focuses on stratified feasibility and balance.)")

In [ ]:
# ============================================================
# MOUDEL4.2-C Extension: add interaction Age × CardioHistory （加入交互项后模型变得极不稳定，属于分离问题，舍）
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
resid = pd.read_excel("patient_residuals_final_choice.xlsx")
t1    = pd.read_excel("Table1-Patient_List_and_Clinical_Information.xlsx")

# ------------------------------------------------------------
# Build combo
# ------------------------------------------------------------
treatment_map = {
    "Hemostatic": "Hemostatic_treatment",
    "ICP lowering": "Reducing_intracranial_pressure_treatment",
    "Neurotrophic": "Neurotrophic_treatment",
    "Sedation/Analgesia": "Sedation_and_analgesic_treatment",
    "EVD": "Ventricular_drainage_treatment",
    "BP lowering": "Blood_pressure_lowering_treatment",
    "Antiemetic/GI-protect": "Stop_vomiting_and_protect_stomach_treatment"
}
order_list = list(treatment_map.keys())

def make_combo(row):
    on = []
    for name, col in treatment_map.items():
        val = pd.to_numeric(row[col], errors="coerce")
        if val == 1:
            on.append(name)
    ordered = [x for x in order_list if x in on]
    return "+".join(ordered) if ordered else "None"

t1["combo"] = t1.apply(make_combo, axis=1)

df = resid.merge(t1, on="ID", how="left")

# ------------------------------------------------------------
# Outcome: bad trajectory (Group 1 or 3)
# ------------------------------------------------------------
df["Y_bad"] = df["Group"].isin([1,3]).astype(int)

# ------------------------------------------------------------
# Identify main combo correctly
# ------------------------------------------------------------
top_combo = df["combo"].value_counts().idxmax()
print("Top combo =", top_combo)

df["TreatmentMain"] = (df["combo"] == top_combo).astype(int)

# ------------------------------------------------------------
# Create merged covariates
# ------------------------------------------------------------
bin_hist_cols = [
    "History_of_Hypertension",
    "History_of_Stroke",
    "History_of_Diabetes",
    "History_of_Coronary_Artery_Disease",
    "History_of_Atrial_Fibrillation",
    "History_of_Smoking",
    "History_of_Alcohol_Consumption"
]

for col in bin_hist_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df["CardioHistory"] = (
    (df["History_of_Hypertension"] == 1) |
    (df["History_of_Stroke"] == 1) |
    (df["History_of_Coronary_Artery_Disease"] == 1) |
    (df["History_of_Atrial_Fibrillation"] == 1)
).astype(int)

df["MetabolicLifestyleRisk"] = (
    (df["History_of_Diabetes"] == 1) |
    (df["History_of_Smoking"] == 1) |
    (df["History_of_Alcohol_Consumption"] == 1)
).astype(int)

df["PreStrokeDisability"] = (
    pd.to_numeric(df["mRS_Score_before_cerebral_hemorrhage"], errors="coerce").fillna(0) > 0
).astype(int)

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Time_from_Onset_to_First_Imaging"] = pd.to_numeric(
    df["Time_from_Onset_to_First_Imaging"], errors="coerce"
)

# ------------------------------------------------------------
# Add interaction term: Age × CardioHistory
# ------------------------------------------------------------
df["AgeXCardio"] = df["Age"] * df["CardioHistory"]

# ------------------------------------------------------------
# Define predictors
# ------------------------------------------------------------
predictors = [
    "TreatmentMain",
    "Age",
    "CardioHistory",
    "AgeXCardio",
    "Time_from_Onset_to_First_Imaging",
    "MetabolicLifestyleRisk",
    "PreStrokeDisability"
]

df_model = df.dropna(subset=["Age", "Time_from_Onset_to_First_Imaging"]).copy()

X = df_model[predictors].copy()
Y = df_model["Y_bad"]

X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

# ------------------------------------------------------------
# Final logistic regression with interaction
# ------------------------------------------------------------
X_logit = sm.add_constant(X)
logit_model = sm.Logit(Y, X_logit).fit(disp=0)

print("\n========== Logistic Regression with Interaction ==========")
print(logit_model.summary())

OR = np.exp(logit_model.params)
CI = np.exp(logit_model.conf_int())

print("\nOdds Ratios:")
print(OR)
print("\n95% CI:")
print(CI)

results_df = pd.DataFrame({
    "variable": X_logit.columns,
    "coef": logit_model.params,
    "OR": OR,
    "CI_low": CI[0],
    "CI_high": CI[1]
})
results_df.to_csv("Q2C_logistic_Ybad_with_interaction.csv", index=False)
print("\nSaved results to Q2C_logistic_Ybad_with_interaction.csv")

In [ ]:
import matplotlib.pyplot as plt

years = [2019, 2020, 2021, 2022, 2023, 2024]
market_size = [39, 25, 33, 41, 52, 64]  # in billion RMB
growth_rate = [-10, -30, 15, 20, 28, 35]  # %

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(years, market_size, color="#3B82F6", label="Market Size (100 Million RMB)")
ax1.set_xlabel("Year")
ax1.set_ylabel("Market Size (100 Million RMB)", color="#3B82F6")

ax2 = ax1.twinx()
ax2.plot(years, growth_rate, color="#16A34A", marker="o", linewidth=2.5, label="Growth Rate (%)")
ax2.set_ylabel("Growth Rate (%)", color="#16A34A")

plt.title("China NFC Juice Market Size and Growth Trend (2019–2024)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# === Data (converted to English labels) ===
labels = [
    "Dietary Fiber", "Vitamin C", "Energy", "Vitamin B",
    "Vitamin E", "Zinc", "Sodium", "Carotene", "Iron",
    "Carrot Element", "Anthocyanins", "Electrolytes",
    "Fruit Acids", "Lutein"
]

values = [
    76.8, 74.8, 46.7, 41.9, 39.6, 25.2, 20.3, 19.5, 
    19.5, 18.9, 18.7, 18.1, 17.3, 13.4
]

# === Plot ===
plt.figure(figsize=(12, 6))
bars = plt.bar(labels, values, color="#4A90E2")  # Blue tone

# Value labels above bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1,
             f"{height}%", ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel("Consumer Preference Rate (%)", fontsize=12)
plt.title("Consumer Preference for NFC Juice Nutrients in China (2023)", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
#MOUDEL-C: Factors associated with bad vs non-bad trajectories
# Outcome: Y_bad = 1 if Group in {1,3}, else 0
# Predictors:
#   - Baseline clinical features (engineered/merged)
#   - TreatmentMain (1 = top combo, 0 = other combos)
#
# Files required (same folder as this script):
#   - patient_residuals_final_choice.xlsx
#   - Table1-Patient_List_and_Clinical_Information.xlsx
#
# Steps:
#   1) Build treatment combo per patient from 7 treatment indicators
#   2) Identify top-frequency combo within the 100-patient cohort
#   3) Define TreatmentMain = 1 if using this top combo
#   4) Define Y_bad = 1 if Group in {1,3}, else 0
#   5) Create merged/engineered covariates (CardioHistory, MetabolicLifestyleRisk, etc.)
#   6) LASSO logistic (for variable importance)
#   7) Final multivariable logistic (all predictors together)
#   8) VIF diagnostics for multicollinearity
#   9) ROC / AUC for overall discrimination
#  10) Bootstrap OR robustness check for TreatmentMain
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_curve, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
resid = pd.read_excel("patient_residuals_final_choice.xlsx")  # Contains ID, Group
t1    = pd.read_excel("Table1-Patient_List_and_Clinical_Information.xlsx")

# ------------------------------------------------------------
# 2. Build treatment combo string from 7 treatment indicators
# ------------------------------------------------------------

# Mapping from display name -> column name in Table1
treatment_map = {
    "Hemostatic": "Hemostatic_treatment",
    "ICP lowering": "Reducing_intracranial_pressure_treatment",
    "Neurotrophic": "Neurotrophic_treatment",
    "Sedation/Analgesia": "Sedation_and_analgesic_treatment",
    "EVD": "Ventricular_drainage_treatment",
    "BP lowering": "Blood_pressure_lowering_treatment",
    "Antiemetic/GI-protect": "Stop_vomiting_and_protect_stomach_treatment"
}

# Fixed order of labels when we build combo strings
order_list = list(treatment_map.keys())

def make_combo(row):
    on = []
    for name, col in treatment_map.items():

        # Read original cell
        raw = str(row[col]).strip().lower()

        # Interpret 1-like values
        if raw in ["1", "1.0", "true", "yes", "y", "√", "✓"]:
            on.append(name)
            continue
        
        # Fallback to numeric
        val = pd.to_numeric(raw, errors="coerce")
        if pd.notna(val) and val >= 1:
            on.append(name)

    # maintain fixed order
    ordered = [x for x in order_list if x in on]
    return "+".join(ordered) if ordered else "None"

t1["combo"] = t1.apply(make_combo, axis=1)

# ------------------------------------------------------------
# 3. Merge residuals (trajectory group) + Table1 (baseline + combo)
# ------------------------------------------------------------
df = resid.merge(t1, on="ID", how="left")

# Restrict to non-missing Group (should already be 100 or 96)
df = df.dropna(subset=["Group"]).copy()

# ------------------------------------------------------------
# 4. Identify top-frequency combo (automatically, based on THIS dataset)
# ------------------------------------------------------------
top_combo = df["combo"].value_counts().idxmax()
print("Top combo detected from data =", top_combo)

# Expected (from your description):
# "Hemostatic+ICP lowering+Neurotrophic+Sedation/Analgesia"
# But we do NOT hard-code it; we always take the real top combo.

df["TreatmentMain"] = (df["combo"] == top_combo).astype(int)
print("\nTreatmentMain counts (0 = other combos, 1 = main combo):")
print(df["TreatmentMain"].value_counts(dropna=False))

# ------------------------------------------------------------
# 5. Define outcome: bad vs non-bad trajectories
#    Bad trajectories: Group in {1,3}
# ------------------------------------------------------------
df["Y_bad"] = df["Group"].isin([1, 3]).astype(int)

print("\nY_bad counts (1 = bad trajectory, 0 = non-bad):")
print(df["Y_bad"].value_counts(dropna=False))

# ------------------------------------------------------------
# 6. Create engineered covariates (merging several risk factors)
# ------------------------------------------------------------

# Convert all binary history variables to numeric 0/1
bin_hist_cols = [
    "History_of_Hypertension",
    "History_of_Stroke",
    "History_of_Diabetes",
    "History_of_Coronary_Artery_Disease",
    "History_of_Atrial_Fibrillation",
    "History_of_Smoking",
    "History_of_Alcohol_Consumption"
]

for col in bin_hist_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# CardioHistory: at least one cardiovascular history
df["CardioHistory"] = (
    (df["History_of_Hypertension"] == 1) |
    (df["History_of_Stroke"] == 1) |
    (df["History_of_Coronary_Artery_Disease"] == 1) |
    (df["History_of_Atrial_Fibrillation"] == 1)
).astype(int)

# Metabolic/Lifestyle: diabetes, smoking, or alcohol
df["MetabolicLifestyleRisk"] = (
    (df["History_of_Diabetes"] == 1) |
    (df["History_of_Smoking"] == 1) |
    (df["History_of_Alcohol_Consumption"] == 1)
).astype(int)

# Pre-stroke disability indicator from mRS (if >0)
df["PreStrokeDisability"] = (
    pd.to_numeric(df["mRS_Score_before_cerebral_hemorrhage"], errors="coerce").fillna(0) > 0
).astype(int)

# Continuous variables: Age, Time_from_Onset_to_First_Imaging
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Time_from_Onset_to_First_Imaging"] = pd.to_numeric(
    df["Time_from_Onset_to_First_Imaging"], errors="coerce"
)

# ------------------------------------------------------------
# 7. Define predictor set for modeling
# ------------------------------------------------------------
predictors = [
    "TreatmentMain",          # whether patient received the top combo
    "Age",
    "Time_from_Onset_to_First_Imaging",
    "CardioHistory",
    "MetabolicLifestyleRisk",
    "PreStrokeDisability"
]

# Drop rows with missing Age or Time (if any)
df_model = df.dropna(subset=["Age", "Time_from_Onset_to_First_Imaging"]).copy()
print("\nModeling sample size:", len(df_model))

X = df_model[predictors].copy()
Y = df_model["Y_bad"]

# Make sure numeric
for col in predictors:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

# ------------------------------------------------------------
# 8. LASSO logistic (for variable importance, optional)
# ------------------------------------------------------------
print("\n========================")
print("LASSO logistic (Y_bad)")
print("========================")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
predictor_names = predictors

lasso = LogisticRegressionCV(
    cv=10,
    penalty="l1",
    solver="liblinear",
    scoring="neg_log_loss",
    max_iter=500,
    refit=True
)
lasso.fit(X_scaled, Y)

coef_lasso = lasso.coef_[0]
selected = [v for v, c in zip(predictor_names, coef_lasso) if abs(c) > 1e-6]

print("LASSO-selected predictors:", selected)

# 可选：保存 LASSO 系数
lasso_coef_df = pd.DataFrame({
    "variable": predictor_names,
    "coef": coef_lasso
})
lasso_coef_df.to_csv("Q2C_LASSO_coefficients_Ybad.csv", index=False)
print("Saved LASSO coefficients to Q2C_LASSO_coefficients_Ybad.csv")

# ------------------------------------------------------------
# 9. Final multivariable logistic model (all predictors)
# ------------------------------------------------------------
print("\n====================================")
print("Final multivariable logistic (Y_bad)")
print("====================================")

X_logit = sm.add_constant(X)
logit_model = sm.Logit(Y, X_logit).fit(disp=0)
print(logit_model.summary())

OR = np.exp(logit_model.params)
CI = np.exp(logit_model.conf_int())

print("\nOdds Ratios:")
print(OR)
print("\n95% CI:")
print(CI)

results_df = pd.DataFrame({
    "variable": X_logit.columns,
    "coef": logit_model.params,
    "OR": OR,
    "CI_low": CI[0],
    "CI_high": CI[1],
    "p_value": logit_model.pvalues
})
results_df.to_csv("Q2C_logistic_Ybad_results.csv", index=False)
print("\nSaved logistic results to Q2C_logistic_Ybad_results.csv")

# ------------------------------------------------------------
# 10. VIF diagnostics (multicollinearity check, no automatic dropping)
# ------------------------------------------------------------
print("\n====================================")
print("VIF diagnostics for predictors")
print("====================================")

X_vif = X.copy()
X_vif_const = sm.add_constant(X_vif)

vif_rows = []
for i, col in enumerate(X_vif_const.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X_vif_const.values, i)
    vif_rows.append({"variable": col, "VIF": vif_val})

vif_df = pd.DataFrame(vif_rows)
print(vif_df)

vif_df.to_csv("Q2C_VIF_predictors_Ybad.csv", index=False)
print("Saved VIF table to Q2C_VIF_predictors_Ybad.csv")

# ------------------------------------------------------------
# 11. ROC / AUC for overall discrimination
# ------------------------------------------------------------
print("\n====================================")
print("ROC / AUC for Y_bad")
print("====================================")

y_true = Y.values
y_prob = logit_model.predict(X_logit)  # predicted probability of Y_bad=1

auc = roc_auc_score(y_true, y_prob)
fpr, tpr, thresholds = roc_curve(y_true, y_prob)

print("AUC:", auc)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Y_bad (final logistic model)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig("Q2C_ROC_Ybad_final_logit.png", dpi=300)
plt.close()
print("Saved ROC curve to Q2C_ROC_Ybad_final_logit.png")

# ------------------------------------------------------------
# 12. Bootstrap OR robustness check for TreatmentMain
# ------------------------------------------------------------
print("\n====================================")
print("Bootstrap OR robustness for TreatmentMain")
print("====================================")

def fit_treatment_or(data):
    """Fit logistic on a bootstrap sample and return OR for TreatmentMain."""
    Xb = data[predictors].copy()
    Yb = data["Y_bad"]
    for col in predictors:
        Xb[col] = pd.to_numeric(Xb[col], errors="coerce").fillna(0)
    Xb_logit = sm.add_constant(Xb)
    try:
        m = sm.Logit(Yb, Xb_logit).fit(disp=0)
        coef = m.params["TreatmentMain"]
        return float(np.exp(coef))
    except Exception:
        return np.nan

B = 300
n = len(df_model)
rng = np.random.default_rng(123)

OR_list = []
for _ in range(B):
    idx = rng.integers(0, n, n)
    sub = df_model.iloc[idx]
    OR_list.append(fit_treatment_or(sub))

OR_series = pd.Series(OR_list).dropna()

if len(OR_series) > 0:
    median_OR = OR_series.median()
    low_OR = OR_series.quantile(0.025)
    high_OR = OR_series.quantile(0.975)
    pct_gt1 = (OR_series > 1).mean()

    print("\nBootstrap summary for TreatmentMain (Y_bad):")
    print("  N bootstrap samples used:", len(OR_series))
    print("  Median OR:", median_OR)
    print("  95% bootstrap CI:", (low_OR, high_OR))
    print("  % of OR > 1:", pct_gt1)

    OR_series.to_csv("Q2C_bootstrap_OR_TreatmentMain_Ybad.csv", index=False)
    print("Saved bootstrap OR distribution to Q2C_bootstrap_OR_TreatmentMain_Ybad.csv.")
else:
    print("Bootstrap failed to produce valid OR estimates (e.g., separation in some resamples).")
# ------------------------------------------------------------
# 12-B. Plot Bootstrap OR distribution (Figure 7-2)
# ------------------------------------------------------------

import matplotlib.pyplot as plt

if len(OR_series) > 0:

    median_OR = OR_series.median()
    low_OR = OR_series.quantile(0.025)
    high_OR = OR_series.quantile(0.975)

    plt.figure(figsize=(7, 5))
    plt.hist(
        OR_series,
        bins=25,
        color="#4A90E2",
        alpha=0.85,
        edgecolor="white"
    )

    # median line
    plt.axvline(median_OR, color="red", linestyle="--",
                label=f"Median OR = {median_OR:.3f}")

    # 95% CI lines
    plt.axvline(low_OR, color="gray", linestyle=":",
                label=f"95% CI lower = {low_OR:.3f}")
    plt.axvline(high_OR, color="gray", linestyle=":",
                label=f"95% CI upper = {high_OR:.3f}")

    plt.xlabel("Odds Ratio (TreatmentMain)")
    plt.ylabel("Frequency")
    plt.title("Bootstrap Distribution of OR for TreatmentMain (300 Resamples)")
    plt.legend()
    plt.tight_layout()

    plt.savefig("Figure7_2_Bootstrap_OR_TreatmentMain.png", dpi=300)
    plt.show()

    print("\nSaved figure as Figure7_2_Bootstrap_OR_TreatmentMain.png")

else:
    print("No valid OR values available for plotting.")
print("\nAll analyses completed.")